# CS781 Capstone — Phase 3: Rhetorical Role prediction (RR) on 2×T4

**Kaggle settings:** Accelerator **GPU T4 ×2**, Internet **on**, secret **`HF`**, **Add Input →** your Phase-1 outputs
(`run_A`, `run_B2`, `run_C`, `run_D` best.pt + `legal_bpe.json`), same as Phase 2.

**Your parts (course rule 9):** `heads.py`, `techniques/*.py` (LoRA, LoRMA, rsLoRA, AdaLoRA, QLoRA, IA³, prompt
tuning, prefix tuning, KD) and every `None` in Stage 3 (configs) and Stage 6 (ensembles).
Everything else is plumbing (declare it): data, sentence windows + context, pooling, training loop, macro-F1,
prediction.json + zip, ensembles.

**Phase-3 technique list (§9.2.1):** LoRA, QLoRA, AdaLoRA, rsLoRA, **LoRMA (compulsory)**, Prefix Tuning, Prompt
Tuning, IA³, **KD (mandatory)** — at least 6 distinct, all four Phase-1 models, both CL and IT.

Order: 0 → 1 → 1b → 2 (data + oracle) → 3 (configs) → 3b (smoke) → 4 (runs, 2 GPUs) → 5 (results) → 6 (ensemble).
Outputs: `/kaggle/working/phase3/runs/<name>/` and all zips in `/kaggle/working/submissions/`.

## Stage 0 — session, folders

In [ ]:
# session clock + folders + GPUs
import os, sys, json, glob, time, hashlib, subprocess, shutil
SESSION_START = time.time()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytorch-crf"], check=True)   # heads.py imports torchcrf
WORK = "/kaggle/working/phase3"
for d in ("phase1_models", "techniques", "configs", "runs", "data"):
    os.makedirs(os.path.join(WORK, d), exist_ok=True)
os.chdir(WORK)
sys.path.insert(0, WORK)
import torch
N_GPUS = torch.cuda.device_count()
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(N_GPUS)])

## Stage 1 — plumbing files (your Phase-1 model code unchanged; declare the rest)

In [ ]:
%%writefile /kaggle/working/phase3/a1_tokenizer.py
# Legal byte-level BPE (your A1 tokenizer, optimised)
# tokenizer.py
#
# Byte-level BPE with an end-of-word marker (A1 design), adapted for the CS781 legal corpus.
#
# Changes vs. the A1 version (original kept in Assignment3/a1_tokenizer.py):
#   * Tokens are integer ids backed by raw bytes, not strings. Previously the byte "<", "/",
#     "w", ">" could merge into a token identical to the "</w>" marker (or to "<pad>"), and
#     decode() would then insert spurious spaces. Ids make that collision impossible.
#   * Legal-text pre-tokenisation inside each whitespace word: letter runs, digit groups of
#     at most 3, and punctuation runs are merged separately ("s.302(1)," -> s . 302 ( 1 ) ,),
#     so "court", "court," and "court." share one vocabulary entry instead of three, and
#     years / section numbers / citations don't flood the vocabulary. Only the last piece of
#     each whitespace word carries </w>, so decode(encode(x)) == " ".join(x.split()).
#   * Incremental BPE training (pair counts + inverted index + lazy max-heap): each merge
#     only touches the words that contain the merged pair, instead of recounting every pair
#     in the corpus for every merge. 32k merges on a legal sample take minutes, not days.
#   * Encoding uses merge ranks (GPT-2 style, same result as applying merges in order) and a
#     bounded per-word cache (the old lru_cache on a method also cached `self` and never freed).
#   * Extra reserved special tokens for later phases (cls/sep/mask, span-corruption sentinels).
#   * encode_many() encodes many documents in parallel worker processes.
#
# Assignment 3 contract is unchanged: train_tokenizer(train_text, vocab_size) returns an object
# with encode(text) -> list[int], decode(ids) -> str and a vocab_size attribute.

import re
import os
import heapq
import string
import json
from collections import Counter, defaultdict


# ============================================================
# SPECIAL TOKENS
# ============================================================

N_SENTINELS = 25

SPECIALS = (
    ["<pad>", "<unk>", "<s>", "</s>", "<cls>", "<sep>", "<mask>"]
    + [f"<extra_id_{i}>" for i in range(N_SENTINELS)]
)   # 32 specials: ids 0..31

MIN_COUNT = 2

# Pieces inside one whitespace word: letter runs | digit groups (<= 3) | punctuation runs | underscores.
# The alternatives cover every non-space character, so "".join(pieces) == word always holds.
PRETOKENIZE = re.compile(r"[^\W\d_]+|\d{1,3}|[^\w\s]+|_+")

MAX_CACHE_WORDS = 2_000_000


# ============================================================
# TEXT CLEANING (A1 helpers, unchanged; not used by the BPE)
# ============================================================

def check_valid_en(word):
    is_en = False
    is_hi = False
    has_digit = False

    for ch in word:
        code = ord(ch)

        # English alphabet
        if 0x0041 <= code <= 0x005A or 0x0061 <= code <= 0x007A:
            is_en = True

        # Arabic numerals
        elif 0x0030 <= code <= 0x0039:
            has_digit = True

        # Devanagari numerals
        elif 0x0966 <= code <= 0x096F:
            has_digit = True

        # Devanagari
        elif 0x0900 <= code <= 0x097F:
            is_hi = True

    if has_digit:
        return False

    return is_en and not is_hi


def clean_text_and_tokenize_en(text):
    text = str(text)
    text = text.lower()
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)
    text = re.sub(r"<.*?>", "", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()
    tokens = text.split()
    return [word for word in tokens if check_valid_en(word)]


# ============================================================
# BYTE-LEVEL BPE
# ============================================================

def split_word(word):
    """Whitespace word -> list of (piece, is_last_piece)."""
    pieces = PRETOKENIZE.findall(word) or [word]
    last = len(pieces) - 1
    return [(p, i == last) for i, p in enumerate(pieces)]


class ByteBPE:

    def __init__(self, vocab_size=16000):

        self.vocab_size = vocab_size

        # Id layout: specials | 256 bytes | </w> | one id per learned merge
        self.n_special = len(SPECIALS)
        self.byte_offset = self.n_special
        self.eow_id = self.byte_offset + 256

        self.merges = []            # list of (left_id, right_id); merge k creates id eow_id + 1 + k
        self._reset_tables()

    # --------------------------------------------------------
    # VOCAB TABLES
    # --------------------------------------------------------

    def _reset_tables(self):
        self.token_bytes = [b""] * self.n_special + [bytes([b]) for b in range(256)] + [b""]
        self.token_eow = [False] * (self.n_special + 256) + [True]
        for left, right in self.merges:
            self.token_bytes.append(self.token_bytes[left] + self.token_bytes[right])
            self.token_eow.append(self.token_eow[right])
        self.ranks = {pair: rank for rank, pair in enumerate(self.merges)}
        self.merge_to_id = {pair: self.eow_id + 1 + rank for rank, pair in enumerate(self.merges)}
        # decode table: bytes of each token, plus a space if it ends a word; specials decode to nothing
        self._decode_table = [
            b"" if i < self.n_special else tb + (b" " if eow else b"")
            for i, (tb, eow) in enumerate(zip(self.token_bytes, self.token_eow))
        ]
        self.itos = list(SPECIALS) + [self._display(i) for i in range(self.n_special, len(self.token_bytes))]
        self.stoi = {s: i for i, s in enumerate(self.itos)}
        self.vocab = self.itos
        self._cache = {}

    def _display(self, idx):
        """Human-readable token string (for inspection only; never used for encoding)."""
        text = self.token_bytes[idx].decode("utf-8", errors="backslashreplace")
        return text + ("</w>" if self.token_eow[idx] else "")

    @property
    def special_ids(self):
        return {tok: i for i, tok in enumerate(SPECIALS)}

    @property
    def eot_id(self):
        """Document separator used in the pretraining stream."""
        return SPECIALS.index("</s>")

    def get_vocab_size(self):
        return len(self.token_bytes)

    # --------------------------------------------------------
    # PIECE -> INITIAL SYMBOLS
    # --------------------------------------------------------

    def _piece_symbols(self, piece, is_last):
        symbols = [self.byte_offset + b for b in piece.encode("utf-8")]
        if is_last:
            symbols.append(self.eow_id)
        return symbols

    # --------------------------------------------------------
    # TRAIN BPE
    # --------------------------------------------------------

    @staticmethod
    def count_words(texts):
        """Whitespace-word frequencies of an iterable of texts."""
        counts = Counter()
        for text in texts:
            counts.update(str(text).split())
        return counts

    def fit(self, tokenized_corpus=None, target_vocab_size=None, min_count=MIN_COUNT,
            word_counts=None, min_word_freq=1, verbose=True):
        """Learn merges.

        tokenized_corpus: iterable of lists of whitespace words (A1/A3 interface), or
        word_counts:      a precomputed {whitespace word: frequency} mapping (faster for big corpora).
        min_word_freq:    ignore whitespace words rarer than this while learning merges (speed knob).
        """
        target = self.vocab_size if target_vocab_size is None else target_vocab_size
        self.vocab_size = target

        if word_counts is None:
            word_counts = Counter()
            for sentence_tokens in tokenized_corpus:
                word_counts.update(sentence_tokens)

        # whitespace words -> (piece, is_last) frequencies
        piece_freq = Counter()
        for word, count in word_counts.items():
            if count < min_word_freq:
                continue
            for piece in split_word(word):
                piece_freq[piece] += count

        words = [self._piece_symbols(p, last) for p, last in piece_freq]
        freqs = list(piece_freq.values())
        del piece_freq

        if verbose:
            print("=" * 60)
            print("BUILDING BYTE-LEVEL BPE")
            print("=" * 60)
            print(f"Unique pre-token pieces: {len(words):,}")
            print(f"Initial vocabulary size: {self.eow_id + 1:,}")
            print(f"Target vocabulary size: {target:,}")

        # pair -> total count, pair -> set of word indices containing it
        pair_counts = defaultdict(int)
        where = defaultdict(set)
        for wi, (symbols, f) in enumerate(zip(words, freqs)):
            for pair in zip(symbols, symbols[1:]):
                pair_counts[pair] += f
                where[pair].add(wi)

        # max-heap on count with deterministic tie-break on ids; stale entries are skipped lazily
        heap = [(-c, pair) for pair, c in pair_counts.items()]
        heapq.heapify(heap)

        self.merges = []
        next_id = self.eow_id + 1

        while next_id < target and heap:
            neg, pair = heapq.heappop(heap)
            current = pair_counts.get(pair, 0)
            if current != -neg:
                if current > 0:
                    heapq.heappush(heap, (-current, pair))
                continue
            if current < min_count:
                if verbose:
                    print(f"\nStopping: best pair frequency = {current}")
                break

            left, right = pair
            new_id = next_id
            next_id += 1
            self.merges.append(pair)
            touched = Counter()

            for wi in list(where.pop(pair, ())):
                symbols, f = words[wi], freqs[wi]
                # remove this word's old pairs
                for p in zip(symbols, symbols[1:]):
                    pair_counts[p] -= f
                    touched[p] += 0
                    s = where.get(p)
                    if s is not None:
                        s.discard(wi)
                # merge left-to-right
                merged, i, n = [], 0, len(symbols)
                while i < n:
                    if i < n - 1 and symbols[i] == left and symbols[i + 1] == right:
                        merged.append(new_id)
                        i += 2
                    else:
                        merged.append(symbols[i])
                        i += 1
                words[wi] = merged
                # add the new pairs
                for p in zip(merged, merged[1:]):
                    pair_counts[p] += f
                    touched[p] += 0
                    where[p].add(wi)

            pair_counts.pop(pair, None)
            for p in touched:
                c = pair_counts.get(p, 0)
                if c <= 0:
                    pair_counts.pop(p, None)
                    where.pop(p, None)
                elif new_id in p:
                    heapq.heappush(heap, (-c, p))   # only pairs with the new symbol can grow

            if verbose and len(self.merges) % 1000 == 0:
                print(f"Merges: {len(self.merges):,} | Vocab: {next_id:,} | Latest pair count: {current:,}")

        self._reset_tables()

        if verbose:
            print("\n" + "=" * 60)
            print("BPE TRAINING COMPLETE")
            print("=" * 60)
            print(f"Final vocabulary size: {self.get_vocab_size():,}")
            print(f"Total merges learned: {len(self.merges):,}")

        return self

    # --------------------------------------------------------
    # ENCODE
    # --------------------------------------------------------

    def _bpe(self, symbols):
        """Repeatedly merge the lowest-rank adjacent pair (== applying merges in learned order)."""
        ranks = self.ranks
        while len(symbols) > 1:
            best, best_rank = None, None
            for pair in zip(symbols, symbols[1:]):
                r = ranks.get(pair)
                if r is not None and (best_rank is None or r < best_rank):
                    best, best_rank = pair, r
            if best is None:
                break
            new_id = self.merge_to_id[best]
            left, right = best
            merged, i, n = [], 0, len(symbols)
            while i < n:
                if i < n - 1 and symbols[i] == left and symbols[i + 1] == right:
                    merged.append(new_id)
                    i += 2
                else:
                    merged.append(symbols[i])
                    i += 1
            symbols = merged
        return symbols

    def encode_word(self, word):
        ids = self._cache.get(word)
        if ids is None:
            ids = []
            for piece, is_last in split_word(word):
                ids.extend(self._bpe(self._piece_symbols(piece, is_last)))
            ids = tuple(ids)
            if len(self._cache) >= MAX_CACHE_WORDS:
                self._cache.clear()
            self._cache[word] = ids
        return ids

    def encode_tokens(self, sentence_tokens):
        ids = []
        for word in sentence_tokens:
            ids.extend(self.encode_word(word))
        return ids

    def encode(self, text):
        """text -> list of integer ids (whitespace is normalised to single spaces)."""
        cache = self._cache
        ids = []
        for word in str(text).split():
            w = cache.get(word)
            if w is None:
                w = self.encode_word(word)
            ids.extend(w)
        return ids

    def pool(self, workers=None):
        """Process pool whose workers each hold a copy of this tokenizer (and keep their word cache).
        Use with the module-level worker functions, e.g. pool.imap(encode_array_worker, texts)."""
        import multiprocessing as mp
        workers = workers or os.cpu_count() or 1
        return mp.get_context("fork").Pool(workers, initializer=_worker_init, initargs=(self,))

    def encode_many(self, texts, workers=None, chunksize=64):
        """Encode many documents in parallel processes; yields lists of ids in input order."""
        with self.pool(workers) as pool:
            yield from pool.imap(_worker_encode, texts, chunksize=chunksize)

    # --------------------------------------------------------
    # DECODE
    # --------------------------------------------------------

    def decode(self, ids):
        table = self._decode_table
        n = len(table)
        out = b"".join(table[int(i)] if 0 <= int(i) < n else b"" for i in ids)
        return out.decode("utf-8", errors="replace").strip()

    def ends_word(self, idx):
        """True if token idx closes a whitespace word (used to count generated words)."""
        return self.token_eow[idx]

    # --------------------------------------------------------
    # SAVE / LOAD
    # --------------------------------------------------------

    def to_json(self, path):
        with open(path, "w", encoding="utf-8") as f:
            json.dump(
                {
                    "format": "legal-bytebpe-v2",
                    "specials": SPECIALS,
                    "merges": self.merges,
                    "vocab_size": self.get_vocab_size(),
                    "itos": self.itos,          # readable, informational only
                },
                f,
                ensure_ascii=False,
            )
        print(f"Saved tokenizer to: {path}")

    @classmethod
    def from_json(cls, path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        if data.get("format") != "legal-bytebpe-v2":
            raise ValueError(f"{path} was not written by this tokenizer version")
        if data["specials"] != SPECIALS:
            raise ValueError("special tokens differ from this tokenizer version")
        tokenizer = cls(vocab_size=data["vocab_size"])
        tokenizer.merges = [tuple(pair) for pair in data["merges"]]
        tokenizer._reset_tables()
        return tokenizer


# worker-process helpers for encode_many (module level so they pickle)
_WORKER_TOKENIZER = None


def _worker_init(tokenizer):
    global _WORKER_TOKENIZER
    _WORKER_TOKENIZER = tokenizer


def _worker_encode(text):
    return _WORKER_TOKENIZER.encode(text)


def encode_array_worker(text):
    """(uint16 id array, whitespace word count) for one document; for tokenizer.pool().imap."""
    import numpy as np
    return np.asarray(_WORKER_TOKENIZER.encode(text), dtype=np.uint16), text.count(" ") + 1


# ============================================================
# ASSIGNMENT 3 TRAINING FUNCTION
# ============================================================

def train_tokenizer(train_text: str, vocab_size: int):
    text = str(train_text)
    print(f"Training tokenizer on {len(text):,} characters")
    tokenizer = ByteBPE(vocab_size=vocab_size)
    tokenizer.fit(
        word_counts=ByteBPE.count_words(text.splitlines()),
        target_vocab_size=vocab_size,
        min_count=MIN_COUNT,
    )
    return tokenizer


# ============================================================
# SIMPLE SMOKE TEST
# ============================================================

if __name__ == "__main__":

    print("=" * 60)
    print("TOKENIZER SMOKE TEST")
    print("=" * 60)

    train_text = """
    the court has jurisdiction over this matter under Section 302(1) of the Code.
    the court shall consider the evidence, vide AIR 1966 SC 1882, dated 18.12.1961.
    the evidence presented before the court </w> <pad> was 12,345 pages.
    """

    tokenizer = train_tokenizer(train_text, vocab_size=400)

    for test_text in [
        "the court has jurisdiction",
        "Section 302(1), AIR 1966 SC 1882 </w><pad> naïve — ₹5,00,000/- (u/s 138)",
    ]:
        ids = tokenizer.encode(test_text)
        decoded = tokenizer.decode(ids)
        print("\nInput:  ", test_text)
        print("Ids:    ", ids)
        print("Pieces: ", [tokenizer.itos[i] for i in ids])
        print("Decoded:", decoded)
        assert decoded == " ".join(test_text.split()), "round trip failed"

    print("\nVocabulary size:", tokenizer.get_vocab_size())

    tokenizer.to_json("bpe_smoke_test.json")
    loaded = ByteBPE.from_json("bpe_smoke_test.json")
    assert loaded.encode(test_text) == tokenizer.encode(test_text)
    os.remove("bpe_smoke_test.json")

    print("\n" + "=" * 60)
    print("SMOKE TEST PASSED")
    print("=" * 60)

In [ ]:
%%writefile /kaggle/working/phase3/phase1_models/rope.py
# your rope.py (two lines fixed, marked FIX (Claude))
"""Rotary Positional Embeddings, implemented from scratch."""
import torch


def build_rope_cache(max_seq_len: int, d_head: int, base: float = 10000.0):
    """
    Returns (cos, sin) tensors of shape (max_seq_len, d_head) ready to be applied to
    query/key tensors of head dimension d_head. d_head must be even. This scaffold uses
    the split-half rotate_half convention: channel i is paired with channel i+d_head/2.
    """
    assert d_head % 2 == 0, "RoPE requires an even head dimension"
    # TODO: compute inverse frequencies: 1 / base^(2i/d_head) for i in [0, d_head/2)
    inverse = [1/(base**(2*i/d_head)) for i in range(0,d_head//2)]
    inverse = torch.tensor(inverse, dtype=torch.float32)  # FIX (Claude): torch.outer needs a tensor, not a list
    # TODO: outer product of positions [0..max_seq_len) with inverse frequencies -> (T, d_head/2)
    pos = torch.arange(max_seq_len, dtype=torch.float32)  # FIX (Claude): float, to match `inverse` in torch.outer
    outer = torch.outer(pos,inverse)
    # TODO: construct angles = torch.cat([freqs, freqs], dim=-1) so the shape
    #       becomes (T, d_head) and matches the split-half rotate_half convention.
    #       Do NOT use repeat_interleave here; it produces a different channel layout.
    #       Then return angles.cos(), angles.sin().
    angles = torch.cat([outer,outer],dim=-1)
    return angles.cos() , angles.sin()
    


def rotate_half(x: torch.Tensor) -> torch.Tensor:
    """Split the last dim in half and return (-x2, x1) concatenated -- the 90-degree
    rotation step used by the "rotate_half" formulation of RoPE."""
    x1,x2 = torch.chunk(x,2,dim=-1)
    x2=-x2
    x = torch.cat([x2,x1],dim=-1)
    return x



def apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    """
    x: (B, n_heads, T, d_head) -- a query or key tensor.
    cos, sin: (T, d_head), sliced from build_rope_cache's output for the current T.
    Returns the rotated tensor, same shape as x.
    Formula: x_rotated = x * cos + rotate_half(x) * sin
    """
    # TODO: broadcast cos/sin to (1, 1, T, d_head) and apply the formula above
    cos = torch.unsqueeze(cos,dim=0)
    cos = torch.unsqueeze(cos,dim=0)
    sin = torch.unsqueeze(sin,dim=0)
    sin = torch.unsqueeze(sin,dim=0)
    x_rotated = x*cos+rotate_half(x)*sin
    
    return x_rotated

In [ ]:
%%writefile /kaggle/working/phase3/phase1_models/arch1.py
# Architecture 1 — attention1.py + decoder1.py (your code)
# ======== attention1.py (your A3 code; cross-file import removed, FIX lines marked) ========
"""Multi-head causal self-attention, implemented from raw tensor ops."""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, max_seq_len: int, dropout: float = 0.0, attention_scale=None):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        # Hook for Q1.4. None means the standard Transformer scaling 1/sqrt(d_head).
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)

        # TODO: define self.q_proj, self.k_proj, self.v_proj, self.out_proj as nn.Linear
        # (no bias). All should map d_model -> d_model.
        self.q_proj = nn.Linear(d_model,d_model,bias=False)
        self.k_proj = nn.Linear(d_model,d_model,bias=False)
        self.v_proj = nn.Linear(d_model,d_model,bias=False)
        self.out_proj = nn.Linear(d_model,d_model,bias=False)

        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        # TODO: register a (max_seq_len, max_seq_len) lower-triangular boolean mask as a
        # buffer named `causal_mask` (True = allowed to attend, False = masked out), so
        # you don't rebuild it on every forward call.
        causal_mask = torch.tril(
            torch.ones(max_seq_len,max_seq_len,dtype=torch.bool)
        )
        self.register_buffer("causal_mask", causal_mask, persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, T, d_model) -> (B, T, d_model)"""
        B, T, C = x.shape

        # TODO 1: project to q, k, v  (each (B, T, C))
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)
        # TODO 2: reshape each to (B, n_heads, T, d_head) — split C into (n_heads, d_head)
        #         and move n_heads next to the batch dim.
        q = torch.reshape(q,(B,T,self.n_heads,self.d_head)).transpose(1,2)
        k = torch.reshape(k,(B,T,self.n_heads,self.d_head)).transpose(1,2)
        v = torch.reshape(v,(B,T,self.n_heads,self.d_head)).transpose(1,2)
        
        # TODO 3: compute attention scores = (q @ k^T) * self.attention_scale
        #         -> (B, n_heads, T, T). Do NOT hard-code 1/sqrt(d_head): Q1.4
        #         changes this hook for the scaling ablation.
        attention_scores = (q@k.transpose(2,3))*self.attention_scale
        # TODO 4: apply the causal mask (set disallowed positions to -inf before softmax)
        attention_scores = attention_scores.masked_fill(
            ~self.causal_mask[:T,:T],float("-inf")
        )
        # TODO 5: softmax over the last dim, apply self.attn_dropout
        attention_weights = torch.softmax(attention_scores, dim=-1)
        attention_weights = self.attn_dropout(attention_weights)
        # TODO 6: weighted sum with v -> (B, n_heads, T, d_head)
        weighted_sum = attention_weights@v
        # TODO 7: reshape back to (B, T, C) and apply self.out_proj + self.resid_dropout
        weighted_sum = torch.reshape(weighted_sum.transpose(1,2),(B,T,C))
        output = self.out_proj(weighted_sum)
        output = self.resid_dropout(output)
        
        return output


# ======== decoder1.py (your A3 code; cross-file import removed, FIX lines marked) ========
"""Decoder-only Transformer LM, built from scratch (no nn.Transformer*)."""
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 512
    n_layers: int = 4
    n_heads: int = 4
    n_kv_heads: int = 4          # == n_heads for plain MHA; used in Question 2
    d_ff: int = 1024
    max_seq_len: int = 256
    dropout: float = 0.1
    pos_emb_type: str = "sinusoidal"   # or "rope", used in Question 3
    attention_scale: float = None      # Q1.4 hook; None -> 1/sqrt(d_head)
    norm_first: bool = False            # norm_first=True is for pre-norm else post-norm


def sinusoidal_positional_encoding(max_len: int, d_model: int) -> torch.Tensor:
    """Return a (max_len, d_model) tensor of the classic Transformer sinusoidal PE."""
    PE = torch.zeros(max_len, d_model)

    for pos in range(max_len):
        PE[pos, 0::2] = torch.sin(
        pos / (10000 ** (torch.arange(0, d_model, 2) / d_model))
    )
        PE[pos, 1::2] = torch.cos(
        pos / (10000 ** (torch.arange(0, d_model, 2) / d_model))
    )
    return PE


class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        # TODO: two linear layers with a GELU in between, plus dropout
        self.net = nn.Sequential(
            nn.Linear(d_model,d_ff),
            nn.GELU(),
            nn.Linear(d_ff,d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    """
    Pre-norm Transformer block: x + Attn(LN(x)), then x + FFN(LN(x)).
    OR
    Post-norm Transformer block: LN(x + Attn(x)), then LN(x + FFN(x)).
    """

    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.cfg = cfg
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.max_seq_len, cfg.dropout,
                                        attention_scale=cfg.attention_scale)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x):
        # TODO: implement pre-norm when norm_first=True,
        # cfg.norm_first = true for pre norm
        if self.cfg.norm_first is True:
            y1 = self.ln1(x)
            y2 = self.attn(y1)+x
            y3 = self.ln2(y2)
            y4 = self.ffn(y3)+y2
            output=y4
        else:
            y1 = self.attn(x)
            y2 = self.ln1(y1+x)
            y3 = self.ffn(y2)
            y4 = self.ln2(y3+y2)
            output=y4
            
        
        # and post-norm when norm_first=False.
        return output


class DecoderLM(nn.Module):
    """The model class the autograder imports directly. Do not rename it."""

    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)

        if cfg.pos_emb_type == "sinusoidal":
            pe = sinusoidal_positional_encoding(cfg.max_seq_len, cfg.d_model)
            self.register_buffer("pos_emb", pe, persistent=False)
        # (RoPE is applied inside attention instead of here — see Question 3)

        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight  # weight tying

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        """idx: (B, T) LongTensor of token ids -> logits (B, T, vocab_size)."""
        B, T = idx.shape
        # TODO: x = token embedding + positional encoding (if sinusoidal)
        x = self.tok_emb(idx)
        if(self.cfg.pos_emb_type == "sinusoidal"):
            x=x+self.pos_emb[:T]        # TODO: run x through every block, then ln_f, then self.head
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        x = self.head(x)
        
        return x

In [ ]:
%%writefile /kaggle/working/phase3/phase1_models/arch3.py
# Architecture 2 option — rope.py + attention3.py + decoder3.py (your code: GQA + RoPE)
# ======== rope.py (your A3 code; cross-file import removed, FIX lines marked) ========
"""Rotary Positional Embeddings, implemented from scratch."""
import torch


def build_rope_cache(max_seq_len: int, d_head: int, base: float = 10000.0):
    """
    Returns (cos, sin) tensors of shape (max_seq_len, d_head) ready to be applied to
    query/key tensors of head dimension d_head. d_head must be even. This scaffold uses
    the split-half rotate_half convention: channel i is paired with channel i+d_head/2.
    """
    assert d_head % 2 == 0, "RoPE requires an even head dimension"
    # TODO: compute inverse frequencies: 1 / base^(2i/d_head) for i in [0, d_head/2)
    inverse = [1/(base**(2*i/d_head)) for i in range(0,d_head//2)]
    inverse = torch.tensor(inverse, dtype=torch.float32)  # FIX (Claude): torch.outer needs a tensor, not a list
    # TODO: outer product of positions [0..max_seq_len) with inverse frequencies -> (T, d_head/2)
    pos = torch.arange(max_seq_len, dtype=torch.float32)  # FIX (Claude): float, to match `inverse` in torch.outer
    outer = torch.outer(pos,inverse)
    # TODO: construct angles = torch.cat([freqs, freqs], dim=-1) so the shape
    #       becomes (T, d_head) and matches the split-half rotate_half convention.
    #       Do NOT use repeat_interleave here; it produces a different channel layout.
    #       Then return angles.cos(), angles.sin().
    angles = torch.cat([outer,outer],dim=-1)
    return angles.cos() , angles.sin()
    


def rotate_half(x: torch.Tensor) -> torch.Tensor:
    """Split the last dim in half and return (-x2, x1) concatenated -- the 90-degree
    rotation step used by the "rotate_half" formulation of RoPE."""
    x1,x2 = torch.chunk(x,2,dim=-1)
    x2=-x2
    x = torch.cat([x2,x1],dim=-1)
    return x



def apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    """
    x: (B, n_heads, T, d_head) -- a query or key tensor.
    cos, sin: (T, d_head), sliced from build_rope_cache's output for the current T.
    Returns the rotated tensor, same shape as x.
    Formula: x_rotated = x * cos + rotate_half(x) * sin
    """
    # TODO: broadcast cos/sin to (1, 1, T, d_head) and apply the formula above
    cos = torch.unsqueeze(cos,dim=0)
    cos = torch.unsqueeze(cos,dim=0)
    sin = torch.unsqueeze(sin,dim=0)
    sin = torch.unsqueeze(sin,dim=0)
    x_rotated = x*cos+rotate_half(x)*sin
    
    return x_rotated


# ======== attention3.py (your A3 code; cross-file import removed, FIX lines marked) ========
"""Attention supporting MHA/MQA/GQA (Q2) with a choice of positional scheme: RoPE applied
to q/k inside attention, or none (if positions are handled by sinusoidal PE at the input,
as in Q1)."""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout=0.0, use_rope=False, attention_scale=None):
        super().__init__()
        assert d_model % n_heads == 0 and n_heads % n_kv_heads == 0
        self.n_heads, self.n_kv_heads = n_heads, n_kv_heads
        self.n_rep = n_heads // n_kv_heads
        self.d_head = d_model // n_heads
        self.use_rope = use_rope
        # Preserve the Q1.4 hook. None -> standard 1/sqrt(d_head).
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)

        self.q_proj = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.out_proj = nn.Linear(n_heads * self.d_head, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", mask, persistent=False)

        if use_rope:
            cos, sin = build_rope_cache(max_seq_len, self.d_head)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)

    def forward(self, x):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)

        if self.use_rope:
            # TODO: apply_rope to q and k (NOT v) using self.rope_cos[:T], self.rope_sin[:T]
            q = apply_rope(q,self.rope_cos[:T],self.rope_sin[:T])
            k = apply_rope(k,self.rope_cos[:T],self.rope_sin[:T])
        

        if self.n_rep > 1:
            k = k.repeat_interleave(self.n_rep, dim=1)
            v = v.repeat_interleave(self.n_rep, dim=1)

        # TODO: same attention as before, but compute scores as
        #       (q @ k.transpose(-2, -1)) * self.attention_scale before masking/softmax
        attention_scores = (q@k.transpose(-2,-1))*self.attention_scale
       
        attention_scores = attention_scores.masked_fill(
            ~self.causal_mask[:T,:T],float("-inf")
        )
       
        attention_weights = torch.softmax(attention_scores, dim=-1)
        attention_weights = self.attn_dropout(attention_weights)
    
        weighted_sum = attention_weights@v
   
        weighted_sum = torch.reshape(weighted_sum.transpose(1,2),(B,T,C))
        output = self.out_proj(weighted_sum)
        output = self.resid_dropout(output)
        
        return output


# ======== decoder3.py (your A3 code; cross-file import removed, FIX lines marked) ========
"""Decoder-only Transformer LM -- Block now passes use_rope through to attention, and
DecoderLM only builds/adds the sinusoidal table when pos_emb_type == "sinusoidal"
(everything else unchanged from Question 2)."""
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 256
    n_layers: int = 4
    n_heads: int = 4
    n_kv_heads: int = 4
    d_ff: int = 1024
    max_seq_len: int = 256
    dropout: float = 0.1
    pos_emb_type: str = "sinusoidal"   # "sinusoidal" or "rope"
    attention_scale: float = None      # preserve Q1.4 hook
    norm_first: bool = False


def sinusoidal_positional_encoding(max_len: int, d_model: int) -> torch.Tensor:
    """Return a (max_len, d_model) tensor of the classic Transformer sinusoidal PE."""
    PE = torch.zeros(max_len, d_model)

    for pos in range(max_len):
        PE[pos, 0::2] = torch.sin(
        pos / (10000 ** (torch.arange(0, d_model, 2) / d_model))
    )
        PE[pos, 1::2] = torch.cos(
        pos / (10000 ** (torch.arange(0, d_model, 2) / d_model))
    )
    return PE  # unchanged from Question 1


class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model,d_ff),
            nn.GELU(),
            nn.Linear(d_ff,d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.cfg=cfg
        self.ln1 = nn.LayerNorm(cfg.d_model)
        # TODO: pass use_rope=(cfg.pos_emb_type == "rope") to CausalSelfAttention
        self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.n_kv_heads, cfg.max_seq_len,
                                        cfg.dropout, use_rope=(cfg.pos_emb_type == "rope"),
                                        attention_scale=cfg.attention_scale)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x):
        if self.cfg.norm_first is True:
            y1 = self.ln1(x)
            y2 = self.attn(y1)+x
            y3 = self.ln2(y2)
            y4 = self.ffn(y3)+y2
            output=y4
        else:
            y1 = self.attn(x)
            y2 = self.ln1(y1+x)
            y3 = self.ffn(y2)
            y4 = self.ln2(y3+y2)
            output=y4
            
        
        # and post-norm when norm_first=False.
        return output


class DecoderLM(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        if cfg.pos_emb_type == "sinusoidal":
            pe = sinusoidal_positional_encoding(cfg.max_seq_len, cfg.d_model)
            self.register_buffer("pos_emb", pe, persistent=False)
        # NOTE: when cfg.pos_emb_type == "rope", we deliberately do NOT build a
        # sinusoidal table at all -- position information is injected inside every
        # attention layer instead of once at the input.
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        B, T = idx.shape
        # TODO: x = token embedding + positional encoding (if sinusoidal)
        x = self.tok_emb(idx)
        if(self.cfg.pos_emb_type == "sinusoidal"):
            x=x+self.pos_emb[:T]        # TODO: run x through every block, then ln_f, then self.head
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        x = self.head(x)
        
        return x  # unchanged from Question 1 (just remember: only add
        # self.pos_emb when self.cfg.pos_emb_type == "sinusoidal")

In [ ]:
%%writefile /kaggle/working/phase3/phase1_models/attention3.py
# your attention3.py (verbatim)
"""Attention supporting MHA/MQA/GQA (Q2) with a choice of positional scheme: RoPE applied
to q/k inside attention, or none (if positions are handled by sinusoidal PE at the input,
as in Q1)."""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from rope import build_rope_cache, apply_rope


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout=0.0, use_rope=False, attention_scale=None):
        super().__init__()
        assert d_model % n_heads == 0 and n_heads % n_kv_heads == 0
        self.n_heads, self.n_kv_heads = n_heads, n_kv_heads
        self.n_rep = n_heads // n_kv_heads
        self.d_head = d_model // n_heads
        self.use_rope = use_rope
        # Preserve the Q1.4 hook. None -> standard 1/sqrt(d_head).
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)

        self.q_proj = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.out_proj = nn.Linear(n_heads * self.d_head, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", mask, persistent=False)

        if use_rope:
            cos, sin = build_rope_cache(max_seq_len, self.d_head)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)

    def forward(self, x):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)

        if self.use_rope:
            # TODO: apply_rope to q and k (NOT v) using self.rope_cos[:T], self.rope_sin[:T]
            q = apply_rope(q,self.rope_cos[:T],self.rope_sin[:T])
            k = apply_rope(k,self.rope_cos[:T],self.rope_sin[:T])
        

        if self.n_rep > 1:
            k = k.repeat_interleave(self.n_rep, dim=1)
            v = v.repeat_interleave(self.n_rep, dim=1)

        # TODO: same attention as before, but compute scores as
        #       (q @ k.transpose(-2, -1)) * self.attention_scale before masking/softmax
        attention_scores = (q@k.transpose(-2,-1))*self.attention_scale
       
        attention_scores = attention_scores.masked_fill(
            ~self.causal_mask[:T,:T],float("-inf")
        )
       
        attention_weights = torch.softmax(attention_scores, dim=-1)
        attention_weights = self.attn_dropout(attention_weights)
    
        weighted_sum = attention_weights@v
   
        weighted_sum = torch.reshape(weighted_sum.transpose(1,2),(B,T,C))
        output = self.out_proj(weighted_sum)
        output = self.resid_dropout(output)
        
        return output

In [ ]:
%%writefile /kaggle/working/phase3/phase1_models/non_causal_self_attention.py
# your non_causal_self_attention.py (verbatim)
"""Attention supporting MHA/MQA/GQA (Q2) with a choice of positional scheme: RoPE applied
to q/k inside attention, or none (if positions are handled by sinusoidal PE at the input,
as in Q1)."""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from rope import build_rope_cache, apply_rope


class SelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout=0.0, use_rope=False, attention_scale=None):
        super().__init__()
        assert d_model % n_heads == 0 and n_heads % n_kv_heads == 0
        self.n_heads, self.n_kv_heads = n_heads, n_kv_heads
        self.n_rep = n_heads // n_kv_heads
        self.d_head = d_model // n_heads
        self.use_rope = use_rope
        # Preserve the Q1.4 hook. None -> standard 1/sqrt(d_head).
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)

        self.q_proj = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.out_proj = nn.Linear(n_heads * self.d_head, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        

        if use_rope:
            cos, sin = build_rope_cache(max_seq_len, self.d_head)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)

    def forward(self, x):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)

        if self.use_rope:
            # TODO: apply_rope to q and k (NOT v) using self.rope_cos[:T], self.rope_sin[:T]
            q = apply_rope(q,self.rope_cos[:T],self.rope_sin[:T])
            k = apply_rope(k,self.rope_cos[:T],self.rope_sin[:T])
        

        if self.n_rep > 1:
            k = k.repeat_interleave(self.n_rep, dim=1)
            v = v.repeat_interleave(self.n_rep, dim=1)

        # TODO: same attention as before, but compute scores as
        #       (q @ k.transpose(-2, -1)) * self.attention_scale before masking/softmax
        attention_scores = (q@k.transpose(-2,-1))*self.attention_scale
       
       
        attention_weights = torch.softmax(attention_scores, dim=-1)
        attention_weights = self.attn_dropout(attention_weights)
    
        weighted_sum = attention_weights@v
   
        weighted_sum = torch.reshape(weighted_sum.transpose(1,2),(B,T,C))
        output = self.out_proj(weighted_sum)
        output = self.resid_dropout(output)
        
        return output

In [ ]:
%%writefile /kaggle/working/phase3/phase1_models/cross_attention.py
# your cross_attention.py (verbatim)
"""Cross Attention supporting MHA/MQA/GQA (Q2) with a choice of positional scheme: RoPE applied
to q/k inside attention, or none (if positions are handled by sinusoidal PE at the input,
as in Q1)."""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from rope import build_rope_cache, apply_rope


class CrossAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout=0.0, use_rope=False, attention_scale=None):
        super().__init__()
        assert d_model % n_heads == 0 and n_heads % n_kv_heads == 0
        self.n_heads, self.n_kv_heads = n_heads, n_kv_heads
        self.n_rep = n_heads // n_kv_heads
        self.d_head = d_model // n_heads
        self.use_rope = use_rope
        # Preserve the Q1.4 hook. None -> standard 1/sqrt(d_head).
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)

        self.q_proj = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.out_proj = nn.Linear(n_heads * self.d_head, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        

        if use_rope:
            cos, sin = build_rope_cache(max_seq_len, self.d_head)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)

    def forward(self, x , memory):
        B, T, C = x.shape
        _,S,_ = memory.shape

        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(memory).view(B, S, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(memory).view(B, S, self.n_kv_heads, self.d_head).transpose(1, 2)

        if self.use_rope:
            # TODO: apply_rope to q and k (NOT v) using self.rope_cos[:T], self.rope_sin[:T]
            q = apply_rope(q,self.rope_cos[:T],self.rope_sin[:T])
            k = apply_rope(k,self.rope_cos[:T],self.rope_sin[:T])
        

        if self.n_rep > 1:
            k = k.repeat_interleave(self.n_rep, dim=1)
            v = v.repeat_interleave(self.n_rep, dim=1)

        # TODO: same attention as before, but compute scores as
        #       (q @ k.transpose(-2, -1)) * self.attention_scale before masking/softmax
        attention_scores = (q@k.transpose(-2,-1))*self.attention_scale
       
       
        attention_weights = torch.softmax(attention_scores, dim=-1)
        attention_weights = self.attn_dropout(attention_weights)
    
        weighted_sum = attention_weights@v
   
        weighted_sum = torch.reshape(weighted_sum.transpose(1,2),(B,T,C))
        output = self.out_proj(weighted_sum)
        output = self.resid_dropout(output)
        
        return output

In [ ]:
%%writefile /kaggle/working/phase3/phase1_models/encoder.py
# your encoder.py (verbatim)
"""Encoder-Block of  Transformer LM -- Block now passes use_rope through to attention, and
EncoderLM only builds/adds the sinusoidal table when pos_emb_type == "sinusoidal"
(everything else unchanged from Question 2)."""
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F
from non_causal_self_attention import SelfAttention


@dataclass
class EncLMConfig:
    vocab_size: int
    d_model: int = 256
    n_layers: int = 4
    n_heads: int = 4
    n_kv_heads: int = 4
    d_ff: int = 1024
    max_seq_len: int = 256
    dropout: float = 0.1
    pos_emb_type: str = "sinusoidal"   # "sinusoidal" or "rope"
    attention_scale: float = None      # preserve Q1.4 hook
    norm_first: bool = False


def sinusoidal_positional_encoding(max_len: int, d_model: int) -> torch.Tensor:
    """Return a (max_len, d_model) tensor of the classic Transformer sinusoidal PE."""
    PE = torch.zeros(max_len, d_model)

    for pos in range(max_len):
        PE[pos, 0::2] = torch.sin(
        pos / (10000 ** (torch.arange(0, d_model, 2) / d_model))
    )
        PE[pos, 1::2] = torch.cos(
        pos / (10000 ** (torch.arange(0, d_model, 2) / d_model))
    )
    return PE  # unchanged from Question 1


class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model,d_ff),
            nn.GELU(),
            nn.Linear(d_ff,d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)


class EncBlock(nn.Module):
    def __init__(self, cfg: EncLMConfig):
        super().__init__()
        self.cfg=cfg
        self.ln1 = nn.LayerNorm(cfg.d_model)
        # TODO: pass use_rope=(cfg.pos_emb_type == "rope") to CausalSelfAttention
        self.attn = SelfAttention(cfg.d_model, cfg.n_heads, cfg.n_kv_heads, cfg.max_seq_len,
                                        cfg.dropout, use_rope=(cfg.pos_emb_type == "rope"),
                                        attention_scale=cfg.attention_scale)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x):
        if self.cfg.norm_first is True:
            y1 = self.ln1(x)
            y2 = self.attn(y1)+x
            y3 = self.ln2(y2)
            y4 = self.ffn(y3)+y2
            output=y4
        else:
            y1 = self.attn(x)
            y2 = self.ln1(y1+x)
            y3 = self.ffn(y2)
            y4 = self.ln2(y3+y2)
            output=y4
            
        
        # and post-norm when norm_first=False.
        return output


class EncoderLM(nn.Module):
    def __init__(self, cfg: EncLMConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        if cfg.pos_emb_type == "sinusoidal":
            pe = sinusoidal_positional_encoding(cfg.max_seq_len, cfg.d_model)
            self.register_buffer("pos_emb", pe, persistent=False)
        # NOTE: when cfg.pos_emb_type == "rope", we deliberately do NOT build a
        # sinusoidal table at all -- position information is injected inside every
        # attention layer instead of once at the input.
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        #self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        #self.head.weight = self.tok_emb.weight

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        B, T = idx.shape
        # TODO: x = token embedding + positional encoding (if sinusoidal)
        x = self.tok_emb(idx)
        if(self.cfg.pos_emb_type == "sinusoidal"):
            x=x+self.pos_emb[:T]        # TODO: run x through every block, then ln_f, then self.head
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        #x = self.head(x)
        
        return x  # unchanged from Question 1 (just remember: only add
        # self.pos_emb when self.cfg.pos_emb_type == "sinusoidal")

In [ ]:
%%writefile /kaggle/working/phase3/phase1_models/decoder4.py
# your decoder4.py (verbatim)
"""Decoder-only Transformer LM -- Block now passes use_rope through to attention, and
DecoderLM only builds/adds the sinusoidal table when pos_emb_type == "sinusoidal"
(everything else unchanged from Question 2)."""
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F
from attention3 import CausalSelfAttention
from cross_attention import CrossAttention


@dataclass
class DecLMConfig:
    vocab_size: int
    d_model: int = 256
    n_layers: int = 4
    n_heads: int = 4
    n_kv_heads: int = 4
    d_ff: int = 1024
    max_seq_len: int = 256
    dropout: float = 0.1
    pos_emb_type: str = "sinusoidal"   # "sinusoidal" or "rope"
    attention_scale: float = None      # preserve Q1.4 hook
    norm_first: bool = False


def sinusoidal_positional_encoding(max_len: int, d_model: int) -> torch.Tensor:
    """Return a (max_len, d_model) tensor of the classic Transformer sinusoidal PE."""
    PE = torch.zeros(max_len, d_model)

    for pos in range(max_len):
        PE[pos, 0::2] = torch.sin(
        pos / (10000 ** (torch.arange(0, d_model, 2) / d_model))
    )
        PE[pos, 1::2] = torch.cos(
        pos / (10000 ** (torch.arange(0, d_model, 2) / d_model))
    )
    return PE  # unchanged from Question 1


class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model,d_ff),
            nn.GELU(),
            nn.Linear(d_ff,d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)


class DecBlock(nn.Module):
    def __init__(self, cfg: DecLMConfig):
        super().__init__()
        self.cfg=cfg
        self.ln1 = nn.LayerNorm(cfg.d_model)
        # TODO: pass use_rope=(cfg.pos_emb_type == "rope") to CausalSelfAttention
        self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.n_kv_heads, cfg.max_seq_len,
                                        cfg.dropout, use_rope=(cfg.pos_emb_type == "rope"),
                                        attention_scale=cfg.attention_scale)
        self.crattn = CrossAttention(cfg.d_model , cfg.n_heads,cfg.n_kv_heads,cfg.max_seq_len,
        cfg.dropout,use_rope=False,attention_scale=cfg.attention_scale)

        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ln3 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x , memory):
        if self.cfg.norm_first is True:
            y1 = self.ln1(x)
            y2 = self.attn(y1)+x
            y3 = self.ln2(y2)
            y4 = self.crattn(y3,memory)+y2
            y5 = self.ln3(y4)
            y6 = self.ffn(y5)+y4
            output=y6
        else:
            y1 = self.attn(x)
            y2 = self.ln1(y1+x)
            y3 = self.crattn(y2,memory)
            y4 = self.ln2(y3+y2)
            y5 = self.ffn(y4)
            y6 = self.ln3(y5+y4)
            output=y6
            
        
        # and post-norm when norm_first=False.
        return output


class DecoderLM(nn.Module):
    def __init__(self, cfg: DecLMConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        if cfg.pos_emb_type == "sinusoidal":
            pe = sinusoidal_positional_encoding(cfg.max_seq_len, cfg.d_model)
            self.register_buffer("pos_emb", pe, persistent=False)
        # NOTE: when cfg.pos_emb_type == "rope", we deliberately do NOT build a
        # sinusoidal table at all -- position information is injected inside every
        # attention layer instead of once at the input.
        self.blocks = nn.ModuleList([DecBlock(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight

    def forward(self, idx: torch.Tensor , memory:torch.Tensor) -> torch.Tensor:
        B, T = idx.shape
        # TODO: x = token embedding + positional encoding (if sinusoidal)
        x = self.tok_emb(idx)
        if(self.cfg.pos_emb_type == "sinusoidal"):
            x=x+self.pos_emb[:T]        # TODO: run x through every block, then ln_f, then self.head
        for block in self.blocks:
            x = block(x,memory)
        x = self.ln_f(x)
        x = self.head(x)
        
        return x  # unchanged from Question 1 (just remember: only add
        # self.pos_emb when self.cfg.pos_emb_type == "sinusoidal")

In [ ]:
%%writefile /kaggle/working/phase3/phase1_models/enc_dec.py
# your enc_dec.py (verbatim)
from dataclasses import dataclass 
import torch  
import torch.nn as nn  
from encoder import EncBlock, sinusoidal_positional_encoding  
from decoder4 import DecBlock

@dataclass
class EncLMConfig:
    vocab_size: int
    d_model: int = 256
    n_layers: int = 4
    n_heads: int = 4
    n_kv_heads: int = 4
    d_ff: int = 1024
    max_seq_len: int = 256
    dropout: float = 0.1
    pos_emb_type: str = "sinusoidal"   # "sinusoidal" or "rope"
    attention_scale: float = None      # preserve Q1.4 hook
    norm_first: bool = False


@dataclass
class DecLMConfig:
    vocab_size: int
    d_model: int = 256
    n_layers: int = 4
    n_heads: int = 4
    n_kv_heads: int = 4
    d_ff: int = 1024
    max_seq_len: int = 256
    dropout: float = 0.1
    pos_emb_type: str = "sinusoidal"   # "sinusoidal" or "rope"
    attention_scale: float = None      # preserve Q1.4 hook
    norm_first: bool = False


class EncDecLM(nn.Module):
    def __init__(self,cfg1:EncLMConfig,cfg2:DecLMConfig):
        super().__init__()
        # Encoder Block Initializations 
        self.cfg1 = cfg1
        self.tok_emb1 = nn.Embedding(cfg1.vocab_size, cfg1.d_model)
        if cfg1.pos_emb_type == "sinusoidal":
            pe = sinusoidal_positional_encoding(cfg1.max_seq_len, cfg1.d_model)
            self.register_buffer("pos_emb_enc", pe, persistent=False)
   
        self.encblocks = nn.ModuleList([EncBlock(cfg1) for _ in range(cfg1.n_layers)])
        self.ln_f_enc = nn.LayerNorm(cfg1.d_model)
        # Decoder Block Initializations
        self.cfg2 = cfg2
        self.tok_emb2 = nn.Embedding(cfg2.vocab_size, cfg2.d_model)
        if cfg2.pos_emb_type == "sinusoidal":
            pe = sinusoidal_positional_encoding(cfg2.max_seq_len, cfg2.d_model)
            self.register_buffer("pos_emb_dec", pe, persistent=False)
       
        self.decblocks = nn.ModuleList([DecBlock(cfg2) for _ in range(cfg2.n_layers)])
        self.ln_f_dec = nn.LayerNorm(cfg2.d_model)
        self.head = nn.Linear(cfg2.d_model, cfg2.vocab_size, bias=False)
        self.head.weight = self.tok_emb2.weight
    def encode(self,src:torch.Tensor)->torch.Tensor:
        # passing it through encoder 
        B,S = src.shape
        
        x_enc = self.tok_emb1(src)
        if(self.cfg1.pos_emb_type == "sinusoidal"):
            x_enc=x_enc+self.pos_emb_enc[:S]
        for block in self.encblocks:
            x_enc = block(x_enc)
        x_enc = self.ln_f_enc(x_enc)
        memory = x_enc
        return memory

    def decode(self,tgt:torch.Tensor,memory:torch.Tensor)->torch.Tensor:
        B,T = tgt.shape 
        x_dec = self.tok_emb2(tgt)
        if(self.cfg2.pos_emb_type == "sinusoidal"):
            x_dec=x_dec+self.pos_emb_dec[:T]
        for block in self.decblocks:
            x_dec = block(x_dec,memory)
        x_dec = self.ln_f_dec(x_dec)
        x_dec = self.head(x_dec)
        return x_dec


    def forward(self ,src:torch.Tensor , tgt:torch.Tensor)->torch.Tensor:
        return self.decode(tgt,self.encode(src))
        
         
        






        
    
        

In [ ]:
%%writefile /kaggle/working/phase3/phase1_models/prefix_attention.py
# your prefix_attention.py (verbatim)
""" Prefix Attention supporting MHA/MQA/GQA (Q2) with a choice of positional scheme: RoPE applied
to q/k inside attention, or none (if positions are handled by sinusoidal PE at the input,
as in Q1)."""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from rope import build_rope_cache, apply_rope


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout=0.0, use_rope=False, attention_scale=None):
        super().__init__()
        assert d_model % n_heads == 0 and n_heads % n_kv_heads == 0
        self.n_heads, self.n_kv_heads = n_heads, n_kv_heads
        self.n_rep = n_heads // n_kv_heads
        self.d_head = d_model // n_heads
        self.use_rope = use_rope
        # Preserve the Q1.4 hook. None -> standard 1/sqrt(d_head).
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)

        self.q_proj = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.out_proj = nn.Linear(n_heads * self.d_head, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        
        self.register_buffer("causal_mask", mask, persistent=False)

        if use_rope:
            cos, sin = build_rope_cache(max_seq_len, self.d_head)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)
     

    def forward(self, x , p):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)

        if self.use_rope:
            # TODO: apply_rope to q and k (NOT v) using self.rope_cos[:T], self.rope_sin[:T]
            q = apply_rope(q,self.rope_cos[:T],self.rope_sin[:T])
            k = apply_rope(k,self.rope_cos[:T],self.rope_sin[:T])
        

        if self.n_rep > 1:
            k = k.repeat_interleave(self.n_rep, dim=1)
            v = v.repeat_interleave(self.n_rep, dim=1)

        # TODO: same attention as before, but compute scores as
        #       (q @ k.transpose(-2, -1)) * self.attention_scale before masking/softmax
        attention_scores = (q@k.transpose(-2,-1))*self.attention_scale

        mask1 = self.causal_mask[:T,:T].clone()
        mask1[:p,:p]=1

        attention_scores = attention_scores.masked_fill(
            ~mask1,float("-inf")
        )
       
        attention_weights = torch.softmax(attention_scores, dim=-1)
        attention_weights = self.attn_dropout(attention_weights)
    
        weighted_sum = attention_weights@v
   
        weighted_sum = torch.reshape(weighted_sum.transpose(1,2),(B,T,C))
        output = self.out_proj(weighted_sum)
        output = self.resid_dropout(output)
        
        return output

In [ ]:
%%writefile /kaggle/working/phase3/phase1_models/prefix_lm.py
# your prefix_lm.py (verbatim)
"""Prefix LM - Decoder-only Transformer LM -- Block now passes use_rope through to attention, and
DecoderLM only builds/adds the sinusoidal table when pos_emb_type == "sinusoidal"
(everything else unchanged from Question 2). Different Architecture as mask changes the way information flows"""
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F
from prefix_attention import CausalSelfAttention


@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 256
    n_layers: int = 4
    n_heads: int = 4
    n_kv_heads: int = 4
    d_ff: int = 1024
    max_seq_len: int = 256
    dropout: float = 0.1
    pos_emb_type: str = "sinusoidal"   # "sinusoidal" or "rope"
    attention_scale: float = None      # preserve Q1.4 hook
    norm_first: bool = False
    prefix:int = 128


def sinusoidal_positional_encoding(max_len: int, d_model: int) -> torch.Tensor:
    """Return a (max_len, d_model) tensor of the classic Transformer sinusoidal PE."""
    PE = torch.zeros(max_len, d_model)

    for pos in range(max_len):
        PE[pos, 0::2] = torch.sin(
        pos / (10000 ** (torch.arange(0, d_model, 2) / d_model))
    )
        PE[pos, 1::2] = torch.cos(
        pos / (10000 ** (torch.arange(0, d_model, 2) / d_model))
    )
    return PE  # unchanged from Question 1


class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model,d_ff),
            nn.GELU(),
            nn.Linear(d_ff,d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.cfg=cfg
        self.ln1 = nn.LayerNorm(cfg.d_model)
        # TODO: pass use_rope=(cfg.pos_emb_type == "rope") to CausalSelfAttention
        self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.n_kv_heads, cfg.max_seq_len,
                                        cfg.dropout, use_rope=(cfg.pos_emb_type == "rope"),
                                        attention_scale=cfg.attention_scale)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x,p):
        if self.cfg.norm_first is True:
            y1 = self.ln1(x)
            y2 = self.attn(y1,p)+x
            y3 = self.ln2(y2)
            y4 = self.ffn(y3)+y2
            output=y4
        else:
            y1 = self.attn(x,p)
            y2 = self.ln1(y1+x)
            y3 = self.ffn(y2)
            y4 = self.ln2(y3+y2)
            output=y4
            
        
        # and post-norm when norm_first=False.
        return output


class DecoderLM(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        if cfg.pos_emb_type == "sinusoidal":
            pe = sinusoidal_positional_encoding(cfg.max_seq_len, cfg.d_model)
            self.register_buffer("pos_emb", pe, persistent=False)
        # NOTE: when cfg.pos_emb_type == "rope", we deliberately do NOT build a
        # sinusoidal table at all -- position information is injected inside every
        # attention layer instead of once at the input.
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        B, T = idx.shape
        # TODO: x = token embedding + positional encoding (if sinusoidal)
        x = self.tok_emb(idx)
        if(self.cfg.pos_emb_type == "sinusoidal"):
            x=x+self.pos_emb[:T]        # TODO: run x through every block, then ln_f, then self.head
        for block in self.blocks:
            x = block(x,self.cfg.prefix)
        x = self.ln_f(x)
        x = self.head(x)
        
        return x  # unchanged from Question 1 (just remember: only add
        # self.pos_emb when self.cfg.pos_emb_type == "sinusoidal")

In [ ]:
%%writefile /kaggle/working/phase3/phase1_models/fastattn_ab.py
# Plumbing: fused SDPA attention + KV cache + generation around your modules
"""Efficiency plumbing around the hand-written A3 models (written with Claude; declare it).

Nothing here changes an architecture: parameters, shapes and the maths of every model stay
exactly as defined in arch1/arch2/arch3. `patch_model` swaps each attention module's forward
for one that computes the *same* function with fused kernels, and adds an optional KV cache
for generation. `check_equivalence` in the notebook verifies outputs match the original code.

* Training: F.scaled_dot_product_attention(is_causal=True). On T4 (sm_75) PyTorch cannot use
  the FlashAttention-2 kernel (needs sm_80+); it dispatches to the memory-efficient fused
  kernel, which gives the same benefit: no (T x T) score matrix materialised in HBM.
* Generation: per-layer KV cache (keys/values for MHA/GQA, the compressed latent for MLA),
  prefill once, then one token per step.
"""
import sys
import types
from contextlib import contextmanager

import torch
import torch.nn.functional as F


# ---------------------------------------------------------------------------
# attention forward (drop-in for all three hand-written attention classes)
# ---------------------------------------------------------------------------

def _sdpa(q, k, v, causal, dropout_p, scale):
    return F.scaled_dot_product_attention(q, k, v, is_causal=causal, dropout_p=dropout_p, scale=scale)


def _fast_attention_forward(self, x):
    B, T, C = x.shape
    cache = getattr(self, "_kv_cache", None)
    start = cache["pos"] if cache is not None else 0
    n_heads, d_head = self.n_heads, self.d_head
    dropout_p = self.attn_dropout.p if self.training else 0.0

    q = self.q_proj(x).view(B, T, n_heads, d_head).transpose(1, 2)
    n_prefix = 0                                            # prefix tuning is only wired for MHA / GQA

    if hasattr(self, "kv_down_proj"):                       # MLA: cache the shared latent
        latent = self.kv_down_proj(x)
        if cache is not None:
            cache["latent"][:, start:start + T] = latent
            latent = cache["latent"][:, :start + T]
        S = latent.shape[1]
        k = self.k_up_proj(latent).view(B, S, n_heads, d_head).transpose(1, 2)
        v = self.v_up_proj(latent).view(B, S, n_heads, d_head).transpose(1, 2)
    else:                                                   # MHA / GQA (+ optional RoPE)
        n_kv = getattr(self, "n_kv_heads", n_heads)
        k = self.k_proj(x).view(B, T, n_kv, d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, n_kv, d_head).transpose(1, 2)
        if getattr(self, "use_rope", False):
            cos = self.rope_cos[start:start + T]
            sin = self.rope_sin[start:start + T]
            q = self._apply_rope(q.float(), cos, sin).to(v.dtype)
            k = self._apply_rope(k.float(), cos, sin).to(v.dtype)
        if cache is not None:
            cache["k"][:, :, start:start + T] = k
            cache["v"][:, :, start:start + T] = v
            k = cache["k"][:, :, :start + T]
            v = cache["v"][:, :, :start + T]
        k, v, n_prefix = _add_prefix(self, k, v, B, cache)
        n_rep = n_heads // n_kv
        if n_rep > 1:
            k = k.repeat_interleave(n_rep, dim=1)
            v = v.repeat_interleave(n_rep, dim=1)

    # prefill (start == 0) is causal; a single decode step attends to the whole cache
    causal = T > 1
    if causal and start > 0:
        raise ValueError("multi-token steps are only supported from an empty cache")
    if n_prefix:                                            # prefix tuning: every query also sees the P prefix keys
        mask = _causal_with_prefix(T, n_prefix, x.device)
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=dropout_p, scale=self.attention_scale)
    else:
        y = _sdpa(q, k, v, causal, dropout_p, self.attention_scale)
    y = y.transpose(1, 2).reshape(B, T, n_heads * d_head)
    return self.resid_dropout(self.out_proj(y))


# ---------------------------------------------------------------------------
# prefix-tuning hook (Phase 3 plumbing): an attention module may carry `prefix_kv`, a callable
# prefix_kv(batch_size) -> (pk, pv), each (B, n_kv_heads, P, d_head). Those P key/value vectors are prepended to the
# keys/values (no RoPE is applied to them) and every query may attend to them. Without `prefix_kv`: unchanged.
# ---------------------------------------------------------------------------

def _add_prefix(attn, k, v, B, cache):
    fn = getattr(attn, "prefix_kv", None)
    if fn is None:
        return k, v, 0
    assert cache is None, "prefix tuning is not supported together with the generation KV cache"
    pk, pv = fn(B)
    assert pk.shape[0] == B and pk.shape[1] == k.shape[1] and pk.shape[3] == k.shape[3], \
        f"prefix_kv must return (B, n_kv_heads, P, d_head) = ({B}, {k.shape[1]}, P, {k.shape[3]}), got {tuple(pk.shape)}"
    return torch.cat([pk.to(k.dtype), k], dim=2), torch.cat([pv.to(v.dtype), v], dim=2), pk.shape[2]


def _causal_with_prefix(T, P, device):
    m = torch.ones(T, P + T, dtype=torch.bool, device=device)
    m[:, P:] = torch.ones(T, T, dtype=torch.bool, device=device).tril()
    return m


def attention_modules(model):
    return [block.attn for block in model.blocks]


def patch_model(model):
    """Swap every attention forward for the fused/cached version (same parameters)."""
    for attn in attention_modules(model):
        attn._kv_cache = None
        if getattr(attn, "use_rope", False):
            # the user's own apply_rope, from the module that defines this attention class
            attn._apply_rope = sys.modules[type(attn).__module__].apply_rope
        attn.forward = types.MethodType(_fast_attention_forward, attn)
    return model


# ---------------------------------------------------------------------------
# KV cache
# ---------------------------------------------------------------------------

@contextmanager
def kv_cache(model, batch_size, max_len, dtype):
    """Allocate a per-layer cache. Inside the context call `step(model, ids)` repeatedly."""
    dev = next(model.parameters()).device
    caches = []
    for attn in attention_modules(model):
        c = {"pos": 0}
        if hasattr(attn, "kv_down_proj"):
            c["latent"] = torch.empty(batch_size, max_len, attn.kv_latent_dim, device=dev, dtype=dtype)
        else:
            n_kv = getattr(attn, "n_kv_heads", attn.n_heads)
            shape = (batch_size, n_kv, max_len, attn.d_head)
            c["k"] = torch.empty(shape, device=dev, dtype=dtype)
            c["v"] = torch.empty(shape, device=dev, dtype=dtype)
        attn._kv_cache = c
        caches.append(c)
    full_pe = getattr(model, "pos_emb", None)
    model._fast_state = {"caches": caches, "full_pe": full_pe}
    try:
        yield
    finally:
        for attn in attention_modules(model):
            attn._kv_cache = None
        if full_pe is not None:
            model.pos_emb = full_pe
        del model._fast_state


def step(model, ids):
    """Run `ids` (B, T) through the model at the current cache position; returns logits (B, T, V)."""
    state = model._fast_state
    pos = state["caches"][0]["pos"]
    if state["full_pe"] is not None:
        # DecoderLM.forward adds self.pos_emb[:T]; shift the table so rows pos..pos+T are used
        model.pos_emb = state["full_pe"][pos:]
    logits = model(ids)
    for c in state["caches"]:
        c["pos"] += ids.shape[1]
    return logits


# ---------------------------------------------------------------------------
# sampling + word-budgeted generation
# ---------------------------------------------------------------------------

def sample_next(logits, temperature, top_k, top_p, banned_ids, prev_ids=None, repetition_penalty=1.0):
    logits = torch.nan_to_num(logits.float(), nan=-1e4)   # a diverged checkpoint must not crash sampling
    if banned_ids is not None:
        logits[:, banned_ids] = -float("inf")
    if repetition_penalty != 1.0 and prev_ids is not None:
        seen = torch.gather(logits, 1, prev_ids)
        seen = torch.where(seen > 0, seen / repetition_penalty, seen * repetition_penalty)
        logits = logits.scatter(1, prev_ids, seen)
    if not temperature:
        return logits.argmax(-1)
    logits = logits / temperature
    if top_k:
        kth = torch.topk(logits, top_k, dim=-1).values[:, -1:]
        logits = logits.masked_fill(logits < kth, -float("inf"))
    if top_p and top_p < 1.0:
        sorted_logits, idx = torch.sort(logits, descending=True)
        probs = sorted_logits.softmax(-1)
        drop = probs.cumsum(-1) - probs > top_p          # keep the smallest set with mass >= top_p
        sorted_logits = sorted_logits.masked_fill(drop, -float("inf"))
        logits = torch.full_like(logits, -float("inf")).scatter(1, idx, sorted_logits)
    return torch.multinomial(logits.softmax(-1), 1).squeeze(1)


def ngram_ban(seq, n, vocab):
    """(B, V) bool: tokens that would complete an n-gram (in token ids) already present in `seq` (B, S).
    For each row, every earlier occurrence of the last n-1 tokens bans the token that followed it."""
    B, S = seq.shape
    if n < 2 or S < n:
        return torch.zeros(B, vocab, dtype=torch.bool, device=seq.device)
    grams = seq.unfold(1, n, 1)                                   # (B, S-n+1, n)
    hit = (grams[:, :, :-1] == seq[:, None, S - n + 1:]).all(-1)  # earlier (n-1)-gram == current suffix
    cnt = torch.zeros(B, vocab, device=seq.device).scatter_add_(1, grams[:, :, -1], hit.float())
    return cnt > 0


@torch.inference_mode()
def generate_words(model, tokenizer, prompts, *, max_seq_len, target_words, max_new_tokens,
                   batch_size, temperature, top_k, top_p, repetition_penalty, device, amp_dtype,
                   log_every=10, no_repeat_ngram=0, ngram_scope="output"):
    """no_repeat_ngram = n > 1 forbids repeating any n-gram of TOKEN ids (HF no_repeat_ngram_size), counted over
    the generated text only (ngram_scope="output") or over prompt + generated text ("all")."""
    assert ngram_scope in ("output", "all"), ngram_scope
    """Continue each prompt by `target_words` whitespace words. Returns list[str] in input order.

    Prompts are bucketed by token length so a batch needs no padding: every prompt in a batch
    keeps its last L tokens, L = shortest prompt in the batch (capped so prompt + new tokens fit
    max_seq_len). Words are counted from the tokenizer's end-of-word flag, so we stop as soon as
    every row has produced target_words complete words.
    """
    model.eval()
    eow = torch.tensor(tokenizer.token_eow, device=device, dtype=torch.bool)
    has_bytes = torch.tensor([len(b) > 0 for b in tokenizer.token_bytes], device=device, dtype=torch.bool)
    banned = torch.arange(tokenizer.n_special, device=device)     # never emit pad/unk/</s>/...
    encoded = [tokenizer.encode(p) for p in prompts]
    order = sorted(range(len(prompts)), key=lambda i: len(encoded[i]))
    max_prompt = max_seq_len - max_new_tokens
    assert max_prompt > 0, "max_new_tokens must be smaller than max_seq_len"
    results = [None] * len(prompts)

    for b, start in enumerate(range(0, len(order), batch_size)):
        idxs = order[start:start + batch_size]
        L = min(min(len(encoded[i]) for i in idxs), max_prompt)
        ids = torch.tensor([encoded[i][-L:] for i in idxs], device=device, dtype=torch.long)
        B = len(idxs)
        words = torch.zeros(B, dtype=torch.int32, device=device)
        in_word = torch.zeros(B, dtype=torch.bool, device=device)     # prompts end on a full word
        done = torch.zeros(B, dtype=torch.bool, device=device)
        out = []
        with torch.autocast("cuda", dtype=amp_dtype, enabled=device.type == "cuda"), \
                kv_cache(model, B, L + max_new_tokens, amp_dtype if device.type == "cuda" else torch.float32):
            logits = step(model, ids)[:, -1]
            history = ids
            for t in range(max_new_tokens):
                # a bare </w> right after a finished word never occurs in encoded text: forbid it
                logits = logits.float().clone()
                logits[~in_word, tokenizer.eow_id] = -float("inf")
                logits[:, len(tokenizer.token_bytes):] = -float("inf")   # ids the tokenizer can't decode
                if no_repeat_ngram and no_repeat_ngram > 1:
                    seq = history if ngram_scope == "all" else (torch.stack(out, 1) if out else ids[:, :0])
                    ban = ngram_ban(seq, no_repeat_ngram, logits.size(1))
                    ban &= ~(ban | torch.isinf(logits)).all(-1, keepdim=True)  # never leave a row with no option
                    logits = logits.masked_fill(ban, -float("inf"))
                nxt = sample_next(logits, temperature, top_k, top_p, banned,
                                  history if repetition_penalty != 1.0 else None, repetition_penalty)
                out.append(nxt)
                # a word ends on an end-of-word token that closes something non-empty
                open_ = in_word | has_bytes[nxt]
                words += (eow[nxt] & open_ & ~done).int()
                in_word = open_ & ~eow[nxt]
                done |= words >= target_words
                if bool(done.all()):
                    break
                if repetition_penalty != 1.0 or (no_repeat_ngram and ngram_scope == "all"):
                    history = torch.cat([history, nxt[:, None]], 1)
                logits = step(model, nxt[:, None])[:, -1]
        gen = torch.stack(out, 1).tolist()
        for row, i in enumerate(idxs):
            text = tokenizer.decode(gen[row])
            results[i] = " ".join(text.split()[:target_words])
        if log_every and (b + 1) % log_every == 0:
            print(f"  generated {min(start + batch_size, len(order))}/{len(order)}", flush=True)
    return results


@torch.inference_mode()
def generate_beam(model, tokenizer, prompts, *, max_seq_len, target_words, max_new_tokens, batch_size,
                  beam_size, device, amp_dtype, no_repeat_ngram=0, ngram_scope="output", length_penalty=1.0,
                  log_every=10):
    """Beam search with the same word budget / token constraints as generate_words.
    Beams are ranked by sum(log p) / length**length_penalty (1.0 = mean log-prob per token). A beam that has
    produced `target_words` words is frozen (score and length fixed) and keeps competing as a finished
    hypothesis. Optional n-gram blocking per beam (same rule as generate_words). Returns list[str] in input order."""
    assert ngram_scope in ("output", "all"), ngram_scope
    model.eval()
    W = int(beam_size)
    eow = torch.tensor(tokenizer.token_eow, device=device, dtype=torch.bool)
    has_bytes = torch.tensor([len(b) > 0 for b in tokenizer.token_bytes], device=device, dtype=torch.bool)
    n_decodable = len(tokenizer.token_bytes)
    encoded = [tokenizer.encode(p) for p in prompts]
    order = sorted(range(len(prompts)), key=lambda i: len(encoded[i]))
    max_prompt = max_seq_len - max_new_tokens
    per_batch = max(1, batch_size // W)
    results = [None] * len(prompts)
    neg_inf = -float("inf")

    for b, start in enumerate(range(0, len(order), per_batch)):
        idxs = order[start:start + per_batch]
        P = len(idxs)
        R = P * W
        L = min(min(len(encoded[i]) for i in idxs), max_prompt)
        prompt_ids = torch.tensor([encoded[i][-L:] for i in idxs], device=device, dtype=torch.long)
        ids = prompt_ids.repeat_interleave(W, 0)                                   # (R, L): row = p * W + beam
        words = torch.zeros(R, dtype=torch.int32, device=device)
        in_word = torch.zeros(R, dtype=torch.bool, device=device)
        done = torch.zeros(R, dtype=torch.bool, device=device)
        cum = torch.zeros(R, device=device)
        length = torch.zeros(R, device=device)
        gen = torch.zeros(R, 0, dtype=torch.long, device=device)
        with torch.autocast("cuda", dtype=amp_dtype, enabled=device.type == "cuda"), \
                kv_cache(model, R, L + max_new_tokens, amp_dtype if device.type == "cuda" else torch.float32):
            logits = step(model, ids)[:, -1]
            for t in range(max_new_tokens):
                lp = torch.nan_to_num(logits.float(), nan=-1e4)
                lp[:, :tokenizer.n_special] = neg_inf
                lp[~in_word, tokenizer.eow_id] = neg_inf
                lp[:, n_decodable:] = neg_inf
                if no_repeat_ngram and no_repeat_ngram > 1:
                    seq = gen if ngram_scope == "output" else torch.cat([ids, gen], 1)
                    ban = ngram_ban(seq, no_repeat_ngram, lp.size(1))
                    ban &= ~(ban | torch.isinf(lp)).all(-1, keepdim=True)
                    lp = lp.masked_fill(ban, neg_inf)
                lp = lp.log_softmax(-1)
                V = lp.size(1)
                # candidate scores (length-normalised); finished beams keep their score via one frozen continuation
                cand = (cum[:, None] + lp) / (length[:, None] + 1).pow(length_penalty)
                frozen_tok = lp.argmax(-1)
                frozen = torch.full_like(cand, neg_inf)
                frozen.scatter_(1, frozen_tok[:, None], (cum / length.clamp(min=1).pow(length_penalty))[:, None])
                cand = torch.where(done[:, None], frozen, cand)
                if t == 0:                                                         # identical beams: expand beam 0 only
                    cand.view(P, W, V)[:, 1:] = neg_inf
                top_s, top_i = cand.view(P, W * V).topk(W, dim=-1)                 # (P, W)
                parent = (top_i // V) + torch.arange(P, device=device)[:, None] * W
                parent, tok = parent.reshape(-1), (top_i % V).reshape(-1)
                was_done = done[parent]
                # reorder beam state + KV cache to the chosen parents
                for c in model._fast_state["caches"]:
                    pos = c["pos"]
                    c["k"][:, :, :pos] = c["k"][parent, :, :pos]
                    c["v"][:, :, :pos] = c["v"][parent, :, :pos]
                words, in_word, done = words[parent], in_word[parent], done[parent].clone()
                cum = torch.where(was_done, cum[parent], cum[parent] + lp[parent, tok])
                length = torch.where(was_done, length[parent], length[parent] + 1)
                gen = torch.cat([gen[parent], tok[:, None]], 1)
                open_ = in_word | has_bytes[tok]
                words = words + (eow[tok] & open_ & ~was_done).int()
                in_word = open_ & ~eow[tok]
                done = was_done | (words >= target_words)
                if bool(done.all()):
                    break
                logits = step(model, tok[:, None])[:, -1]
        score = (cum / length.clamp(min=1).pow(length_penalty)).view(P, W)
        score = torch.where(done.view(P, W), score, score - 1e6)                     # prefer finished beams
        best = score.argmax(-1) + torch.arange(P, device=device) * W
        rows = gen[best].tolist()
        for row, i in enumerate(idxs):
            text = tokenizer.decode(rows[row])
            results[i] = " ".join(text.split()[:target_words])
        if log_every and (b + 1) % log_every == 0:
            print(f"  beam generated {min(start + per_batch, len(order))}/{len(order)}", flush=True)
    return results


# ---------------------------------------------------------------------------
# verification helpers (run in the notebook before training)
# ---------------------------------------------------------------------------

@torch.no_grad()
def check_equivalence(make_model, vocab_size, seq_len=48, rtol=1e-4):
    """Original forward vs patched forward vs KV-cached incremental decoding, fp32 on CPU.
    Returns max |difference| relative to the largest |logit| of the original model."""
    torch.manual_seed(0)
    ref = make_model().eval()
    fast = make_model().eval()
    fast.load_state_dict(ref.state_dict())
    patch_model(fast)
    idx = torch.randint(0, vocab_size, (2, seq_len))
    a = ref(idx)
    scale = a.abs().max().item()
    err_full = (a - fast(idx)).abs().max().item() / scale
    with kv_cache(fast, 2, seq_len, torch.float32):
        pre = step(fast, idx[:, :seq_len // 2])
        steps = [step(fast, idx[:, t:t + 1]) for t in range(seq_len // 2, seq_len)]
    err_cache = (a - torch.cat([pre] + steps, 1)).abs().max().item() / scale
    assert err_full < rtol and err_cache < rtol, (err_full, err_cache)
    return err_full, err_cache

In [ ]:
%%writefile /kaggle/working/phase3/phase1_models/fastattn_cd.py
# Plumbing: fused SDPA attention + KV caches + generation around your #3 / #4 modules
"""Efficiency plumbing around the hand-written encoder-decoder (#3) and prefix-LM (#4) (written with Claude; declare it).

Nothing here changes an architecture: parameters, shapes and the maths of every model stay exactly as defined in
your files. `patch_model` swaps each attention module's forward for one that computes the *same* function with
F.scaled_dot_product_attention (memory-efficient fused kernel on T4), and adds optional caches for generation:

  encoder SelfAttention (non_causal_self_attention.py)  -> bidirectional, no cache (the encoder runs once per prompt)
  decoder CausalSelfAttention (attention3.py)           -> causal, self-attention KV cache
  CrossAttention (cross_attention.py)                   -> K/V of the encoder memory computed once per prompt, cached
  prefix CausalSelfAttention (prefix_attention.py)      -> mask = causal OR [:p, :p] block, KV cache for decoding

`check_equivalence_*` (run in the notebook before training) verify the patched model against your original forward,
and cached step-by-step decoding against a full forward pass.
"""
import sys
import types
from contextlib import contextmanager

import torch
import torch.nn.functional as F


# ---------------------------------------------------------------------------
# attention forwards
# ---------------------------------------------------------------------------

def _dropout_p(self):
    return self.attn_dropout.p if self.training else 0.0


def _project_self(self, x, start):
    """q, k, v of a self-attention module, RoPE at absolute positions start..start+T (as in your code)."""
    B, T, _ = x.shape
    q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
    k = self.k_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
    v = self.v_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
    if getattr(self, "use_rope", False):
        cos = self.rope_cos[start:start + T]
        sin = self.rope_sin[start:start + T]
        q = self._apply_rope(q.float(), cos, sin).to(v.dtype)
        k = self._apply_rope(k.float(), cos, sin).to(v.dtype)
    return q, k, v


def _use_cache(self, k, v, start):
    cache = getattr(self, "_kv_cache", None)
    if cache is None:
        return k, v
    T = k.shape[2]
    cache["k"][:, :, start:start + T] = k
    cache["v"][:, :, start:start + T] = v
    return cache["k"][:, :, :start + T], cache["v"][:, :, :start + T]


def _expand(self, k, v):
    if self.n_rep > 1:
        k = k.repeat_interleave(self.n_rep, dim=1)
        v = v.repeat_interleave(self.n_rep, dim=1)
    return k, v


def _finish(self, y):
    B, H, T, D = y.shape
    return self.resid_dropout(self.out_proj(y.transpose(1, 2).reshape(B, T, H * D)))


def _sdpa(self, q, k, v, attn_mask=None, is_causal=False):
    return F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask, is_causal=is_causal,
                                          dropout_p=_dropout_p(self), scale=self.attention_scale)


def _start(self):
    cache = getattr(self, "_kv_cache", None)
    return cache["pos"] if cache is not None else 0


def _prefix_hook(self, k, v, B):
    """prefix-tuning hook (Phase 3 plumbing): optional self.prefix_kv(B) -> (pk, pv), each (B, n_kv_heads, P, d_head),
    prepended to keys/values (no RoPE on them). Returns (k, v, P)."""
    fn = getattr(self, "prefix_kv", None)
    if fn is None:
        return k, v, 0
    assert getattr(self, "_kv_cache", None) is None, "prefix tuning is not supported with the generation KV cache"
    pk, pv = fn(B)
    assert pk.shape[0] == B and pk.shape[1] == k.shape[1] and pk.shape[3] == k.shape[3], \
        f"prefix_kv must return (B, n_kv_heads, P, d_head) = ({B}, {k.shape[1]}, P, {k.shape[3]}), got {tuple(pk.shape)}"
    return torch.cat([pk.to(k.dtype), k], dim=2), torch.cat([pv.to(v.dtype), v], dim=2), pk.shape[2]


def _with_prefix_cols(mask_TT, P):
    """(T, T) bool mask -> (T, P + T): the P prefix columns are visible to every query."""
    T = mask_TT.shape[0]
    return torch.cat([torch.ones(T, P, dtype=torch.bool, device=mask_TT.device), mask_TT], dim=1)


def _fwd_bidirectional(self, x):
    """encoder SelfAttention: no mask."""
    q, k, v = _project_self(self, x, 0)
    k, v, P = _prefix_hook(self, k, v, x.shape[0])
    k, v = _expand(self, k, v)
    return _finish(self, _sdpa(self, q, k, v))


def _fwd_causal(self, x):
    """decoder CausalSelfAttention: causal; one-token steps attend to the whole cache."""
    start = _start(self)
    q, k, v = _project_self(self, x, start)
    k, v = _use_cache(self, k, v, start)
    k, v, P = _prefix_hook(self, k, v, x.shape[0])
    k, v = _expand(self, k, v)
    T = x.shape[1]
    if T > 1 and start > 0:
        raise ValueError("multi-token steps are only supported from an empty cache")
    if P:
        mask = _with_prefix_cols(torch.ones(T, T, dtype=torch.bool, device=x.device).tril(), P)
        return _finish(self, _sdpa(self, q, k, v, attn_mask=mask))
    return _finish(self, _sdpa(self, q, k, v, is_causal=T > 1))


def _prefix_mask(self, T, p, device):
    key = (T, int(p), str(device))
    masks = self.__dict__.setdefault("_mask_cache", {})
    if key not in masks:
        m = torch.ones(T, T, dtype=torch.bool, device=device).tril()
        m[:p, :p] = True                       # same rule as your prefix_attention.py
        masks[key] = m
    return masks[key]


def _fwd_prefix(self, x, p):
    """prefix attention: causal OR [:p, :p]; one-token decoding steps attend to the whole cache."""
    start = _start(self)
    q, k, v = _project_self(self, x, start)
    k, v = _use_cache(self, k, v, start)
    k, v, P = _prefix_hook(self, k, v, x.shape[0])
    k, v = _expand(self, k, v)
    T = x.shape[1]
    if start == 0:
        mask = _prefix_mask(self, T, p, x.device)
        if P:
            mask = _with_prefix_cols(mask, P)
        return _finish(self, _sdpa(self, q, k, v, attn_mask=mask))
    if T == 1:
        return _finish(self, _sdpa(self, q, k, v))
    raise ValueError("multi-token steps are only supported from an empty cache")


def _fwd_cross(self, x, memory):
    """CrossAttention: q from the decoder state, k/v from the encoder memory (cached once per prompt)."""
    assert not getattr(self, "use_rope", False), "RoPE in cross-attention is not supported (your design: use_rope=False)"
    B, T, _ = x.shape
    q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
    cache = getattr(self, "_kv_cache", None)
    if cache is not None and "k" in cache:
        k, v = cache["k"], cache["v"]
    else:
        S = memory.shape[1]
        k = self.k_proj(memory).view(B, S, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(memory).view(B, S, self.n_kv_heads, self.d_head).transpose(1, 2)
        if cache is not None:
            cache["k"], cache["v"] = k, v
    k, v = _expand(self, k, v)
    return _finish(self, _sdpa(self, q, k, v))


_FORWARDS = {
    ("non_causal_self_attention", "SelfAttention"): _fwd_bidirectional,
    ("attention3", "CausalSelfAttention"): _fwd_causal,
    ("cross_attention", "CrossAttention"): _fwd_cross,
    ("prefix_attention", "CausalSelfAttention"): _fwd_prefix,
}


def patch_model(model):
    """Swap every attention forward for the fused/cached version (same parameters)."""
    n = 0
    for mod in model.modules():
        fwd = _FORWARDS.get((type(mod).__module__, type(mod).__name__))
        if fwd is None:
            continue
        mod._kv_cache = None
        if getattr(mod, "use_rope", False):
            mod._apply_rope = sys.modules[type(mod).__module__].apply_rope   # your own apply_rope
        mod.forward = types.MethodType(fwd, mod)
        n += 1
    assert n > 0, "no attention module recognised; did a module/class name change?"
    return model


# ---------------------------------------------------------------------------
# generation sessions: prefill once, then one token per step
# ---------------------------------------------------------------------------

def _alloc_self_cache(attn, B, max_len, dtype, device):
    shape = (B, attn.n_kv_heads, max_len, attn.d_head)
    attn._kv_cache = {"pos": 0, "k": torch.empty(shape, device=device, dtype=dtype),
                      "v": torch.empty(shape, device=device, dtype=dtype)}
    return attn._kv_cache


@contextmanager
def prefix_session(model, ids, max_new_tokens, dtype):
    """#4: prefill the prompt (mask uses the model's own cfg.prefix), then decode with the KV cache."""
    B, L = ids.shape
    dev = ids.device
    caches = [_alloc_self_cache(b.attn, B, L + max_new_tokens, dtype, dev) for b in model.blocks]
    full_pe = getattr(model, "pos_emb", None)

    def run(tok):
        pos = caches[0]["pos"]
        if full_pe is not None:
            model.pos_emb = full_pe[pos:]     # DecoderLM adds pos_emb[:T]; shift it to rows pos..pos+T
        logits = model(tok)
        for c in caches:
            c["pos"] += tok.shape[1]
        return logits[:, -1]

    try:
        yield run(ids), run
    finally:
        for b in model.blocks:
            b.attn._kv_cache = None
        if full_pe is not None:
            model.pos_emb = full_pe


@contextmanager
def encdec_session(model, ids, max_new_tokens, dtype, start_id=None):
    """#3: encode the prompt once, cache every decoder layer's cross-attention K/V, decode token by token.
    start_id=None -> the decoder starts from the last source token, else from start_id (e.g. <s>)."""
    B = ids.shape[0]
    dev = ids.device
    memory = model.encode(ids)
    caches = [_alloc_self_cache(b.attn, B, max_new_tokens + 1, dtype, dev) for b in model.decblocks]
    for b in model.decblocks:
        b.crattn._kv_cache = {}
    full_pe = getattr(model, "pos_emb_dec", None)

    def run(tok):
        pos = caches[0]["pos"]
        if full_pe is not None:
            model.pos_emb_dec = full_pe[pos:]
        logits = model.decode(tok, memory)
        for c in caches:
            c["pos"] += tok.shape[1]
        return logits[:, -1]

    first = ids[:, -1:] if start_id is None else torch.full((B, 1), start_id, device=dev, dtype=ids.dtype)
    try:
        yield run(first), run
    finally:
        for b in model.decblocks:
            b.attn._kv_cache = None
            b.crattn._kv_cache = None
        if full_pe is not None:
            model.pos_emb_dec = full_pe


# ---------------------------------------------------------------------------
# sampling + word-budgeted generation
# ---------------------------------------------------------------------------

def sample_next(logits, temperature, top_k, top_p, banned_ids, prev_ids=None, repetition_penalty=1.0):
    logits = torch.nan_to_num(logits.float(), nan=-1e4)   # a diverged checkpoint must not crash sampling
    if banned_ids is not None:
        logits[:, banned_ids] = -float("inf")
    if repetition_penalty != 1.0 and prev_ids is not None:
        seen = torch.gather(logits, 1, prev_ids)
        seen = torch.where(seen > 0, seen / repetition_penalty, seen * repetition_penalty)
        logits = logits.scatter(1, prev_ids, seen)
    if not temperature:
        return logits.argmax(-1)
    logits = logits / temperature
    if top_k:
        kth = torch.topk(logits, top_k, dim=-1).values[:, -1:]
        logits = logits.masked_fill(logits < kth, -float("inf"))
    if top_p and top_p < 1.0:
        sorted_logits, idx = torch.sort(logits, descending=True)
        probs = sorted_logits.softmax(-1)
        drop = probs.cumsum(-1) - probs > top_p
        sorted_logits = sorted_logits.masked_fill(drop, -float("inf"))
        logits = torch.full_like(logits, -float("inf")).scatter(1, idx, sorted_logits)
    return torch.multinomial(logits.softmax(-1), 1).squeeze(1)


@torch.inference_mode()
def generate_words(model, tokenizer, prompts, *, kind, max_prompt, target_words, max_new_tokens, batch_size,
                   temperature, top_k, top_p, repetition_penalty, device, amp_dtype, start_id=None, log_every=10):
    """Continue each prompt by `target_words` whitespace words. Returns list[str] in input order.

    kind = "prefix" (#4) or "encdec" (#3). Prompts are bucketed by token length so a batch needs no padding: each
    prompt keeps its last L tokens, L = shortest prompt in the batch (capped at max_prompt)."""
    model.eval()
    eow = torch.tensor(tokenizer.token_eow, device=device, dtype=torch.bool)
    has_bytes = torch.tensor([len(b) > 0 for b in tokenizer.token_bytes], device=device, dtype=torch.bool)
    banned = torch.arange(tokenizer.n_special, device=device)     # never emit pad/unk/</s>/...
    encoded = [tokenizer.encode(p) for p in prompts]
    order = sorted(range(len(prompts)), key=lambda i: len(encoded[i]))
    results = [None] * len(prompts)
    cache_dtype = amp_dtype if device.type == "cuda" else torch.float32

    for b, start in enumerate(range(0, len(order), batch_size)):
        idxs = order[start:start + batch_size]
        L = min(min(len(encoded[i]) for i in idxs), max_prompt)
        ids = torch.tensor([encoded[i][-L:] for i in idxs], device=device, dtype=torch.long)
        B = len(idxs)
        words = torch.zeros(B, dtype=torch.int32, device=device)
        in_word = torch.zeros(B, dtype=torch.bool, device=device)     # prompts end on a full word
        done = torch.zeros(B, dtype=torch.bool, device=device)
        out = []
        session = (prefix_session(model, ids, max_new_tokens, cache_dtype) if kind == "prefix"
                   else encdec_session(model, ids, max_new_tokens, cache_dtype, start_id))
        with torch.autocast("cuda", dtype=amp_dtype, enabled=device.type == "cuda"), session as (logits, step):
            history = ids
            for t in range(max_new_tokens):
                logits = logits.float().clone()
                logits[~in_word, tokenizer.eow_id] = -float("inf")       # a bare </w> after a finished word
                logits[:, len(tokenizer.token_bytes):] = -float("inf")   # ids the tokenizer can't decode
                nxt = sample_next(logits, temperature, top_k, top_p, banned,
                                  history if repetition_penalty != 1.0 else None, repetition_penalty)
                out.append(nxt)
                open_ = in_word | has_bytes[nxt]
                words += (eow[nxt] & open_ & ~done).int()
                in_word = open_ & ~eow[nxt]
                done |= words >= target_words
                if bool(done.all()) or t == max_new_tokens - 1:
                    break
                if repetition_penalty != 1.0:
                    history = torch.cat([history, nxt[:, None]], 1)
                logits = step(nxt[:, None])
        gen = torch.stack(out, 1).tolist()
        for row, i in enumerate(idxs):
            text = tokenizer.decode(gen[row])
            results[i] = " ".join(text.split()[:target_words])
        if log_every and (b + 1) % log_every == 0:
            print(f"  generated {min(start + batch_size, len(order))}/{len(order)}", flush=True)
    return results


# ---------------------------------------------------------------------------
# verification helpers (run in the notebook before training)
# ---------------------------------------------------------------------------

@torch.no_grad()
def check_equivalence_encdec(make_model, vocab_size, src_len=20, tgt_len=12, rtol=1e-4):
    """Your forward vs patched forward vs the generation path (encdec_session: encode once, cached cross-attention
    K/V, one decoder token per step). Returns relative max errors (fp32, CPU)."""
    torch.manual_seed(0)
    ref = make_model().eval()
    fast = make_model().eval()
    fast.load_state_dict(ref.state_dict())
    patch_model(fast)
    src = torch.randint(0, vocab_size, (2, src_len))
    tgt = torch.randint(0, vocab_size, (2, tgt_len))
    tgt[:, 0] = src[:, -1]                    # session with start_id=None starts from the last source token
    a = ref(src, tgt)
    scale = a.abs().max().item()
    err_full = (a - fast(src, tgt)).abs().max().item() / scale
    with encdec_session(fast, src, tgt_len, torch.float32, start_id=None) as (first, step):
        outs = [first[:, None]]
        for t in range(1, tgt_len):
            outs.append(step(tgt[:, t:t + 1])[:, None])
    err_cache = (a - torch.cat(outs, 1)).abs().max().item() / scale
    assert err_full < rtol and err_cache < rtol, (err_full, err_cache)
    return err_full, err_cache


@torch.no_grad()
def check_equivalence_prefix(make_model, vocab_size, prefix, seq_len=None, rtol=1e-4):
    """Your forward vs patched forward vs (prefill >= prefix tokens, then cached one-token steps)."""
    torch.manual_seed(0)
    ref = make_model().eval()
    fast = make_model().eval()
    fast.load_state_dict(ref.state_dict())
    patch_model(fast)
    seq_len = seq_len or prefix + 12
    idx = torch.randint(0, vocab_size, (2, seq_len))
    a = ref(idx)
    scale = a.abs().max().item()
    err_full = (a - fast(idx)).abs().max().item() / scale
    n_new = seq_len - prefix
    with prefix_session(fast, idx[:, :prefix], n_new, torch.float32) as (first, step):
        outs = [first[:, None]]
        for t in range(prefix, seq_len - 1):
            outs.append(step(idx[:, t:t + 1])[:, None])
    err_cache = (a[:, prefix - 1:seq_len - 1] - torch.cat(outs, 1)).abs().max().item() / scale
    assert err_full < rtol and err_cache < rtol, (err_full, err_cache)
    return err_full, err_cache

In [ ]:
%%writefile /kaggle/working/phase3/fetch_data.py
#!/usr/bin/env python3
"""Download Phase_3_RR/ from the gated Hugging Face dataset (plumbing; declare it).

Token lookup order: Kaggle secret "HF" -> HF_TOKEN environment variable -> hidden prompt.
The token is used for these requests only: never printed, never written to disk.

  python fetch_data.py                      # -> ./data/Phase_3_RR/...
  python fetch_data.py --out /kaggle/working/data
"""
import argparse
import getpass
import json
import os
import urllib.error
import urllib.request

REPO = "https://huggingface.co/datasets/Exploration-Lab/CS-781-Capstone/resolve/main/"
FILES = [
    "Phase_3_RR/Train/CL_train.json",
    "Phase_3_RR/Train/CL_dev.json",
    "Phase_3_RR/Train/IT_train.json",
    "Phase_3_RR/Train/IT_dev.json",
    "Phase_3_RR/Train/label_vocab.json",
    "Phase_3_RR/Test/CL_test.json",
    "Phase_3_RR/Test/IT_test.json",
]


def get_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF")
    except Exception:
        pass
    return os.environ.get("HF_TOKEN") or getpass.getpass("Hugging Face token (input hidden): ").strip()


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", default=os.path.join(os.path.dirname(os.path.abspath(__file__)), "data"))
    args = ap.parse_args()
    if all(os.path.exists(os.path.join(args.out, rel)) for rel in FILES):
        print("all Phase-3 files already present in", args.out)
        return
    token = get_token()
    try:
        for rel in FILES:
            dst = os.path.join(args.out, rel)
            if os.path.exists(dst):
                print("exists:", dst)
                continue
            req = urllib.request.Request(REPO + rel, headers={"Authorization": f"Bearer {token}"})
            try:
                with urllib.request.urlopen(req, timeout=120) as r:
                    data = r.read()
            except urllib.error.HTTPError as e:
                hint = "check the token / dataset access" if e.code in (401, 403) else e.reason
                raise SystemExit(f"download failed for {rel}: HTTP {e.code} ({hint})")
            if rel.endswith(".json"):
                json.loads(data)                       # validate before saving
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            with open(dst, "wb") as f:
                f.write(data)
            print(f"saved {dst} ({len(data) / 1e6:.1f} MB)")
    finally:
        del token


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/phase3/rr_data.py
"""Phase 3 (Rhetorical Roles) data: documents, labels, sentence encoding, context windows (plumbing; declare it).

A document is an ordered list of sentences (+ one label per sentence for train/dev). The model predicts exactly one
label per sentence. Sentences are encoded separately with your ByteBPE and then packed into windows:

  window = [left-context sentences] [TARGET sentences] [right-context sentences]      (<= window_len tokens)

Only TARGET sentences are classified in a window; every sentence of a document is the target of exactly one window
(checked by check_windows()), so predictions come out complete and in the original order.

context config (cfg["context"], your choices):
  window_len        max tokens per window (<= backbone max_seq_len)
  max_sent_tokens   a sentence longer than this is truncated to its first max_sent_tokens tokens
  max_targets       max target sentences per window (1 = classic sentence-level classification)
  target_tokens     max tokens of the target group (>= 1 sentence is always taken)
  left_tokens       left-context budget in tokens (0 = none)
  right_tokens      right-context budget in tokens (0 = none)
  sep_token         True: put <sep> between sentences (the pooled span excludes it)
"""
import json
import os

import numpy as np

PAD_ID, UNK_ID, SEP_ID = 0, 1, 5          # a1_tokenizer SPECIALS: <pad>=0, <unk>=1, <sep>=5


def data_root(root):
    for cand in (root, os.path.join(root, "Phase_3_RR")):
        if os.path.isdir(os.path.join(cand, "Train")):
            return cand
    raise FileNotFoundError(f"no Phase_3_RR/Train under {root} (run fetch_data.py first)")


def load_labels(root):
    v = json.load(open(os.path.join(data_root(root), "Train", "label_vocab.json"), encoding="utf-8"))
    if isinstance(v, dict):                          # {label: index} or {"labels": [...]}
        if "labels" in v and isinstance(v["labels"], list):
            return list(v["labels"])
        return [k for k, _ in sorted(v.items(), key=lambda kv: kv[1])]
    return list(v)


def load_docs(path):
    """-> list of {"id", "domain"?, "sentences": [...], "labels": [...] | None} in file order."""
    data = json.load(open(path, encoding="utf-8"))
    if isinstance(data, dict):                        # {doc_id: {...}} or {doc_id: [sentences]}
        items = []
        for k, v in data.items():
            if isinstance(v, dict):
                v = dict(v, id=v.get("id", k))
            else:
                v = {"id": k, "text": v}
            items.append(v)
        data = items
    docs = []
    for d in data:
        sents = d.get("text", d.get("sentences"))
        labels = d.get("labels", d.get("label"))
        assert isinstance(sents, list), f"document {d.get('id')}: 'text' is not a list of sentences"
        if labels is not None:
            assert len(labels) == len(sents), f"document {d.get('id')}: {len(labels)} labels for {len(sents)} sentences"
        docs.append({"id": str(d.get("id")), "sentences": [str(s) for s in sents], "labels": labels})
    return docs


def load_domain(root, domain, split):
    sub = "Test" if split == "test" else "Train"
    docs = load_docs(os.path.join(data_root(root), sub, f"{domain}_{split}.json"))
    for d in docs:
        d["domain"] = domain
    return docs


# ---------------------------------------------------------------------------
# encoding + windows
# ---------------------------------------------------------------------------

def encode_doc(tok, doc, max_sent_tokens):
    """Token ids per sentence; an empty sentence becomes [<unk>] so it still gets a representation."""
    out = []
    for s in doc["sentences"]:
        ids = tok.encode(s)[:max_sent_tokens]
        out.append(ids if ids else [UNK_ID])
    return out


def build_windows(sent_ids, ctx):
    """-> list of windows: {"ids": [...], "targets": [(sent_index, start, end)], } with end exclusive."""
    n = len(sent_ids)
    sep = [SEP_ID] if ctx["sep_token"] else []
    lens = [len(x) + len(sep) for x in sent_ids]
    windows, i = [], 0
    while i < n:
        # target group: >= 1 sentence, within target_tokens / max_targets / window_len
        j, tlen = i, 0
        while j < n and (j == i or (tlen + lens[j] <= ctx["target_tokens"]
                                    and (ctx["max_targets"] is None or j - i < ctx["max_targets"])
                                    and tlen + lens[j] <= ctx["window_len"])):
            tlen += lens[j]
            j += 1
        budget = ctx["window_len"] - tlen
        # left context (closest sentences first), then right context, within their budgets and the window
        left, ltoks, k = [], 0, i - 1
        while k >= 0 and ltoks + lens[k] <= min(ctx["left_tokens"], budget):
            left.insert(0, k)
            ltoks += lens[k]
            k -= 1
        budget -= ltoks
        right, rtoks, k = [], 0, j
        while k < n and rtoks + lens[k] <= min(ctx["right_tokens"], budget):
            right.append(k)
            rtoks += lens[k]
            k += 1
        ids, targets = [], []
        for si in left + list(range(i, j)) + right:
            start = len(ids)
            ids.extend(sent_ids[si])
            if i <= si < j:
                targets.append((si, start, start + len(sent_ids[si])))
            ids.extend(sep)
        windows.append({"ids": ids, "targets": targets})
        i = j
    return windows


def check_windows(n_sentences, windows, window_len):
    seen = [t[0] for w in windows for t in w["targets"]]
    assert seen == list(range(n_sentences)), "every sentence must be a target exactly once, in order"
    assert all(len(w["ids"]) <= window_len for w in windows), "window longer than window_len"
    assert all(e > s for w in windows for _, s, e in w["targets"]), "empty target span"


def label_counts(docs, labels):
    c = {l: 0 for l in labels}
    for d in docs:
        for l in d["labels"] or []:
            c[l] = c.get(l, 0) + 1
    return c

In [ ]:
%%writefile /kaggle/working/phase3/rr_metrics.py
"""Sentence-level rhetorical-role metrics (plumbing; declare it). Counts only: interpretation is yours (rule 9).

macro over labels: per-label P/R/F1 averaged over the labels that occur in gold or predictions (the usual
scikit-learn behaviour; a label with gold but no predictions counts as F1 0). Also micro-F1 (= accuracy here),
per-label support, and the confusion matrix (gold -> predicted counts).
"""
from collections import Counter


def rr_scores(gold, pred, labels):
    """gold, pred: {doc_id: [label, ...]} with equal lengths per doc."""
    tp, fp, fn, support = Counter(), Counter(), Counter(), Counter()
    conf = Counter()
    n = correct = 0
    for d, g in gold.items():
        p = pred[d]
        assert len(p) == len(g), f"doc {d}: {len(p)} predictions for {len(g)} sentences"
        for a, b in zip(g, p):
            n += 1
            support[a] += 1
            conf[(a, b)] += 1
            if a == b:
                tp[a] += 1
                correct += 1
            else:
                fp[b] += 1
                fn[a] += 1
    present = [l for l in labels if support[l] or tp[l] or fp[l]]
    per = {}
    for l in present:
        p = tp[l] / (tp[l] + fp[l]) if tp[l] + fp[l] else 0.0
        r = tp[l] / (tp[l] + fn[l]) if tp[l] + fn[l] else 0.0
        f = 2 * p * r / (p + r) if p + r else 0.0
        per[l] = dict(p=round(p, 4), r=round(r, 4), f1=round(f, 4), support=support[l], tp=tp[l], fp=fp[l], fn=fn[l])
    k = max(1, len(per))
    return {
        "macro_f1": round(sum(v["f1"] for v in per.values()) / k, 4),
        "macro_p": round(sum(v["p"] for v in per.values()) / k, 4),
        "macro_r": round(sum(v["r"] for v in per.values()) / k, 4),
        "micro_f1": round(correct / max(1, n), 4),
        "sentences": n,
        "per_label": per,
        "confusion": {f"{a}->{b}": c for (a, b), c in conf.most_common() if a != b},
    }

In [ ]:
%%writefile /kaggle/working/phase3/backbones.py
"""Phase-1 checkpoints -> token feature extractors for L-NER (plumbing; declare it).

Your Phase-1 model code is used unchanged (phase1_models/: arch1/arch3 for A/B, enc_dec + parts for C,
prefix_lm + prefix_attention for D). The LM head is replaced by nn.Identity, so YOUR forward() returns the
final-LayerNorm hidden states (B, T, d_model) instead of vocabulary logits. Attention runs through the same
fastattn patches as Phase 1 (fused SDPA, identical maths; smoke_test.py re-checks equality).

Backbone options (cfg["backbone"]):
  prefix         D only: "full" -> prefix = window length (every token sees the whole window),
                 int p -> bidirectional over the first p tokens, causal after (your Phase-1 D used 640),
                 0 -> fully causal.
  encdec_mode    C only: "encoder" -> encoder states only (decoder weights dropped; bidirectional),
                 "decoder" -> the window is fed to the encoder AND to the decoder (causal self-attention +
                 cross-attention to the encoder states); features = decoder final states.
  truncate_layers keep only the first k transformer blocks (e.g. a smaller KD student); None = all.
  dropout        override every nn.Dropout p in the backbone; None = keep the checkpoint's value.
"""
import importlib
import os
import sys

import torch
import torch.nn as nn

HERE = os.path.dirname(os.path.abspath(__file__))
for _p in (HERE, os.path.join(HERE, "phase1_models")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

import fastattn_ab   # noqa: E402  (A/B attention: arch1 / arch3)
import fastattn_cd   # noqa: E402  (C/D attention: encoder / decoder / cross / prefix)


def _clean_state(sd):
    out = {}
    for k, v in sd.items():
        for pre in ("_orig_mod.", "module."):
            if k.startswith(pre):
                k = k[len(pre):]
        out[k] = v
    return out


def build_lm(spec):
    """spec = the 'config' dict saved inside a Phase-1 best.pt -> (lm, kind)."""
    kind = spec.get("kind", "decoder")
    arch = importlib.import_module(spec["module"])
    if kind == "decoder":
        lm = arch.DecoderLM(arch.LMConfig(**spec["model"]))
    elif kind == "prefix":
        lm = arch.DecoderLM(arch.LMConfig(**spec["model"]))
    elif kind == "encdec":
        lm = arch.EncDecLM(arch.EncLMConfig(**spec["encoder"]), arch.DecLMConfig(**spec["decoder"]))
    else:
        raise ValueError(f"unknown model kind {kind}")
    return lm, kind


def load_phase1(path, load_weights=True):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    spec = ck["config"]
    lm, kind = build_lm(spec)
    if load_weights:
        lm.load_state_dict(_clean_state(ck["model"]))
    info = dict(path=path, kind=kind, module=spec["module"], tag=spec.get("tag"), step=ck.get("step"),
                best_val=ck.get("best_val"))
    return lm, kind, spec, info


class Backbone(nn.Module):
    """forward(ids (B, T) long) -> hidden states (B, T, d_model).
    Attributes for technique code: .lm (your Phase-1 module), .kind, .blocks (list of transformer blocks,
    input -> output order; C "decoder" mode = encoder blocks then decoder blocks), .d_model, .max_len."""

    def __init__(self, lm, kind, prefix=None, encdec_mode=None, truncate_layers=None, dropout=None):
        super().__init__()
        self.kind = kind
        if kind == "encdec":
            assert encdec_mode in ("encoder", "decoder"), "C needs backbone.encdec_mode = 'encoder' | 'decoder'"
            self.encdec_mode = encdec_mode
            if encdec_mode == "encoder":
                for name in ("decblocks", "tok_emb2", "ln_f_dec", "head"):
                    delattr(lm, name)
                lm.ln_f_dec = lm.head = None
            else:
                lm.head = nn.Identity()
            self.d_model = lm.cfg1.d_model
            self.max_len = lm.cfg1.max_seq_len if encdec_mode == "encoder" else min(lm.cfg1.max_seq_len,
                                                                                    lm.cfg2.max_seq_len)
        else:
            if kind == "prefix":
                assert prefix is not None, "D needs backbone.prefix = 'full' | int (0 = causal)"
            self.prefix = prefix
            lm.head = nn.Identity()
            self.d_model = lm.cfg.d_model
            self.max_len = lm.cfg.max_seq_len
        self.lm = lm
        self.n_virtual_tokens = 0          # prompt tuning (Phase 3): set by your technique; see forward()
        if truncate_layers:
            self._truncate(int(truncate_layers))
        if dropout is not None:
            for m in self.modules():
                if isinstance(m, nn.Dropout):
                    m.p = float(dropout)

    @property
    def blocks(self):
        lm = self.lm
        if self.kind == "encdec":
            return list(lm.encblocks) + (list(lm.decblocks) if self.encdec_mode == "decoder" else [])
        return list(lm.blocks)

    def _truncate(self, k):
        lm = self.lm
        if self.kind == "encdec":
            assert self.encdec_mode == "encoder", "truncate_layers for C is only supported in encoder mode"
            lm.encblocks = nn.ModuleList(list(lm.encblocks)[:k])
        else:
            lm.blocks = nn.ModuleList(list(lm.blocks)[:k])

    # ---- prompt-tuning hook (Phase 3 plumbing) ------------------------------------------------------------------
    # If a technique sets self.n_virtual_tokens = P > 0, forward() prepends P placeholder ids (<pad>) to every window
    # and drops the first P output states again, so callers always get (B, T, d) for their real tokens. The technique
    # makes those P positions learnable by wrapping the input embedding (get_input_embedding / set_input_embedding):
    # its wrapper returns the normal embeddings but replaces positions [:, :P] with its learned prompt vectors.
    # Positional encodings / masks see a sequence of length P + T, so windows must satisfy P + T <= max_len.
    def get_input_embedding(self):
        return self.lm.tok_emb1 if self.kind == "encdec" else self.lm.tok_emb

    def set_input_embedding(self, module):
        if self.kind == "encdec":
            self.lm.tok_emb1 = module      # encoder input (the decoder in "decoder" mode keeps its own embedding)
        else:
            self.lm.tok_emb = module

    def forward(self, ids):
        P = self.n_virtual_tokens
        if P:
            ids = torch.cat([torch.zeros(ids.shape[0], P, dtype=ids.dtype, device=ids.device), ids], dim=1)
        out = self._forward(ids)
        return out[:, P:] if P else out

    def _forward(self, ids):
        lm = self.lm
        if self.kind == "decoder":
            return lm(ids)
        if self.kind == "prefix":
            lm.cfg.prefix = ids.shape[1] if self.prefix == "full" else int(self.prefix) + self.n_virtual_tokens
            return lm(ids)
        memory = lm.encode(ids)
        if self.encdec_mode == "encoder":
            return memory
        return lm.decode(ids, memory)


def patch_fast_attention(lm, kind):
    if kind == "decoder":
        fastattn_ab.patch_model(lm)
    else:
        fastattn_cd.patch_model(lm)
    return lm


def build_backbone(ckpt_path, bcfg, vocab_size=None, load_weights=True, fast_attention=True):
    lm, kind, spec, info = load_phase1(ckpt_path, load_weights)
    if vocab_size is not None:
        v = spec["model"]["vocab_size"] if "model" in spec else spec["encoder"]["vocab_size"]
        assert v == vocab_size, f"{ckpt_path}: vocab {v} != tokenizer vocab {vocab_size} (different tokenizer?)"
    if fast_attention:
        patch_fast_attention(lm, kind)
    bb = Backbone(lm, kind, prefix=bcfg.get("prefix"), encdec_mode=bcfg.get("encdec_mode"),
                  truncate_layers=bcfg.get("truncate_layers"), dropout=bcfg.get("dropout"))
    return bb, spec, info

In [ ]:
%%writefile /kaggle/working/phase3/inject.py
"""Generic module-surgery helpers for your fine-tuning techniques (plumbing; declare it).

These only find / replace / freeze modules; they contain no adaptation method. Typical use in your files:

    from inject import freeze, replace_linears, wrap_modules
    freeze(backbone)                                            # base weights frozen
    replace_linears(backbone, lambda name, lin: name.endswith(("q_proj", "v_proj")),
                    lambda name, lin: YourWrapper(lin, ...))    # your LoRA / LoRMA module around nn.Linear
    wrap_modules(backbone, lambda name, m: name.endswith(".ffn"),
                 lambda name, m: YourAdapterWrapper(m, ...))    # your adapter after a sublayer

Linear layer names in the Phase-1 models (relative to backbone, e.g. "lm.blocks.3.attn.q_proj"):
  A/B/D blocks:  attn.q_proj, attn.k_proj, attn.v_proj, attn.out_proj, ffn.net.0 (d->d_ff), ffn.net.2 (d_ff->d)
  C encoder:     lm.encblocks.<i>.attn.{q,k,v,out}_proj, lm.encblocks.<i>.ffn.net.{0,2}
  C decoder:     lm.decblocks.<i>.attn.*, lm.decblocks.<i>.crattn.{q,k,v,out}_proj, lm.decblocks.<i>.ffn.net.{0,2}
Attention projections have no bias (bias=False); FFN linears and LayerNorms do.
Sublayer call signatures: attn(x) (D: attn(x, p)), crattn(x, memory), ffn(x) -> a wrapper should take
(*args, **kwargs) and pass them through to the wrapped module.
"""
import torch.nn as nn


def freeze(module):
    for p in module.parameters():
        p.requires_grad_(False)
    return module


def unfreeze(module):
    for p in module.parameters():
        p.requires_grad_(True)
    return module


def _set_child(root, name, new):
    parent_name, _, child = name.rpartition(".")
    parent = root.get_submodule(parent_name) if parent_name else root
    setattr(parent, child, new)


def list_linears(root):
    return [name for name, m in root.named_modules() if isinstance(m, nn.Linear)]


def replace_linears(root, match, factory):
    """Replace every nn.Linear for which match(name, linear) is True by factory(name, linear). -> replaced names."""
    targets = [(n, m) for n, m in root.named_modules() if isinstance(m, nn.Linear) and match(n, m)]
    for name, lin in targets:
        _set_child(root, name, factory(name, lin))
    return [n for n, _ in targets]


def wrap_modules(root, match, factory):
    """Replace every submodule for which match(name, module) is True by factory(name, module). -> replaced names.
    Matches are collected first, so a wrapper containing the original module is not matched again."""
    targets = [(n, m) for n, m in root.named_modules() if n and match(n, m)]
    for name, mod in targets:
        _set_child(root, name, factory(name, mod))
    return [n for n, _ in targets]


def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {"total": total, "trainable": trainable, "trainable_pct": round(100 * trainable / max(1, total), 4)}

In [ ]:
%%writefile /kaggle/working/phase3/techniques/__init__.py
"""Technique registry (plumbing). The config's technique.name selects techniques/<name>.py.

Contract for every technique file (YOUR code, course rule 9):
    apply(backbone, **cfg) -> None
        Called once, after the Phase-1 weights are loaded and before the optimizer is built. On entry EVERY backbone
        parameter has requires_grad=True. On exit, requires_grad must be True exactly on what should train
        (plus any modules you inserted). The head is separate and always trained.
        backbone: backbones.Backbone (.lm = your Phase-1 module, .blocks, .kind, .d_model). See inject.py for
        freeze / replace_linears / wrap_modules helpers and the linear-layer names.
    optional: param_groups(model, base_lr) -> list of optimizer param-group dicts (e.g. a different lr for your
        inserted modules). If absent, every trainable backbone parameter uses train.lr and the head uses
        train.head_lr (or train.lr).
"""
import importlib


def get(name):
    return importlib.import_module(f"techniques.{name}")

In [ ]:
%%writefile /kaggle/working/phase3/techniques/full_ft.py
"""Full fine-tuning: every backbone parameter trains (plumbing-level: nothing to implement)."""


def apply(backbone, **cfg):
    assert not cfg, f"full_ft takes no options, got {cfg}"
    for p in backbone.parameters():
        p.requires_grad_(True)

In [ ]:
%%writefile /kaggle/working/phase3/train_rr.py
#!/usr/bin/env python3
"""Fine-tune one Phase-1 model for Rhetorical Role prediction and write its Codabench submission (plumbing; declare it).

  python train_rr.py run_config.json

Pipeline: CL/IT documents -> sentences encoded with your ByteBPE -> context windows (rr_data.build_windows)
-> Phase-1 backbone (backbones) + YOUR technique (techniques/<name>.py) -> one pooled vector per target sentence
-> YOUR head (heads.py) -> one label per sentence. AdamW (fp16 autocast on GPU), macro-F1 on the official CL/IT dev
sets after every epoch, best checkpoint -> dev report -> test predictions for CL_test + IT_test -> prediction.json + zip.

Outputs in <out_dir>/<name>/: config.json, labels.json, log.jsonl, best_model.pt, dev_scores.json (all / CL / IT),
dev_predictions.json, prediction.json, submission_<name>.zip, summary.json (scores, params, memory, timings),
dev_probs.json / test_probs.json: {"labels": [label order], "decode": "argmax" | "head.decode",
  "probs": {doc_id: [[p_label0, ..., p_label12] per sentence, same order as the predictions]}} = softmax of the head's
  per-sentence logits of the best model (for soft voting / thresholds; with a CRF head the predictions come from
  head.decode (Viterbi), so they need not equal the argmax of these probabilities).

Head contract (your heads.py, unchanged from Phase 2): build_head(d_model, n_labels, name, **cfg); head(h) maps
(1, N, d_model) sentence vectors -> (1, N, n_labels) logits. Optional head.loss(logits, labels) / head.decode(logits)
are used per window (training) and per document (prediction), so a CRF head runs over the sentence sequence.

Technique contract (techniques/<name>.py): apply(backbone, **cfg) as in Phase 2, plus OPTIONAL hooks:
  extra_loss(model) -> scalar added to the training loss every micro-batch      (e.g. AdaLoRA's orthogonality term)
  on_step(model, step, total_steps) -> None, called after every optimizer step, BEFORE gradients are zeroed
                                                                                 (e.g. AdaLoRA's rank budget update)
  param_groups(model, base_lr) -> optimizer param groups                          (as in Phase 2)
"""
import glob
import inspect
import json
import math
import os
import random
import sys
import time
import zipfile
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)

import backbones                       # noqa: E402
import heads                           # noqa: E402
import rr_data                         # noqa: E402
import techniques                      # noqa: E402
from a1_tokenizer import ByteBPE       # noqa: E402
from inject import count_params        # noqa: E402
from rr_metrics import rr_scores       # noqa: E402

DOMAINS = ("CL", "IT")
REQUIRED = {
    "context": ["window_len", "max_sent_tokens", "target_tokens", "left_tokens", "right_tokens", "sep_token"],
    "train": ["epochs", "batch_size", "micro_batch", "lr", "weight_decay", "warmup_frac", "schedule", "grad_clip"],
}


def check_config(cfg):
    missing = [f"{sec}.{k}" for sec, keys in REQUIRED.items() for k in keys if cfg.get(sec, {}).get(k) is None]
    for key in ("pooling",):
        if cfg.get(key) is None:
            missing.append(key)
    if cfg.get("head", {}).get("name") is None:
        missing.append("head.name")
    if cfg.get("technique", {}).get("name") is None:
        missing.append("technique.name")
    assert not missing, "fill these config values first: " + ", ".join(missing)
    c, tr = cfg["context"], cfg["train"]
    assert cfg["pooling"] in ("mean", "last", "first", "max"), "pooling: mean | last | first | max"
    assert c["max_sent_tokens"] <= c["window_len"], "max_sent_tokens must be <= window_len"
    assert tr["batch_size"] % tr["micro_batch"] == 0, "train.batch_size must be a multiple of train.micro_batch"
    assert tr["schedule"] in ("linear", "cosine", "constant")
    assert set(cfg.get("train_domains", DOMAINS)) <= set(DOMAINS)
    tech = cfg["technique"]
    unset = [f"technique.{k}" for k, v in tech.items() if v is None]
    if cfg.get("kd"):
        unset += [f"kd.{k}" for k, v in cfg["kd"].items() if v is None]
    assert not unset, "fill these config values first: " + ", ".join(unset)
    sig = inspect.signature(techniques.get(tech["name"]).apply)
    try:
        sig.bind(None, **{k: v for k, v in tech.items() if k != "name"})
    except TypeError as e:
        raise AssertionError(f"technique '{tech['name']}' options do not fit apply{sig}: {e}")


class Logger:
    def __init__(self, path):
        self.path = path

    def __call__(self, msg, **kv):
        rec = {"t": round(time.time(), 1), "msg": msg, **kv}
        print(json.dumps(rec), flush=True)
        with open(self.path, "a") as f:
            f.write(json.dumps(rec) + "\n")


# ---------------------------------------------------------------------------
# model: backbone -> pooled target-sentence vectors -> head
# ---------------------------------------------------------------------------

def pool(h, rows, starts, ends, how):
    """h: (B, T, d); one vector per (row, start, end) span -> (N, d)."""
    out = []
    for r, s, e in zip(rows, starts, ends):
        seg = h[r, s:e]
        if how == "mean":
            out.append(seg.mean(0))
        elif how == "last":
            out.append(seg[-1])
        elif how == "first":
            out.append(seg[0])
        else:
            out.append(seg.max(0).values)
    return torch.stack(out)


class RRModel(nn.Module):
    def __init__(self, backbone, head, pooling):
        super().__init__()
        self.backbone, self.head, self.pooling = backbone, head, pooling

    def sentence_logits(self, ids, rows, starts, ends):
        h = self.backbone(ids)
        vec = pool(h, rows, starts, ends, self.pooling)
        return self.head(vec.unsqueeze(0)).squeeze(0)          # (N, n_labels)


def build_model(cfg, n_labels, vocab_size, load_weights=True):
    bb, spec, info = backbones.build_backbone(cfg["phase1_ckpt"], cfg.get("backbone", {}), vocab_size,
                                              load_weights=load_weights, fast_attention=cfg.get("fast_attention", True))
    for p in bb.parameters():
        p.requires_grad_(True)
    tech = cfg["technique"]
    tmod = techniques.get(tech["name"])
    tmod.apply(bb, **{k: v for k, v in tech.items() if k != "name"})
    if not cfg.get("fast_attention", True):           # the prefix_kv hook lives only in the fused attention
        assert not any(hasattr(m, "prefix_kv") for m in bb.modules()), "prefix tuning needs fast_attention: true"
    head = heads.build_head(bb.d_model, n_labels, **cfg["head"])
    for p in head.parameters():
        p.requires_grad_(True)
    return RRModel(bb, head, cfg["pooling"]), tmod, info


def save_model(model, path):
    torch.save(model.state_dict(), path + ".tmp")
    os.replace(path + ".tmp", path)


def load_run(run_dir, device, vocab_size):
    cfg = json.load(open(os.path.join(run_dir, "config.json")))
    labels = json.load(open(os.path.join(run_dir, "labels.json")))
    if not os.path.exists(cfg["phase1_ckpt"]):       # run from an earlier session: same checkpoint, new input path
        tail = os.path.join(*cfg["phase1_ckpt"].split(os.sep)[-2:])          # e.g. run_B2/best.pt
        hits = sorted(glob.glob(os.path.join("/kaggle/input", "**", tail), recursive=True))
        assert hits, f"{run_dir}: Phase-1 checkpoint {cfg['phase1_ckpt']} not found (attach it; looked for */{tail})"
        cfg["phase1_ckpt"] = hits[0]
    model, _, _ = build_model(cfg, len(labels), vocab_size, load_weights=False)
    model.load_state_dict(torch.load(os.path.join(run_dir, "best_model.pt"), map_location="cpu"))
    return model.to(device).eval(), cfg, labels


# ---------------------------------------------------------------------------
# data -> windows -> micro-batches
# ---------------------------------------------------------------------------

def prepare(docs, tok, ctx, lab2id):
    """-> list of windows {"doc", "ids", "targets": [(sent_idx, s, e)], "labels": [...]}, checked complete."""
    out = []
    for d in docs:
        sent_ids = rr_data.encode_doc(tok, d, ctx["max_sent_tokens"])
        wins = rr_data.build_windows(sent_ids, ctx)
        rr_data.check_windows(len(sent_ids), wins, ctx["window_len"])
        for w in wins:
            w["doc"] = d["id"]
            w["labels"] = [lab2id[d["labels"][si]] for si, _, _ in w["targets"]] if d["labels"] is not None else None
            out.append(w)
    return out


def batches(wins, micro, shuffle, rng):
    """Length-bucketed micro-batches (similar lengths together -> little padding)."""
    order = sorted(range(len(wins)), key=lambda i: len(wins[i]["ids"]))
    chunks = [order[i:i + micro] for i in range(0, len(order), micro)]
    if shuffle:
        rng.shuffle(chunks)
    return chunks


def collate(wins, idx, device):
    L = max(len(wins[i]["ids"]) for i in idx)
    ids = torch.full((len(idx), L), rr_data.PAD_ID, dtype=torch.long)
    rows, starts, ends, labels, per_win = [], [], [], [], []
    for r, i in enumerate(idx):
        w = wins[i]
        ids[r, :len(w["ids"])] = torch.tensor(w["ids"])
        per_win.append(len(w["targets"]))
        for k, (_, s, e) in enumerate(w["targets"]):
            rows.append(r); starts.append(s); ends.append(e)
        if w["labels"] is not None:
            labels.extend(w["labels"])
    lab = torch.tensor(labels, dtype=torch.long, device=device) if labels else None
    return ids.to(device), rows, starts, ends, lab, per_win


def class_weights(wins, n, mode, none_weight, none_id):
    if mode is None and none_weight is None:
        return None
    w = torch.ones(n)
    if mode is not None:
        c = np.bincount([l for x in wins for l in x["labels"]], minlength=n).astype(np.float64)
        c = np.maximum(c, 1.0)
        w = torch.tensor((c.sum() / c) ** {"inv_freq": 1.0, "inv_sqrt": 0.5}[mode], dtype=torch.float32)
        w = w / w.mean()
    if none_weight is not None and none_id is not None:
        w[none_id] = float(none_weight)
    return w


@torch.no_grad()
def predict(model, wins, docs, micro, device, amp, id2lab, probs_out=None):
    """-> {doc_id: [label, ...]} with exactly one label per sentence, in order.
    probs_out (optional dict) is filled with {doc_id: [[softmax probabilities per class] per sentence]}."""
    model.eval()
    per_doc = {d["id"]: [None] * len(d["sentences"]) for d in docs}
    for idx in batches(wins, micro, False, None):
        ids, rows, starts, ends, _, _ = collate(wins, idx, device)
        with torch.autocast("cuda", dtype=torch.float16, enabled=amp):
            lg = model.sentence_logits(ids, rows, starts, ends).float().cpu()
        k = 0
        for i in idx:
            for si, _, _ in wins[i]["targets"]:
                per_doc[wins[i]["doc"]][si] = lg[k]
                k += 1
    dec = getattr(model.head, "decode", None)
    out = {}
    for d in docs:
        L = torch.stack(per_doc[d["id"]])
        if probs_out is not None:
            probs_out[d["id"]] = torch.softmax(L, dim=-1).tolist()
        tags = dec(L.unsqueeze(0).to(device)).squeeze(0).cpu().tolist() if dec is not None else L.argmax(-1).tolist()
        out[d["id"]] = [id2lab[int(t)] for t in tags]
        assert len(out[d["id"]]) == len(d["sentences"])
    return out


def lr_factor(step, total, warmup, schedule):
    if warmup and step < warmup:
        return (step + 1) / warmup
    if schedule == "constant":
        return 1.0
    prog = min(1.0, (step - warmup) / max(1, total - warmup))
    return 1.0 - prog if schedule == "linear" else 0.5 * (1 + math.cos(math.pi * prog))


def score_domains(gold_docs, pred, labels):
    res = {"all": rr_scores({d["id"]: d["labels"] for d in gold_docs}, pred, labels)}
    for dom in DOMAINS:
        g = {d["id"]: d["labels"] for d in gold_docs if d["domain"] == dom}
        if g:
            res[dom] = rr_scores(g, {k: pred[k] for k in g}, labels)
    return res


# ---------------------------------------------------------------------------
# main
# ---------------------------------------------------------------------------

def main(cfg_path):
    cfg = json.load(open(cfg_path))
    check_config(cfg)
    name, tr, ctx = cfg["name"], cfg["train"], cfg["context"]
    out = os.path.join(cfg["out_dir"], name)
    os.makedirs(out, exist_ok=True)
    log = Logger(os.path.join(out, "log.jsonl"))
    json.dump(cfg, open(os.path.join(out, "config.json"), "w"), indent=2)
    seed = tr.get("seed", 0)
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    amp = device.type == "cuda"
    t_start = time.time()
    deadline = t_start + 60 * tr["max_minutes"] if tr.get("max_minutes") else None

    # ---- data
    tok = ByteBPE.from_json(cfg["tokenizer"])
    labels = rr_data.load_labels(cfg["data_dir"])
    lab2id, id2lab = {l: i for i, l in enumerate(labels)}, dict(enumerate(labels))
    json.dump(labels, open(os.path.join(out, "labels.json"), "w"))
    train_docs = [d for dom in cfg.get("train_domains", DOMAINS) for d in rr_data.load_domain(cfg["data_dir"], dom, "train")]
    dev_docs = [d for dom in DOMAINS for d in rr_data.load_domain(cfg["data_dir"], dom, "dev")]
    test_docs = [d for dom in DOMAINS for d in rr_data.load_domain(cfg["data_dir"], dom, "test")]
    assert len({d["id"] for d in test_docs}) == len(test_docs), "test document ids collide between CL and IT"
    smoke = cfg.get("smoke")                 # {"steps": n}: real model + real windows, tiny data (not submittable)
    if smoke:
        train_docs = train_docs[:4]
        dev_docs = [d for dom in DOMAINS for d in [x for x in dev_docs if x["domain"] == dom][:1]]
        test_docs = [d for dom in DOMAINS for d in [x for x in test_docs if x["domain"] == dom][:1]]
        tr["epochs"] = 1
    unknown = sorted({l for d in train_docs + dev_docs for l in d["labels"]} - set(labels))
    assert not unknown, f"labels not in label_vocab.json: {unknown}"
    tr_w, dev_w, te_w = prepare(train_docs, tok, ctx, lab2id), prepare(dev_docs, tok, ctx, lab2id), prepare(test_docs, tok, ctx, lab2id)
    log("data", train_docs=len(train_docs), dev_docs=len(dev_docs), test_docs=len(test_docs),
        train_sentences=sum(len(d["sentences"]) for d in train_docs), train_windows=len(tr_w),
        mean_targets_per_window=round(np.mean([len(w["targets"]) for w in tr_w]), 2),
        mean_window_tokens=round(np.mean([len(w["ids"]) for w in tr_w]), 1), labels=len(labels))

    # ---- model
    model, tmod, info = build_model(cfg, len(labels), tok.get_vocab_size())
    model.to(device)
    P = model.backbone.n_virtual_tokens
    assert ctx["window_len"] + P <= model.backbone.max_len, \
        f"window_len {ctx['window_len']} + {P} virtual tokens > backbone max_seq_len {model.backbone.max_len}"
    params = count_params(model)
    params["head"] = sum(p.numel() for p in model.head.parameters())
    log("model", **info, **params, virtual_tokens=P)

    teacher = None
    if cfg.get("kd"):
        kd = dict(cfg["kd"])
        teacher, tcfg, tlabels = load_run(kd.pop("teacher_run"), device, tok.get_vocab_size())
        assert tlabels == labels, "teacher and student must share label_vocab"
        assert ctx["window_len"] + teacher.backbone.n_virtual_tokens <= teacher.backbone.max_len
        t_par = count_params(teacher)["total"]
        assert params["total"] < t_par, f"KD student ({params['total']:,}) must be smaller than the teacher ({t_par:,})"
        for p in teacher.parameters():
            p.requires_grad_(False)
        kd_loss = techniques.get("kd").kd_loss
        pass_w = kd.pop("pass_class_weights", False)
        log("teacher", run=tcfg["name"], params=t_par, student_params=params["total"])

    # ---- optimizer
    if hasattr(tmod, "param_groups"):
        groups = tmod.param_groups(model, tr["lr"])
    else:
        groups = [{"params": [p for p in model.backbone.parameters() if p.requires_grad], "lr": tr["lr"]},
                  {"params": list(model.head.parameters()), "lr": tr.get("head_lr") or tr["lr"]}]
    for g in groups:
        g["params"] = list(g["params"])
        g["base_lr"] = g["lr"]
    groups = [g for g in groups if g["params"]]
    opt = torch.optim.AdamW(groups, weight_decay=tr["weight_decay"])
    scaler = torch.amp.GradScaler("cuda", enabled=amp)
    cw = class_weights(tr_w, len(labels), tr.get("class_weights"), tr.get("none_weight"), lab2id.get("None"))
    if cw is not None:
        cw = cw.to(device)
        log("class_weights", weights={l: round(float(x), 3) for l, x in zip(labels, cw)})
    if teacher is not None and pass_w:
        assert cw is not None, "kd.pass_class_weights needs train.class_weights"
        kd["weight"] = cw
    extra_loss, on_step = getattr(tmod, "extra_loss", None), getattr(tmod, "on_step", None)

    rng = random.Random(seed)
    micro, accum = tr["micro_batch"], tr["batch_size"] // tr["micro_batch"]
    steps_per_epoch = math.ceil(len(batches(tr_w, micro, False, None)) / accum)
    total = steps_per_epoch * tr["epochs"]
    warmup = int(tr["warmup_frac"] * total)
    log("schedule", steps_per_epoch=steps_per_epoch, total_steps=total, warmup_steps=warmup)

    best, best_epoch, bad, step, seen = -1.0, -1, 0, 0, 0
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    t_train, stop = time.time(), False
    for epoch in range(1, tr["epochs"] + 1):
        model.train()
        mbs = batches(tr_w, micro, True, rng)
        run_loss, n_loss, t_ep = 0.0, 0, time.time()
        for i in range(0, len(mbs), accum):
            group = mbs[i:i + accum]
            for g in opt.param_groups:
                g["lr"] = g["base_lr"] * lr_factor(step, total, warmup, tr["schedule"])
            for idx in group:
                ids, rows, starts, ends, lab, per_win = collate(tr_w, idx, device)
                with torch.autocast("cuda", dtype=torch.float16, enabled=amp):
                    logits = model.sentence_logits(ids, rows, starts, ends)
                    if teacher is not None:
                        with torch.no_grad():
                            t_logits = teacher.sentence_logits(ids, rows, starts, ends)
                logits = logits.float()
                if teacher is not None:
                    loss = kd_loss(logits.unsqueeze(0), t_logits.float().unsqueeze(0), lab.unsqueeze(0), **kd)
                elif hasattr(model.head, "loss"):                 # e.g. CRF: one sequence per window
                    parts, k = [], 0
                    for n in per_win:
                        parts.append(model.head.loss(logits[k:k + n].unsqueeze(0), lab[k:k + n].unsqueeze(0)))
                        k += n
                    loss = torch.stack(parts).mean()
                else:
                    loss = F.cross_entropy(logits, lab, weight=cw)
                if extra_loss is not None:
                    loss = loss + extra_loss(model)
                scaler.scale(loss / len(group)).backward()
                run_loss += loss.item(); n_loss += 1; seen += len(rows)
            if tr["grad_clip"]:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_([p for g in opt.param_groups for p in g["params"]], tr["grad_clip"])
            scaler.step(opt); scaler.update()
            step += 1
            if on_step is not None:          # before zero_grad: your hook can still read p.grad (AdaLoRA importance)
                on_step(model, step, total)
            opt.zero_grad(set_to_none=True)
            if smoke and step >= smoke["steps"]:
                stop = True
                break
            if step % tr.get("log_every", 20) == 0:
                log("train", epoch=epoch, step=step, loss=round(run_loss / max(1, n_loss), 4),
                    lr=opt.param_groups[0]["lr"], sent_s=round(seen / (time.time() - t_train)))
            if deadline and time.time() > deadline:
                log("deadline reached: stopping training", step=step)
                stop = True
                break
        rec = dict(epoch=epoch, step=step, train_loss=round(run_loss / max(1, n_loss), 4), epoch_s=round(time.time() - t_ep, 1))
        t_ev = time.time()
        sc = score_domains(dev_docs, predict(model, dev_w, dev_docs, tr.get("infer_micro_batch") or micro * 2, device, amp, id2lab), labels)
        sel = sc["all"]["macro_f1"]
        rec.update(dev_macro_f1=sel, dev_CL=sc.get("CL", {}).get("macro_f1"), dev_IT=sc.get("IT", {}).get("macro_f1"),
                   eval_s=round(time.time() - t_ev, 1))
        if sel > best:
            best, best_epoch, bad = sel, epoch, 0
            save_model(model, os.path.join(out, "best_model.pt"))
            rec["saved"] = True
        else:
            bad += 1
        log("epoch", **rec)
        if stop or (tr.get("patience") and bad >= tr["patience"]):
            break
    train_s = time.time() - t_train
    peak = torch.cuda.max_memory_allocated() / 1e9 if device.type == "cuda" else None

    # ---- best model -> dev report + test predictions
    model.load_state_dict(torch.load(os.path.join(out, "best_model.pt"), map_location="cpu"))
    im = tr.get("infer_micro_batch") or micro * 2
    dev_probs, test_probs = {}, {}
    dev_pred = predict(model, dev_w, dev_docs, im, device, amp, id2lab, probs_out=dev_probs)
    dev_sc = score_domains(dev_docs, dev_pred, labels)
    json.dump(dev_sc, open(os.path.join(out, "dev_scores.json"), "w"), indent=2)
    json.dump(dev_pred, open(os.path.join(out, "dev_predictions.json"), "w"))
    t_inf = time.time()
    test_pred = predict(model, te_w, test_docs, im, device, amp, id2lab, probs_out=test_probs)
    inf_s = time.time() - t_inf
    decode = "head.decode" if getattr(model.head, "decode", None) is not None else "argmax"
    for fn, pr in (("dev_probs.json", dev_probs), ("test_probs.json", test_probs)):
        json.dump({"labels": labels, "decode": decode, "probs": pr}, open(os.path.join(out, fn), "w"))
    for d in test_docs:
        assert len(test_pred[d["id"]]) == len(d["sentences"]) and set(test_pred[d["id"]]) <= set(labels)
    pred_path = os.path.join(out, "prediction.json")
    json.dump(test_pred, open(pred_path, "w"))
    zip_path = os.path.join(out, f"submission_{name}.zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(pred_path, arcname="prediction.json")
    summary = dict(name=name, phase1=info, technique=cfg["technique"], head=cfg["head"], pooling=cfg["pooling"],
                   context=ctx, kd=bool(cfg.get("kd")), params=params, best_epoch=best_epoch,
                   dev_macro_f1=dev_sc["all"]["macro_f1"], dev_macro_p=dev_sc["all"]["macro_p"],
                   dev_macro_r=dev_sc["all"]["macro_r"], dev_CL_macro_f1=dev_sc.get("CL", {}).get("macro_f1"),
                   dev_IT_macro_f1=dev_sc.get("IT", {}).get("macro_f1"), train_seconds=round(train_s, 1),
                   peak_mem_GB=round(peak, 2) if peak else None, test_infer_seconds=round(inf_s, 1),
                   test_sentences=sum(len(d["sentences"]) for d in test_docs), total_seconds=round(time.time() - t_start, 1))
    json.dump(summary, open(os.path.join(out, "summary.json"), "w"), indent=2)
    log("done", **{k: v for k, v in summary.items() if k not in ("phase1", "context")}, zip=zip_path)


if __name__ == "__main__":
    main(sys.argv[1])

In [ ]:
%%writefile /kaggle/working/phase3/run_many.py
#!/usr/bin/env python3
"""Run several train_rr.py configs over the available GPUs IN PARALLEL (plumbing; declare it).

  python run_many.py --gpus 0,1 configs/B_full.json configs/C_full.json ...

One worker per GPU; every worker runs one training process at a time on its own GPU (CUDA_VISIBLE_DEVICES), so with
--gpus 0,1 two fine-tuning runs train simultaneously, one per T4. Scheduling uses a SHARED queue: whenever a GPU becomes
free it takes the first runnable config in list order (no GPU sits idle while work remains). A config with "kd" is only
runnable once its teacher has written summary.json; until then the free GPU takes the next job instead. If the teacher
fails (or is not part of this batch and has no summary.json), the KD run is marked FAILED instead of waiting forever.
With --stop-at UNIX_TIME a run is only STARTED if it can finish before then (its train.max_minutes + --margin minutes,
or --default-minutes when max_minutes is unset); runs that no longer fit are marked SKIPPED, so the session limit never
kills a run halfway.

Logs: <out_dir>/<name>.stdout (+ <out_dir>/<name>/log.jsonl). A status line every --every seconds, a summary at the end.
Exit code 1 if any run failed.
"""
import argparse
import json
import os
import subprocess
import sys
import threading
import time

HERE = os.path.dirname(os.path.abspath(__file__))


class Scheduler:
    def __init__(self, cfg_paths, stop_at=None, margin=10, default_minutes=60):
        self.stop_at, self.margin, self.default_minutes = stop_at, margin, default_minutes
        self.lock = threading.Lock()
        self.cfgs = {}
        for p in cfg_paths:
            c = json.load(open(p))
            self.cfgs[c["name"]] = (p, c)
        self.pending = list(self.cfgs)                        # names, in the order given
        self.status = {n: "queued" for n in self.pending}
        self.started = {}

    def _teacher(self, cfg):
        if not cfg.get("kd"):
            return None, None
        d = cfg["kd"]["teacher_run"].rstrip("/")
        return os.path.basename(d), os.path.join(d, "summary.json")

    def next_job(self):
        """-> (name, path, cfg) to run now | "wait" (only KD runs whose teacher is still training) | None (nothing left)."""
        with self.lock:
            if not self.pending:
                return None
            for name in list(self.pending):
                path, cfg = self.cfgs[name]
                teacher, done = self._teacher(cfg)
                if teacher is not None and not os.path.exists(done):
                    t_status = self.status.get(teacher)
                    if t_status is None or t_status.startswith(("FAILED", "SKIPPED", "done")):
                        self.pending.remove(name)
                        self.status[name] = f"FAILED (teacher {teacher} has no summary.json)"
                        continue
                    self.status[name] = f"waiting for teacher {teacher}"
                    continue
                if self.stop_at is not None:
                    need = ((cfg.get("train", {}).get("max_minutes") or self.default_minutes) + self.margin) * 60
                    if time.time() + need > self.stop_at:
                        self.pending.remove(name)
                        self.status[name] = f"SKIPPED (needs ~{need / 60:.0f} min, session limit too close)"
                        continue
                self.pending.remove(name)
                return name, path, cfg
            return "wait" if self.pending else None

    def worker(self, gpu):
        while True:
            job = self.next_job()
            if job is None:
                return
            if job == "wait":
                time.sleep(10)
                continue
            name, path, cfg = job
            os.makedirs(cfg["out_dir"], exist_ok=True)
            env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED="1", OMP_NUM_THREADS="2")
            with self.lock:
                self.status[name] = f"running on GPU {gpu}"
                self.started[name] = time.time()
            with open(os.path.join(cfg["out_dir"], f"{name}.stdout"), "w") as log:
                rc = subprocess.call([sys.executable, os.path.join(HERE, "train_rr.py"), path], env=env,
                                     stdout=log, stderr=subprocess.STDOUT, cwd=os.getcwd())
            mins = (time.time() - self.started[name]) / 60
            with self.lock:
                self.status[name] = f"done on GPU {gpu} in {mins:.1f} min" if rc == 0 else \
                    f"FAILED (exit {rc}) on GPU {gpu} after {mins:.1f} min"


def last_log_line(cfg):
    logf = os.path.join(cfg["out_dir"], cfg["name"], "log.jsonl")
    if not os.path.exists(logf):
        return None
    lines = open(logf).read().strip().splitlines()
    return lines[-1][:250] if lines else None


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("configs", nargs="+")
    ap.add_argument("--gpus", default="0")
    ap.add_argument("--every", type=int, default=300)
    ap.add_argument("--stop-at", type=float, default=None, help="unix time: do not start runs that cannot finish by then")
    ap.add_argument("--margin", type=float, default=10, help="minutes added to max_minutes for evaluation + prediction")
    ap.add_argument("--default-minutes", type=float, default=60, help="assumed run length when max_minutes is unset")
    args = ap.parse_args()
    gpus = [g for g in args.gpus.split(",") if g != ""]
    sched = Scheduler(args.configs, args.stop_at, args.margin, args.default_minutes)
    print(f"{len(sched.cfgs)} runs on GPUs {gpus} ({len(gpus)} in parallel): {list(sched.cfgs)}", flush=True)
    threads = [threading.Thread(target=sched.worker, args=(g,), daemon=True, name=f"gpu{g}") for g in gpus]
    for t in threads:
        t.start()
    t0 = time.time()
    while any(t.is_alive() for t in threads):
        for t in threads:
            t.join(timeout=args.every / len(threads))
        with sched.lock:
            status = dict(sched.status)
        print(f"\n{time.strftime('%H:%M:%S')} (+{(time.time() - t0) / 60:.1f} min)", flush=True)
        for name, st in status.items():
            line = last_log_line(sched.cfgs[name][1]) if st.startswith("running") else None
            print(f"  {name:<20} {st}" + (f"\n      {line}" if line else ""), flush=True)

    print("\nsummary")
    failed = []
    for name, (path, cfg) in sched.cfgs.items():
        sp = os.path.join(cfg["out_dir"], name, "summary.json")
        if os.path.exists(sp):
            s = json.load(open(sp))
            print(f"  {name:<20} dev macro-F1 {s.get('dev_macro_f1')} (CL {s.get('dev_CL_macro_f1')}, IT {s.get('dev_IT_macro_f1')})  trainable {s['params']['trainable_pct']}%  "
                  f"train {s['train_seconds'] / 60:.1f} min  peak {s['peak_mem_GB']} GB  ({sched.status[name]})")
        else:
            print(f"  {name:<20} no summary ({sched.status[name]})")
        if not sched.status[name].startswith("done"):
            failed.append(name)
    if failed:
        sys.exit(f"failed runs: {failed} (see <out_dir>/<name>.stdout)")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/phase3/check_pipeline_rr.py
#!/usr/bin/env python3
"""Oracle check of the RR prediction path (plumbing; declare it). No model involved.

For several context settings: build the windows for every train/dev/test document (each sentence must be a target
exactly once, in order, windows within window_len), then feed the GOLD labels through train_rr.predict() (same
batching, collation, per-document assembly) and the prediction.json formatting. Dev macro-F1 must be 1.0 and every
prediction list must have exactly one label per sentence.

  python check_pipeline_rr.py --data data --tokenizer ../phase1/ab_inputs/legal_bpe.json
"""
import argparse
import json
import os
import sys

import torch
import torch.nn as nn

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import rr_data                        # noqa: E402
import train_rr                       # noqa: E402
from a1_tokenizer import ByteBPE      # noqa: E402
from rr_metrics import rr_scores      # noqa: E402


class Oracle(nn.Module):
    """Returns one-hot logits of the gold labels, in the exact order train_rr.predict() asks for them."""

    def __init__(self, wins, micro, n_labels):
        super().__init__()
        self.queue = [l for idx in train_rr.batches(wins, micro, False, None) for i in idx for l in wins[i]["labels"]]
        self.n, self.pos, self.head = n_labels, 0, nn.Identity()

    def sentence_logits(self, ids, rows, starts, ends):
        k = len(rows)
        lab = torch.tensor(self.queue[self.pos:self.pos + k])
        self.pos += k
        return torch.nn.functional.one_hot(lab, self.n).float() * 10


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default="data")
    ap.add_argument("--tokenizer", required=True)
    args = ap.parse_args()
    tok = ByteBPE.from_json(args.tokenizer)
    labels = rr_data.load_labels(args.data)
    lab2id, id2lab = {l: i for i, l in enumerate(labels)}, dict(enumerate(labels))
    print(f"labels ({len(labels)}): {labels}")
    splits = {s: [d for dom in ("CL", "IT") for d in rr_data.load_domain(args.data, dom, s)] for s in ("train", "dev", "test")}
    for s, docs in splits.items():
        n = sum(len(d["sentences"]) for d in docs)
        print(f"{s}: {len(docs)} docs, {n} sentences" + (f", ids unique: {len({d['id'] for d in docs}) == len(docs)}"))
    settings = {
        "sentence-only": dict(window_len=128, max_sent_tokens=128, max_targets=1, target_tokens=128, left_tokens=0, right_tokens=0, sep_token=False),
        "packed, no context": dict(window_len=512, max_sent_tokens=256, max_targets=None, target_tokens=512, left_tokens=0, right_tokens=0, sep_token=True),
        "1 target + context": dict(window_len=512, max_sent_tokens=128, max_targets=1, target_tokens=128, left_tokens=192, right_tokens=192, sep_token=True),
        "packed + context": dict(window_len=768, max_sent_tokens=256, max_targets=None, target_tokens=384, left_tokens=192, right_tokens=192, sep_token=True),
    }
    for name, ctx in settings.items():
        wins = {s: train_rr.prepare(docs, tok, ctx, lab2id) for s, docs in splits.items()}   # includes check_windows
        dev = splits["dev"]
        pred = train_rr.predict(Oracle(wins["dev"], 8, len(labels)), wins["dev"], dev, 8, torch.device("cpu"), False, id2lab)
        pred = json.loads(json.dumps(pred))                                                  # through the JSON format
        sc = rr_scores({d["id"]: d["labels"] for d in dev}, pred, labels)
        ok = all(len(pred[d["id"]]) == len(d["sentences"]) for d in dev)
        trunc = sum(len(tok.encode(x)) > ctx["max_sent_tokens"] for d in splits["train"] for x in d["sentences"])
        print(f"{name:20s} windows train/dev/test {len(wins['train'])}/{len(wins['dev'])}/{len(wins['test'])} | "
              f"oracle dev macro-F1 {sc['macro_f1']} | counts match {ok} | train sentences truncated {trunc}")
        assert sc["macro_f1"] == 1.0 and ok


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/phase3/rr_ensemble.py
#!/usr/bin/env python3
"""Sentence-level voting ensemble of finished RR runs (plumbing; declare it). Members / weights are your choice.

  python rr_ensemble.py --name ENS --members runs/A runs/B runs/C [--weights 1 1 2] --out runs
  python rr_ensemble.py --name ENS --members a.zip b.zip c.zip --out ensembles      # test predictions only

For every sentence the label with the largest summed member weight wins; ties go to the label predicted by the
earliest-listed member among the tied ones. Run directories also give a dev score (from dev_predictions.json).
"""
import argparse
import json
import os
import sys
import zipfile
from collections import defaultdict

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))


def read_pred(path):
    if path.endswith(".zip"):
        with zipfile.ZipFile(path) as z:
            return json.loads(z.read("prediction.json"))
    return json.load(open(path))


def vote(preds, weights):
    out = {}
    for d in preds[0]:
        n = len(preds[0][d])
        assert all(len(p[d]) == n for p in preds), f"doc {d}: members disagree on the number of sentences"
        labs = []
        for i in range(n):
            score, first = defaultdict(float), {}
            for m, (p, w) in enumerate(zip(preds, weights)):
                score[p[d][i]] += w
                first.setdefault(p[d][i], m)
            best = max(score.values())
            labs.append(min((l for l, s in score.items() if s == best), key=lambda l: first[l]))
        out[d] = labs
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--name", required=True)
    ap.add_argument("--members", nargs="+", required=True)
    ap.add_argument("--weights", nargs="+", type=float, default=None)
    ap.add_argument("--out", required=True)
    args = ap.parse_args()
    w = args.weights or [1.0] * len(args.members)
    assert len(w) == len(args.members) >= 2
    zips = all(m.endswith(".zip") for m in args.members)
    test = vote([read_pred(m if zips else os.path.join(m, "prediction.json")) for m in args.members], w)
    out = os.path.join(args.out, args.name)
    os.makedirs(out, exist_ok=True)
    summary = {"name": args.name, "members": args.members, "weights": w}
    if not zips:
        import rr_data
        from rr_metrics import rr_scores
        cfg = json.load(open(os.path.join(args.members[0], "config.json")))
        labels = json.load(open(os.path.join(args.members[0], "labels.json")))
        dev = vote([json.load(open(os.path.join(m, "dev_predictions.json"))) for m in args.members], w)
        docs = [d for dom in ("CL", "IT") for d in rr_data.load_domain(cfg["data_dir"], dom, "dev")]
        if len(dev) < len(docs):                      # smoke runs predict only a few dev docs
            print(f"note: members predicted {len(dev)}/{len(docs)} dev docs; dev score over those only")
            docs = [d for d in docs if d["id"] in dev]
        sc = rr_scores({d["id"]: d["labels"] for d in docs}, dev, labels)
        json.dump(sc, open(os.path.join(out, "dev_scores.json"), "w"), indent=2)
        json.dump(dev, open(os.path.join(out, "dev_predictions.json"), "w"))
        summary.update(dev_macro_f1=sc["macro_f1"], dev_macro_p=sc["macro_p"], dev_macro_r=sc["macro_r"])
    pj = os.path.join(out, "prediction.json")
    json.dump(test, open(pj, "w"))
    zp = os.path.join(out, f"submission_{args.name}.zip")
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(pj, arcname="prediction.json")
    json.dump(summary, open(os.path.join(out, "summary.json"), "w"), indent=2)
    print(json.dumps(summary, indent=2)); print("zip:", zp)


if __name__ == "__main__":
    main()

## Stage 1b — YOUR files: head + techniques (contracts and instructions in each docstring)

In [ ]:
%%writefile /kaggle/working/phase3/heads.py
"""YOUR FILE: prediction heads for L-NER (course rule 9: model design is yours; plumbing only calls this).

Contract used by train_ner.py
-----------------------------
build_head(d_model, n_tags, name, **cfg) -> nn.Module    called with the config's "head" dict as keywords,
                                                           e.g. {"name": "linear"} or {"name": "custom", "hidden": ...}

head.forward(h) -> logits                                  h: (B, T, d_model) backbone states, logits: (B, T, n_tags)

Optional methods (used automatically when present):
head.loss(logits, tags) -> scalar      tags: (B, T) long, -100 = ignore. If absent: (class-weighted) cross-entropy.
                                       Implement this for e.g. a CRF head.
head.decode(logits) -> tag ids         logits: (1, N, n_tags) for one whole document (windows already merged),
                                       returns (1, N) long. If absent: argmax.

Tag ids: 0 = "O", then for each label in labels.txt order: B-, I- (BIO) or B-, I-, E-, S- (BIOES).
The head is always trained (requires_grad=True), whatever the technique.
"""
import torch
import torch.nn as nn
from torchcrf import CRF
class Linear_head(nn.Module):
    def __init__(self,d_model,n_tags,**cfg):
        super().__init__()
        self.in_features = d_model
        self.out_features = n_tags
        self.p = cfg.get("dropout",0.0)
        self.linear = nn.Linear(self.in_features,self.out_features)
        self.dropout = nn.Dropout(p=self.p)
    def forward(self,x:torch.Tensor)->torch.Tensor:
        B,T,C = x.shape
        x = self.dropout(x)
        logits = self.linear(x)
        return logits

class Mlp_head(nn.Module):
    def __init__(self,d_model,n_tags,**cfg):
        super().__init__()
        self.in_features = d_model
        self.out_features = n_tags
        self.hidden = cfg.get("hidden",256)
        self.activation = nn.GELU()
        self.p = cfg.get("dropout",0.0)
        self.linear1 = nn.Linear(self.in_features,self.hidden)
        self.linear2 = nn.Linear(self.hidden,self.out_features)
        self.dropout = nn.Dropout(p=self.p)
    def forward(self,x:torch.Tensor)->torch.Tensor:
        B,T,C = x.shape
        x = self.linear1(x)
        x = self.activation(x)
        x = self.dropout(x)
        logits = self.linear2(x)
        return logits

class Mlp_Crf__head(nn.Module):
    def __init__(self,d_model,n_tags,**cfg):
        super().__init__()
        self.in_features = d_model
        self.out_features = n_tags
        self.hidden = cfg.get("hidden",256)
        self.activation = nn.GELU()
        self.p = cfg.get("dropout",0.0)
        self.linear1 = nn.Linear(self.in_features,self.hidden)
        self.linear2 = nn.Linear(self.hidden,self.out_features)
        self.dropout = nn.Dropout(p=self.p)
        self.crf = CRF(n_tags, batch_first=True)
        
    def forward(self,x:torch.Tensor)->torch.Tensor:
        B,T,C = x.shape
        x = self.linear1(x)
        x = self.activation(x)
        x = self.dropout(x)
        logits = self.linear2(x)
        return logits
    def loss(self,logits:torch.Tensor ,tags:torch.Tensor):
        # need to use this logits and find crossentropy error
        mask = tags!=-100
        safe_tags=tags.masked_fill(~mask,0)
        log_likelihood = self.crf(logits,safe_tags,mask=mask,reduction="mean")
        return -log_likelihood
    def decode(self, logits):
        mask = torch.ones(
            logits.shape[:2],
        dtype=torch.bool,
        device=logits.device
    )

        predictions = self.crf.decode(logits, mask=mask)

        return torch.tensor(
        predictions,
        dtype=torch.long,
        device=logits.device
    )






















def build_head(d_model, n_tags, name, **cfg):
    if(name == "linear"):
        return Linear_head(d_model,n_tags,**cfg)
    elif(name ==  "mlp"):
        return Mlp_head(d_model,n_tags,**cfg)
    elif(name == "mlp_crf"):
        return Mlp_Crf__head(d_model,n_tags,**cfg)
    else:
        raise ValueError(f"please provide name of classification head in cfg , availible options : linear , mlp , mlp_crf")
   

In [ ]:
%%writefile /kaggle/working/phase3/techniques/lora.py
"""YOUR FILE: LoRA (Hu et al., 2021) [3].

Contract: see techniques/__init__.py.  apply(backbone, **cfg) -> None
e.g. freeze(backbone); replace_linears(backbone, <which projections>, <your LoRALinear around nn.Linear>)
cfg = the config's "technique" dict minus "name" (your options, e.g. rank / alpha / targets / ...).
"""
import torch
import torch.nn as nn
import math
from inject import freeze, unfreeze, replace_linears, wrap_modules   # noqa: F401  (helpers, optional)

class LoRALinear(nn.Module):
    def __init__(self,base:nn.Linear,r,alpha):
        super().__init__()
        self.base_layer=base
        self.in_features = base.in_features
        self.out_features = base.out_features
        for param in self.base_layer.parameters():
            param.requires_grad = False
        # define A and B as dimension B is out features cross r and A is r cross in features
        self.A = nn.Parameter(torch.randn(r, self.in_features))
        
        self.A = nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(self.out_features,r))
        self.scaling  = alpha/r
    def forward (self,x):
        base_output = self.base_layer(x)
        scaled_output = (x@self.A.T)@self.B.T
        return base_output+self.scaling*scaled_output
        



def apply(backbone, r, alpha, targets):
    freeze(backbone)
    replaced = replace_linears(backbone,
                               lambda name, lin: any(target in name for target in targets),
                               lambda name, lin: LoRALinear(lin, r, alpha))

In [ ]:
%%writefile /kaggle/working/phase3/techniques/lorma.py
"""YOUR FILE: LoRMA: Low-Rank Multiplicative Adaptation (Bihany, Patel, Modi, 2025) [12] -- COMPULSORY.

Contract: see techniques/__init__.py.  apply(backbone, **cfg) -> None
e.g. freeze(backbone); replace_linears(backbone, <which projections>, <your LoRMALinear around nn.Linear>)
cfg = the config's "technique" dict minus "name" (your options, e.g. rank / alpha / targets / ...).
"""
import torch
import torch.nn as nn
import math
from inject import freeze, unfreeze, replace_linears, wrap_modules   # noqa: F401  (helpers, optional)

class LoRMALinear(nn.Module):
    def __init__(self,base:nn.Linear,r,alpha):
        super().__init__()
        self.base_layer=base
        self.in_features = base.in_features
        self.out_features = base.out_features
        for param in self.base_layer.parameters():
            param.requires_grad = False
        # define A and B as dimension B is out features cross r and A is r cross in features
        self.A = nn.Parameter(torch.empty(r, self.out_features))
        
        self.A = nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(self.out_features,r))
        self.scaling  = alpha/r
    def forward (self,x):
        base_output = self.base_layer(x)
        scaled_output = (base_output@self.A.T)@self.B.T
        return base_output+self.scaling*scaled_output
        



def apply(backbone, r, alpha, targets):
    freeze(backbone)
    replaced = replace_linears(backbone,
                               lambda name, lin: any(target in name for target in targets),
                               lambda name, lin: LoRMALinear(lin, r, alpha))

In [ ]:
%%writefile /kaggle/working/phase3/techniques/kd.py
"""YOUR FILE: knowledge-distillation loss (Hinton et al., 2015) [2] -- COMPULSORY experiment.

Used when the config has a "kd" block: {"teacher_run": <run dir of a finished teacher>, ...your options}.
The plumbing loads the teacher (frozen, eval mode), runs it on exactly the same windows as the student and calls

kd_loss(student_logits, teacher_logits, tags, **cfg) -> scalar
    student_logits, teacher_logits: (B, T, n_tags) float32 (same tag set: checked)
    tags: (B, T) long gold tags, -100 = ignore
    cfg: the "kd" dict minus "teacher_run" (e.g. your temperature / mixing weight)
This loss REPLACES the normal task loss for the student, so include the gold-label term yourself if you want it.
The student's own adaptation is chosen as usual with "technique" (e.g. full_ft on a smaller backbone).
"""

"""YOUR FILE: knowledge-distillation loss (Hinton et al., 2015) [2]."""

import torch
import torch.nn.functional as F


def kd_loss(student_logits, teacher_logits, tags, **cfg):
    """
    Knowledge-distillation loss for token classification / NER.

    Args:
        student_logits: (B, T, C) student logits.
        teacher_logits: (B, T, C) teacher logits.
        tags: (B, T) gold tag IDs; -100 means ignore.
        cfg:
            temperature: softmax temperature (default 2.0).
            alpha: weight for KD loss (default 0.5).
                   Total loss = alpha * KD + (1-alpha) * CE.

    Returns:
        Scalar loss.
    """
    temperature = cfg.get("temperature", 2.0)
    alpha = cfg.get("alpha", 0.5)

    if temperature <= 0:
        raise ValueError("temperature must be positive")

    if not 0.0 <= alpha <= 1.0:
        raise ValueError("alpha must be between 0 and 1")

    if student_logits.shape != teacher_logits.shape:
        raise ValueError(
            "Student and teacher logits must have the same shape"
        )

    mask = tags != -100

  
    if not mask.any():
        return student_logits.sum() * 0.0

    
    teacher_logits = teacher_logits.detach()

  
    student_log_probs = F.log_softmax(
        student_logits / temperature, dim=-1
    )
    teacher_probs = F.softmax(
        teacher_logits / temperature, dim=-1
    )

    
    kl_per_token = F.kl_div(
        student_log_probs,
        teacher_probs,
        reduction="none"
    ).sum(dim=-1)

    kd_term = kl_per_token[mask].mean() * (temperature ** 2)

   
    ce_term = F.cross_entropy(
        student_logits[mask],
        tags[mask],
        weight=cfg.get("weight"),
    )

   
    return alpha * kd_term + (1.0 - alpha) * ce_term

In [ ]:
%%writefile /kaggle/working/phase3/techniques/rslora.py
"""YOUR FILE: Rank-Stabilized LoRA (rsLoRA) — Kalajdzievski (2023) [11].

WHAT IT IS
  Identical to LoRA (h = W0 x + scale * B(A(x)), A: r x in, B: out x r, B zero-initialised, W0 frozen) except for the
  scaling factor:
      LoRA:   scale = alpha / r
      rsLoRA: scale = alpha / sqrt(r)
  With alpha / r the update shrinks as the rank grows, so high ranks learn slowly; alpha / sqrt(r) keeps the update
  magnitude stable across ranks (that is the paper's point). Everything else (targets, init, freezing) as your LoRA.

WHAT THE PLUMBING CALLS
  apply(backbone, **cfg) -> None
      Called once after the Phase-1 weights are loaded. On entry every backbone parameter has requires_grad=True;
      on exit only your inserted A/B matrices should be trainable (the head is handled by the plumbing).
      cfg = the config's "technique" dict minus "name", e.g. {"name": "rslora", "r": ..., "alpha": ..., "targets": [...]}
      -> write it as  def apply(backbone, r, alpha, targets):  so a typo in the config is caught before training.

SUGGESTED STRUCTURE (mirrors your lora.py; you can import your LoRALinear and only change the scale)
  class RsLoRALinear(nn.Module):
      __init__(self, base: nn.Linear, r, alpha): keep `base` frozen, create A (r, in_features) and B (out_features, r),
          init A like your LoRA (kaiming_uniform_, a=sqrt(5)) and B = 0, scaling = alpha / math.sqrt(r)
      forward(self, x): base(x) + scaling * B(A(x))
  def apply(backbone, r, alpha, targets):
      freeze(backbone); replace_linears(backbone, lambda name, lin: <name matches targets>,
                                        lambda name, lin: RsLoRALinear(lin, r, alpha))

HELPERS: inject.freeze / replace_linears (see inject.py for the linear-layer names of A/B/C/D).
"""
import math

import torch
import torch.nn as nn

from inject import freeze, unfreeze, replace_linears, wrap_modules   # noqa: F401

class RSLoRALinear(nn.Module):
    def __init__(self,base:nn.Linear,r,alpha):
        super().__init__()
        self.base_layer=base
        self.in_features = base.in_features
        self.out_features = base.out_features
        for param in self.base_layer.parameters():
            param.requires_grad = False
        # define A and B as dimension B is out features cross r and A is r cross in features
        self.A = nn.Parameter(torch.randn(r, self.in_features))
        
        self.A = nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(self.out_features,r))
        self.scaling  = alpha/math.sqrt(r)
    def forward (self,x):
        base_output = self.base_layer(x)
        scaled_output = (x@self.A.T)@self.B.T
        return base_output+self.scaling*scaled_output
        



def apply(backbone, r, alpha, targets):
    freeze(backbone)
    replaced = replace_linears(backbone,
                               lambda name, lin: any(target in name for target in targets),
                               lambda name, lin: RSLoRALinear(lin, r, alpha))

In [ ]:
%%writefile /kaggle/working/phase3/techniques/ia3.py
"""YOUR FILE: IA³ — "Infused Adapter by Inhibiting and Amplifying Inner Activations", Liu et al. (2022) [13].

WHAT IT IS
  The backbone is frozen; the only new parameters are three learned VECTORS per block that rescale activations
  element-wise (no matrices, so very few parameters):
      keys:    k' = l_k ⊙ k        l_k ∈ R^{dim of k_proj output}
      values:  v' = l_v ⊙ v        l_v ∈ R^{dim of v_proj output}
      FFN:     h' = l_ff ⊙ act(W1 x)   applied to the FFN's inner activation, i.e. to the INPUT of the second FFN
                                        linear (ffn.net.2), l_ff ∈ R^{d_ff}
  All vectors are initialised to ONES, so at the start the model is exactly the pretrained one.
  (Note: in your models k_proj / v_proj output n_kv_heads * d_head — 256 for B/C/D, 768 for A — use out_features.)

WHAT THE PLUMBING CALLS
  apply(backbone, **cfg) -> None      (same contract as your other techniques)
      e.g. config {"name": "ia3"} (no options), or your own options such as which blocks / which of k, v, ffn.
      On exit only the l_k / l_v / l_ff vectors (plus nothing else) should have requires_grad=True.

SUGGESTED STRUCTURE
  class ScaleOutput(nn.Module):  wraps an nn.Linear; forward(x) = base(x) * self.scale   (scale = ones(out_features))
      -> use for attn.k_proj and attn.v_proj via replace_linears
  class ScaleInput(nn.Module):   wraps an nn.Linear; forward(x) = base(x * self.scale)   (scale = ones(in_features))
      -> use for ffn.net.2 (its input IS act(W1 x)), via replace_linears
  def apply(backbone, **cfg):
      freeze(backbone); replace_linears(... k_proj / v_proj -> ScaleOutput ...); replace_linears(... "ffn.net.2" -> ScaleInput ...)
  Parameter count check: per block len(l_k) + len(l_v) + d_ff (B: 256 + 256 + 3456 = 3,968 per block).
"""
import torch
import torch.nn as nn

from inject import freeze, unfreeze, replace_linears, wrap_modules   # noqa: F401

class ScaledOutput(nn.Module):
    def __init__(self,base:nn.Linear):
        super().__init__()
        self.base = base
        self.out_features = base.out_features
        self.in_features = base.in_features
        self.scale = nn.Parameter(torch.ones(self.out_features))
    def forward(self,x:torch.Tensor):
        x = self.base(x)
        return x*self.scale

class ScaledInput(nn.Module):
    def __init__(self,base:nn.Linear):
        super().__init__()
        self.base = base
        self.out_features = base.out_features
        self.in_features = base.in_features
        self.scale = nn.Parameter(torch.ones(self.in_features))
    def forward(self,x:torch.Tensor):
        x = self.base(self.scale*x)
        return x



def apply(backbone):
    freeze(backbone)

    replace_linears(
        backbone,
        lambda name, lin: name.endswith(("k_proj", "v_proj", "ffn.net.2")),
        lambda name, lin: (
            ScaledOutput(lin)
            if name.endswith(("k_proj", "v_proj"))
            else ScaledInput(lin)
        )
    )

In [ ]:
%%writefile /kaggle/working/phase3/techniques/prompt_tuning.py
"""YOUR FILE: Prompt Tuning — Lester, Al-Rfou & Constant (2021) [5].

WHAT IT IS
  The whole backbone is frozen. P learnable "soft prompt" vectors (each of size d_model) are prepended to the
  input embeddings of every window; only those P x d_model numbers (plus the head) train.

PLUMBING HOOK ALREADY IN PLACE (backbones.Backbone)
  * backbone.n_virtual_tokens: set it to P in apply(). From then on the backbone itself prepends P placeholder ids
    (<pad>, id 0) to every window and removes the first P output positions again, so pooling / heads are unchanged.
    The plumbing checks window_len + P <= max_seq_len.
  * backbone.get_input_embedding() / backbone.set_input_embedding(module): the token embedding the prompt goes into
    (A/B/D: lm.tok_emb; C: the encoder's lm.tok_emb1).
  Your job: make positions [:, :P] of the embedding output learnable.

WHAT THE PLUMBING CALLS
  apply(backbone, **cfg) -> None, e.g. config {"name": "prompt_tuning", "n_tokens": ..., "init": ...}
      -> def apply(backbone, n_tokens, init=...):

SUGGESTED STRUCTURE
  class SoftPromptEmbedding(nn.Module):
      __init__(self, base_emb: nn.Embedding, n_tokens, init):
          keep base_emb (frozen); self.prompt = nn.Parameter(tensor of shape (n_tokens, d_model))
          init options from the paper: random (small normal) or copied from the embeddings of real vocabulary tokens
      forward(self, ids):  e = base_emb(ids)                     # (B, P + T, d), first P ids are placeholders
                           return torch.cat([self.prompt.expand(B, -1, -1).to(e.dtype), e[:, P:]], dim=1)
  def apply(backbone, n_tokens, init):
      freeze(backbone)
      backbone.set_input_embedding(SoftPromptEmbedding(backbone.get_input_embedding(), n_tokens, init))
      backbone.n_virtual_tokens = n_tokens
  Note: A/B/D tie the LM head to tok_emb; the head is not used in Phase 3 (replaced by Identity), so wrapping is safe.
"""
import torch
import torch.nn as nn

from inject import freeze, unfreeze   # noqa: F401

class SoftPromptEmbedding(nn.Module):
  def __init__(self,base_emb:nn.Embedding,n_tokens,init):
      super().__init__()
      self.base_emb = base_emb
      self.d_model = base_emb.embedding_dim
      self.n_tokens = n_tokens
     
      self.prompt = nn.Parameter(torch.empty(self.n_tokens, self.d_model))

      if init == "random":
            nn.init.normal_(self.prompt, mean=0.0, std=0.02)

      else:
            raise ValueError(f"Unknown prompt initialization: {init}")



  def forward(self,ids):
    e = self.base_emb(ids)
    B,_,_ = e.shape
    return torch.cat([self.prompt.expand(B, -1, -1).to(e.dtype), e[:, self.n_tokens:]], dim=1)
def apply(backbone,n_tokens,init):
  freeze(backbone)
  backbone.set_input_embedding(SoftPromptEmbedding(backbone.get_input_embedding(), n_tokens, init))
  backbone.n_virtual_tokens = n_tokens


    

In [ ]:
%%writefile /kaggle/working/phase3/techniques/prefix_tuning.py
"""YOUR FILE: Prefix Tuning — Li & Liang (2021) [4].

WHAT IT IS
  The backbone is frozen. Every self-attention layer gets P learnable key vectors and P learnable value vectors
  (a "prefix") that all queries can attend to, in addition to the real tokens. Unlike prompt tuning (input layer
  only), this steers every layer. The paper reparameterises the prefix through a small MLP during training for
  stability (prefix = MLP(embedding)), which you may or may not do.

PLUMBING HOOK ALREADY IN PLACE (phase1_models/fastattn_ab.py and fastattn_cd.py)
  If an attention module has an attribute `prefix_kv`, the fused attention calls
        pk, pv = attn.prefix_kv(batch_size)        # each (B, n_kv_heads, P, d_head)
  prepends them to that layer's keys/values (BEFORE the GQA head repetition; no RoPE is applied to them) and extends
  the attention mask so every query sees all P prefix positions (causal / bidirectional / prefix-LM rules unchanged
  for the real tokens). Without `prefix_kv` attention is exactly as before. Supported: self-attention of A/B
  (fastattn_ab), C encoder/decoder self-attention and D prefix-LM attention (fastattn_cd); not cross-attention.
  Shapes: A has n_kv_heads = n_heads = 12; B/C/D have n_kv_heads = 4; d_head = 64 everywhere.

WHAT THE PLUMBING CALLS
  apply(backbone, **cfg) -> None, e.g. config {"name": "prefix_tuning", "n_prefix": ..., "reparam_hidden": ...}

SUGGESTED STRUCTURE
  class LayerPrefix(nn.Module):
      __init__(self, n_kv_heads, d_head, P, ...): self.k = nn.Parameter((n_kv_heads, P, d_head)), self.v = same
          (or an embedding + MLP producing them, for the reparameterised version)
      forward / __call__(self, B): return (self.k.expand(B, ...), self.v.expand(B, ...))
  def apply(backbone, n_prefix, ...):
      freeze(backbone)
      for block in backbone.blocks:                       # input -> output order; C "decoder" mode: enc + dec blocks
          attn = block.attn
          prefix = LayerPrefix(attn.n_kv_heads if hasattr(attn, "n_kv_heads") else attn.n_heads, attn.d_head, n_prefix)
          attn.prefix_module = prefix                     # register as a submodule so it is trained and saved
          attn.prefix_kv = prefix                         # the hook the fused attention calls
  IMPORTANT: assign the module as a submodule (setattr on the attention module) so its parameters are part of
  model.parameters() / state_dict; `prefix_kv` must be callable with the batch size.
"""



import torch
import torch.nn as nn

from inject import freeze, unfreeze   # noqa: F401


class LayerPrefix(nn.Module):
  def __init__(self,n_kv_heads,d_head,P):
    super().__init__()
    self.n_kv_heads = n_kv_heads
    self.d_head = d_head
    self.P=P
    self.k = nn.Parameter(torch.empty(n_kv_heads,P,d_head))
    self.v = nn.Parameter(torch.empty(n_kv_heads,P,d_head))
    nn.init.normal_(self.k,mean=0,std=0.02)
    nn.init.normal_(self.v,mean=0,std=0.02)
  def forward(self,B):
    return (self.k.unsqueeze(0).expand(B,-1,-1,-1),self.v.unsqueeze(0).expand(B,-1,-1,-1))







def apply(backbone,n_prefix):
  freeze(backbone)
  for block in backbone.blocks:
    attn=block.attn
    prefix = LayerPrefix(attn.n_kv_heads if hasattr(attn, "n_kv_heads") else attn.n_heads, attn.d_head, n_prefix)
    attn.prefix_module = prefix                     # register as a submodule so it is trained and saved
    attn.prefix_kv = prefix                         # the hook the fused attention calls

    

In [ ]:
%%writefile /kaggle/working/phase3/techniques/adalora.py
"""YOUR FILE: AdaLoRA — Zhang et al. (2023) [10].

WHAT IT IS
  LoRA with an ADAPTIVE rank per weight matrix. Each adapted linear gets an SVD-shaped update
      h = W0 x + scale * P diag(lam) Q x          P: (out, r), lam: (r,), Q: (r, in); lam starts at 0
  Training starts with an initial rank r_init everywhere and, on a schedule, prunes singular values (sets entries of
  lam to 0 / masks them) in the LESS important matrices until a total budget (target average rank) is reached, so
  important layers keep more rank. Importance of each triplet (P[:, i], lam[i], Q[i, :]) is computed from
  |param * grad| (sensitivity), smoothed with exponential moving averages (EMA of sensitivity and of its uncertainty).
  An orthogonality regulariser keeps P and Q close to orthogonal:  R = ||P^T P - I||_F^2 + ||Q Q^T - I||_F^2.

PLUMBING HOOKS (train_rr.py) — define what you need:
  apply(backbone, **cfg) -> None
      insert your AdaLoRALinear modules (freeze(backbone) + replace_linears), e.g. config
      {"name": "adalora", "r_init": ..., "target_rank": ..., "alpha": ..., "targets": [...], "t_init": ..., "t_final": ..., "orth_reg": ...}
  extra_loss(model) -> scalar tensor                                   (optional; called every micro-batch)
      return orth_reg * sum of the orthogonality terms over all AdaLoRA modules; it is ADDED to the task loss.
  on_step(model, step, total_steps) -> None                            (optional; called after every optimizer step)
      Called AFTER optimizer.step() and BEFORE the gradients are zeroed, so p.grad is still available here.
      (If train.grad_clip is None the gradients are still multiplied by the fp16 GradScaler factor; that factor is
      the same for every parameter, so importance RANKINGS from |p * p.grad| are unaffected.)
      1. update the importance EMAs from |lam * lam.grad| (and the P, Q terms if you use them);
      2. following the cubic budget schedule between t_init and total_steps - t_final, compute the current budget
         and mask the lowest-importance singular values globally across modules (lam[i] = 0 and keep them 0).
  Find the AdaLoRA modules from `model` with e.g. [m for m in model.modules() if isinstance(m, AdaLoRALinear)].
  Keep the masks as buffers so they are saved with the checkpoint.
"""
import math

import torch
import torch.nn as nn

from inject import freeze, unfreeze, replace_linears   # noqa: F401

class AdaLoRALinear(nn.Module):
    def __init__(self,base,r,scale,target_rank, t_init, t_final, delta_t, beta1, beta2, orth_reg):
        super().__init__()
        self.base_layer = base 
        self.in_features = base.in_features
        self.out_features = base.out_features
        self.alpha=scale/r
        self.r = r 
        self.P = nn.Parameter(torch.empty(self.out_features,self.r))
        self.Q = nn.Parameter(torch.empty(self.r,self.in_features))
        self.P=nn.init.xavier_normal_(self.P, gain=0.1)
        self.Q=nn.init.xavier_normal_(self.Q, gain=0.1)
        self.lam = nn.Parameter(torch.zeros(self.r))
        mask = torch.ones(self.r)
        ipt_ema = torch.zeros(self.r)
        unc_ema = torch.zeros(self.r)
        self.register_buffer('mask',mask)
        self.register_buffer('ipt_ema',ipt_ema)
        self.register_buffer('unc_ema',unc_ema)
        self.target_rank=target_rank
        self.t_init = t_init
        self.t_final=t_final
        self.delta_t=delta_t
        self.beta1 = beta1
        self.beta2=beta2
        self.orth_reg = orth_reg
    def forward(self,x:torch.Tensor):
        frozen = self.base_layer(x)
        proj_down = x@self.Q.T
        proj_down = proj_down*(self.lam*self.mask)
        proj_up = proj_down@self.P.T
        return frozen+self.alpha*proj_up

def extra_loss(model):
    mods = [m for m in model.modules() if isinstance(m, AdaLoRALinear)]

    if not mods:
        return next(model.parameters()).new_zeros(())

    loss = 0.0

    for m in mods:
        P = m.P
        Q = m.Q
        I = torch.eye(m.r, device=P.device, dtype=P.dtype)

        loss = loss + torch.norm(P.T @ P - I, p="fro") ** 2
        loss = loss + torch.norm(Q @ Q.T - I, p="fro") ** 2

    return mods[0].orth_reg * loss
def on_step(model, step, total_steps):
    mods = [
        m for m in model.modules()
        if isinstance(m, AdaLoRALinear)
    ]

    if not mods:
        return

    params = [
        p for m in mods for p in (m.P, m.Q, m.lam)
    ]

    # Skip the update if any required gradient is missing
    # or non-finite.
    if any(
        p.grad is None or not torch.isfinite(p.grad).all()
        for p in params
    ):
        return

    with torch.no_grad():
        # Update importance EMAs EVERY step.
        for m in mods:
            p_score = (m.P * m.P.grad).abs().mean(dim=0)
            q_score = (m.Q * m.Q.grad).abs().mean(dim=1)
            lam_score = (m.lam * m.lam.grad).abs()

            sensitivity = p_score + q_score + lam_score

            old_ipt = m.ipt_ema.clone()

            m.ipt_ema.mul_(m.beta1).add_(
                sensitivity, alpha=1 - m.beta1
            )

            uncertainty = (sensitivity - old_ipt).abs()

            m.unc_ema.mul_(m.beta2).add_(
                uncertainty, alpha=1 - m.beta2
            )

        # Only update the rank budget and masks every delta_t steps.
        if step % mods[0].delta_t != 0:
            return

        # Keep your existing cubic budget schedule and
        # global pruning code here.





def apply(backbone,r,alpha,targets,target_rank,t_init,t_final,delta_t,beta1,beta2,orth_reg):
    freeze(backbone)
    replaced = replace_linears(backbone,
                               lambda name, lin: any(target in name for target in targets),
                               lambda name, lin: AdaLoRALinear(lin, r, alpha,target_rank,t_init,t_final,delta_t,beta1,beta2,orth_reg))
    


# Optional hooks (delete them if you do not use them; the plumbing only calls what exists):
# def extra_loss(model): ...
# def on_step(model, step, total_steps): ...

In [ ]:
%%writefile /kaggle/working/phase3/techniques/qlora.py
"""YOUR FILE: QLoRA — Dettmers et al. (2023) [9].

WHAT IT IS
  LoRA on top of a QUANTISED frozen base: the frozen W0 of every adapted linear is stored in 4 bits and dequantised
  on the fly in the forward pass; only the LoRA A/B matrices (kept in 16/32-bit) train. The paper's pieces:
    * NF4 ("4-bit NormalFloat"): 16 fixed quantisation levels placed at quantiles of a standard normal, so normally
      distributed weights use all 16 levels evenly;
    * block-wise quantisation: weights split into blocks (e.g. 64 values); each block stores its absmax as the scale,
      values stored as the index (0..15) of the nearest NF4 level after dividing by absmax;
    * double quantisation (optional): the per-block absmax constants are themselves quantised (8-bit) to save memory;
    * paged optimizers (memory paging) — not needed here.
  Forward: W0_hat = dequant(codes, absmax);  h = x @ W0_hat^T + bias + scale * B(A(x)).

WHAT THE PLUMBING CALLS
  apply(backbone, **cfg) -> None, e.g. config {"name": "qlora", "r": ..., "alpha": ..., "targets": [...], "block_size": ...}

SUGGESTED STRUCTURE (from scratch, PyTorch only)
  NF4_LEVELS = torch.tensor([...16 values...])           # the paper / bitsandbytes NF4 table
  def quantize_nf4(w, block_size) -> (codes uint8, absmax float)   # per block: absmax, then nearest-level index
  def dequantize_nf4(codes, absmax, shape, block_size) -> tensor
  class QLoRALinear(nn.Module):
      __init__(self, base: nn.Linear, r, alpha, block_size):
          quantise base.weight once; register codes / absmax as BUFFERS (not parameters); drop the fp weight
          create LoRA A, B as in your lora.py (B = 0)
      forward(self, x): F.linear(x, dequantize(...).to(x.dtype), base.bias) + scaling * B(A(x))
  def apply(backbone, r, alpha, targets, block_size): freeze(backbone); replace_linears(... -> QLoRALinear ...)
  Note: the quantisation error of the frozen weights is part of the method; compare memory (peak_mem_GB in
  summary.json) with plain LoRA. (Using the bitsandbytes library instead would be a library implementation — if you
  do, cite it and declare it.)
"""
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

from inject import freeze, unfreeze, replace_linears   # noqa: F401

NF4_LEVELS = torch.tensor([
    -1.0, -0.6961928009986877, -0.5250730514526367,
    -0.39491748809814453, -0.28444138169288635,
    -0.18477343022823334, -0.09105003625154495, 0.0,
    0.07958029955625534, 0.16093020141124725,
    0.24611230194568634, 0.33791524171829224,
    0.44070982933044434, 0.5626170039176941,
    0.7229568362236023, 1.0
], dtype=torch.float32)
def quantize_nf4(w,block_size):
  assert w.numel() % block_size == 0
  blocks = w.reshape(-1,block_size)
  absmax = blocks.abs().max(dim=1).values.clamp(min=1e-12)
  blocks = blocks/absmax[:, None]
  dist  = (blocks[..., None] - NF4_LEVELS.to(blocks.device)).abs()
  codes = dist.argmin(dim=-1).to(torch.uint8)
  return (codes,absmax)
def dequantize_nf4(codes,absmax,shape,block_size):
  output = NF4_LEVELS.to(codes.device)[codes.long()]
  output = output*absmax[:, None]
  return output.reshape(shape)
class QLoRALinear(nn.Module):
  def __init__(self,base:nn.Linear,r,alpha,block_size):
    super().__init__()
    self.r = r 
    self.alpha=alpha
    self.block_size=block_size
    codes, absmax = quantize_nf4(base.weight.detach(), block_size)
    self.register_buffer("codes", codes)
    self.register_buffer("absmax", absmax)
    self.in_features=base.in_features
    self.out_features=base.out_features
    self.shape = tuple(base.weight.shape)
    if base.bias is not None:
      self.register_buffer("bias", base.bias.detach().clone())
    else:
      self.bias = None
    self.A = nn.Parameter(torch.randn(r, self.in_features))
        
    self.A = nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
    self.B = nn.Parameter(torch.zeros(self.out_features,r))
    self.scaling  = alpha/r
  def forward(self,x:torch.Tensor):
    base = F.linear(x, dequantize_nf4(self.codes,self.absmax,self.shape,self.block_size).to(x.dtype), self.bias)
    lora  = (x@self.A.T)@self.B.T
    return base+self.scaling*lora











def apply(backbone, r, alpha, targets,block_size):
    freeze(backbone)
    replaced = replace_linears(backbone,
                               lambda name, lin: any(target in name for target in targets),
                               lambda name, lin: QLoRALinear(lin, r, alpha,block_size))

In [ ]:
%%writefile /kaggle/working/phase3/techniques/partial_ft.py
"""YOUR FILE: Partial / layer-wise fine-tuning.

Contract: see techniques/__init__.py.  apply(backbone, **cfg) -> None
e.g. freeze(backbone); then unfreeze the parts you choose (backbone.blocks is input -> output order)
cfg = the config's "technique" dict minus "name" (your options, e.g. rank / alpha / targets / ...).
"""
import torch
import torch.nn as nn

from inject import freeze, unfreeze, replace_linears, wrap_modules   # noqa: F401  (helpers, optional)


def apply(backbone,k):
    freeze(backbone)
    if(k>0):
        
    # unfreeze the final norm layer
        if(backbone.kind == "encdec"):
            if backbone.encdec_mode == "encoder":
                unfreeze(backbone.lm.ln_f_enc)
            else:
                unfreeze(backbone.lm.ln_f_dec)


            
           
        else:
            unfreeze(backbone.lm.ln_f)
        for block in  backbone.blocks[-k:]:
            unfreeze(block)
        
        
    
    




   

In [ ]:
%%writefile /kaggle/working/phase3/techniques/bitfit.py
"""YOUR FILE: BitFit (Ben Zaken et al., 2022) [8].

Contract: see techniques/__init__.py.  apply(backbone, **cfg) -> None
decide which parameters stay trainable (note: attention projections are bias=False; FFN linears and LayerNorms have biases)
cfg = the config's "technique" dict minus "name" (your options, e.g. rank / alpha / targets / ...).
"""
import torch
import torch.nn as nn

from inject import freeze, unfreeze, replace_linears, wrap_modules   # noqa: F401  (helpers, optional)


def apply(backbone, **cfg):
    freeze(backbone)
    for name , param in backbone.named_parameters():
        if name.endswith("bias") is True:
            param.requires_grad = True
        

## Stage 2 — data, Phase-1 inputs, oracle check

In [ ]:
# Phase-3 data from the gated HF dataset (Kaggle secret "HF"), then the Phase-1 inputs
subprocess.run([sys.executable, "fetch_data.py", "--out", os.path.join(WORK, "data")], check=True)
DATA_DIR = os.path.join(WORK, "data", "Phase_3_RR")

def find_all(pattern):
    return sorted(glob.glob(os.path.join("/kaggle/input", "**", pattern), recursive=True))

def find_one(pattern):
    hits = find_all(pattern)
    if len(hits) > 1:
        print(f"  note: {len(hits)} matches for {pattern}, using the first: {hits}")
    return hits[0] if hits else None

def sha8(path):
    return hashlib.sha256(open(path, "rb").read()).hexdigest()[:8]

# every attached tokenizer must be the same file (the models were trained in different notebooks)
TOKS = find_all("legal_bpe.json")
assert TOKS, "legal_bpe.json not found under /kaggle/input (Add Input: your Phase-1 notebook outputs)"
tok_hashes = {t: sha8(t) for t in TOKS}
for t, h in tok_hashes.items():
    print(f"tokenizer {h}  {t}")
assert len(set(tok_hashes.values())) == 1, ("the attached Phase-1 outputs contain DIFFERENT legal_bpe.json files: "
                                            "models trained with different tokenizers cannot share one pipeline")
TOKENIZER = TOKS[0]
print("tokenizer sha256:", tok_hashes[TOKENIZER], "(Phase-1 legal BPE was 38f51f14)")

PHASE1 = {                                     # first match under /kaggle/input; or set explicit paths here
    "A": find_one("run_A/best.pt"),
    "B": find_one("run_B2/best.pt") or find_one("run_B/best.pt"),
    "C": find_one("run_C/best.pt"),
    "D": find_one("run_D/best.pt"),
}
for k, v in PHASE1.items():
    if v is None:
        print(f"Phase-1 {k}: NOT FOUND")
        continue
    ck = torch.load(v, map_location="cpu", weights_only=False)
    c = ck["config"]
    tok_used = c.get("tokenizer_path") or "?"
    print(f"Phase-1 {k}: {v}\n    tag={c.get('tag')} module={c['module']} step={ck.get('step')} "
          f"best_val={ck.get('best_val')} | trained with tokenizer at {tok_used}")
    del ck
missing = [k for k, v in PHASE1.items() if v is None]
if missing:
    print("WARNING: no checkpoint found for", missing, "-> runs using them will fail (Add Input: that notebook's output)")

In [ ]:
# oracle check (no model, ~1-2 min): gold labels through the real windows -> prediction -> JSON path for 4 context
# settings. Expected: dev macro-F1 1.0 and "counts match True" for every setting.
subprocess.run([sys.executable, "check_pipeline_rr.py", "--data", DATA_DIR, "--tokenizer", TOKENIZER], check=True)

## Stage 3 — configuration (yours)

In [ ]:
# =====================================================================================
# YOUR CHOICES (course rule 9) — values transcribed from the user's config (20-run campaign).
# Field meanings: docstrings of train_rr.py / rr_data.py.
# Window limits: A/B/D max_seq_len 896, C encoder 640, C decoder mode 256 (window_len + prompt tokens <= limit).
# =====================================================================================
import copy

BASE = {
    "data_dir": DATA_DIR,
    "tokenizer": TOKENIZER,
    "out_dir": os.path.join(WORK, "runs"),
    "fast_attention": True,

    "backbone": {
        "prefix": None,
        "encdec_mode": None,
        "truncate_layers": None,
        "dropout": 0.0,
    },

    "train_domains": ["CL", "IT"],

    "context": {
        "window_len": 768,
        "max_sent_tokens": 128,
        "max_targets": 4,
        "target_tokens": 128,
        "left_tokens": 64,
        "right_tokens": 64,
        "sep_token": True,
    },

    "pooling": "mean",

    "head": {
        "name": "mlp",
        "hidden": 256,
        "dropout": 0.1,
    },

    "technique": {"name": "lora"},
    "kd": None,

    "train": {
        "epochs": 8,
        "batch_size": 16,
        "micro_batch": 4,
        "lr": 1e-4,
        "head_lr": 3e-4,
        "weight_decay": 0.01,
        "warmup_frac": 0.06,
        "schedule": "cosine",
        "grad_clip": 1.0,
        "class_weights": "inv_sqrt",
        "none_weight": None,
        "patience": None,         # campaign 3: no early stopping -> all epochs; the best epoch is still kept
        "max_minutes": 40,
        "seed": 42,
        "log_every": 20,
        "infer_micro_batch": 8,
    },
}


def deep_update(d, u):
    for k, v in u.items():
        if isinstance(v, dict) and isinstance(d.get(k), dict):
            deep_update(d[k], v)
        else:
            d[k] = copy.deepcopy(v)
    return d


LORA_OPTS = {
    "r": 8,
    "alpha": 16,
    "targets": ["attn", "ffn"],
}

TECH_OPTS = {
    "lora": LORA_OPTS,
    "lorma": LORA_OPTS,
    "rslora": LORA_OPTS,

    "qlora": {
        **LORA_OPTS,
        "block_size": 64,
    },

    "adalora": {
        **LORA_OPTS,
        "target_rank": 2,
        "t_init": 100,
        "t_final": 100,
        "delta_t": 10,
        "beta1": 0.85,
        "beta2": 0.85,
        "orth_reg": 0.01,
    },

    "ia3": {},

    "prompt_tuning": {
        "n_tokens": 8,
        "init": "random",
    },

    "prefix_tuning": {
        "n_prefix": 8,
    },
}


def tech(name, **opts):
    return {
        "name": name,
        **copy.deepcopy(TECH_OPTS[name]),
        **opts,
    }


def train(**opts):
    return {"train": opts}


def exp(model, technique, **overrides):
    return {
        "model": model,
        "technique": technique,
        **overrides,
    }


# ============================================================
# CAMPAIGN 3 (2026-10-16): the 27 runs of campaigns 1 + 2, unchanged except BASE patience None (all epochs,
# best epoch kept). Names get the suffix _allep so outputs / zips / Codabench entries are distinguishable.
# KD teachers are the re-trained B_lora_allep of THIS session (no earlier output needs to be attached).
# Ensembles: Stage 6 (left empty; decided after these runs).
# ============================================================
SUFFIX = "_allep"

# ---------------- campaign 1 (20 runs) ----------------
C1 = {
    # --------------------------------------------------------
    # 1. B2 BASELINES: ALL EIGHT REQUIRED PEFT TECHNIQUES
    # --------------------------------------------------------

    "B_lorma": exp(
        "B", tech("lorma", r=8, alpha=16),
        **train(lr=2e-4),
    ),

    "B_lora": exp(
        "B", tech("lora", r=8, alpha=16),
        **train(lr=1e-4),
    ),

    "B_rslora": exp(
        "B", tech("rslora", r=8, alpha=16),
        **train(lr=1e-4),
    ),

    "B_adalora": exp(
        "B", tech(
            "adalora",
            r=8, alpha=16, target_rank=2,
            t_init=100, t_final=100, delta_t=10,
            beta1=0.85, beta2=0.85, orth_reg=0.01,
        ),
        **train(lr=1e-4),
    ),

    "B_qlora": exp(
        "B", tech("qlora", r=8, alpha=16, block_size=64),
        **train(lr=1e-4),
    ),

    "B_ia3": exp(
        "B", tech("ia3"),
        **train(lr=5e-4),
    ),

    "B_prompt_tuning": exp(
        "B", tech("prompt_tuning", n_tokens=8, init="random"),
        **train(lr=1e-3),
    ),

    "B_prefix_tuning": exp(
        "B", tech("prefix_tuning", n_prefix=8),
        **train(lr=5e-4),
    ),

    # --------------------------------------------------------
    # 2. REQUIRED ARCHITECTURE COVERAGE
    # --------------------------------------------------------

    "A_run": exp(
        "A", tech("lora", r=8, alpha=16),
        **train(lr=1e-4),
    ),

    "C_run": exp(
        "C", tech("lora", r=8, alpha=16),
        backbone={"encdec_mode": "encoder"},
        context={"window_len": 512},
        **train(lr=1e-4),
    ),

    "D_run": exp(
        "D", tech("lora", r=8, alpha=16),
        backbone={"prefix": 640},
        **train(lr=1e-4),
    ),

    # KD teacher: B_lora. Must remain after B_lora in RUN.
    "KD": exp(
        "B",
        tech("lora", r=8, alpha=16),
        backbone={"truncate_layers": 12},
        kd={
            "teacher_run": os.path.join(WORK, "runs", "B_lora"),
            "temperature": 2.0,
            "alpha": 0.5,
        },
        **train(lr=1e-4, head_lr=3e-4),
    ),

    # --------------------------------------------------------
    # 3. TARGETED B2 ABLATIONS
    # --------------------------------------------------------

    # Lower adapter learning rate
    "B_lora_lr5e5": exp(
        "B", tech("lora", r=8, alpha=16),
        **train(lr=5e-5),
    ),

    # Higher adapter learning rate
    "B_lora_lr2e4": exp(
        "B", tech("lora", r=8, alpha=16),
        **train(lr=2e-4),
    ),

    # More LoRA capacity
    "B_lora_r16": exp(
        "B", tech("lora", r=16, alpha=32),
        **train(lr=1e-4),
    ),

    # More rsLoRA capacity
    "B_rslora_r16": exp(
        "B", tech("rslora", r=16, alpha=32),
        **train(lr=1e-4),
    ),

    # Retain more AdaLoRA components
    "B_adalora_r4": exp(
        "B", tech(
            "adalora",
            r=8, alpha=16, target_rank=4,
            t_init=100, t_final=100, delta_t=10,
            beta1=0.85, beta2=0.85, orth_reg=0.01,
        ),
        **train(lr=1e-4),
    ),

    # More virtual prompt tokens
    "B_prompt_p16": exp(
        "B", tech("prompt_tuning", n_tokens=16, init="random"),
        **train(lr=1e-3),
    ),

    # Stronger rare-class weighting
    "B_lora_inv_freq": exp(
        "B", tech("lora", r=8, alpha=16),
        **train(class_weights="inv_freq"),
    ),

    # Change actual context budgets, not just window_len.
    # Compare against BASE: left/right = 64/64.
    "B_lora_ctx_local": exp(
        "B", tech("lora", r=8, alpha=16),
        context={
            "window_len": 768,
            "target_tokens": 128,
            "left_tokens": 0,
            "right_tokens": 0,
        },
        **train(lr=1e-4),
    ),
}


# Required runs first, then eight targeted B2 experiments.
C1_RUN = [
    "B_lorma",
    "B_lora",
    "B_rslora",
    "B_adalora",
    "B_qlora",
    "B_ia3",
    "B_prompt_tuning",
    "B_prefix_tuning",

    "A_run",
    "C_run",
    "D_run",
    "KD",

    "B_lora_lr5e5",
    "B_lora_lr2e4",
    "B_lora_r16",
    "B_rslora_r16",
    "B_adalora_r4",
    "B_prompt_p16",
    "B_lora_inv_freq",
    "B_lora_ctx_local",
]

assert len(C1_RUN) == 20 and len(set(C1_RUN)) == 20

# ---------------- campaign 2 (7 runs) ----------------
C2 = {
    # 7-10: New B2 training runs
    "B_rslora_r16_lr5e5": exp(
        "B", tech("rslora", r=16, alpha=32, targets=["attn", "ffn"]),
        **train(lr=5e-5, head_lr=3e-4, seed=42),
    ),
    "B_rslora_r16_lr2e4": exp(
        "B", tech("rslora", r=16, alpha=32, targets=["attn", "ffn"]),
        **train(lr=2e-4, head_lr=3e-4, seed=42),
    ),
    "B_lora_seed123": exp(
        "B", tech("lora", r=8, alpha=16, targets=["attn", "ffn"]),
        **train(lr=1e-4, head_lr=3e-4, seed=123),
    ),
    "KD_seed123": exp(
        "B", tech("lora", r=8, alpha=16, targets=["attn", "ffn"]),
        backbone={"truncate_layers": 12},     # as in the campaign-1 KD run (BASE dropout 0.0); only the seed differs
        kd={
            "teacher_run": "B_lora",          # campaign-1 run (resolved below to the attached output)
            "temperature": 2.0,
            "alpha": 0.5,
        },
        **train(lr=1e-4, head_lr=3e-4, seed=123),
    ),
    # 11: CRF head
    "B_lora_crf": exp(
        "B", tech("lora", r=8, alpha=16, targets=["attn", "ffn"]),
        head={"name": "mlp_crf"},
        **train(lr=1e-4, head_lr=3e-4, class_weights="inv_sqrt", seed=42),
    ),
    # 12: Larger actual context budget
    "B_lora_ctx_wide": exp(
        "B", tech("lora", r=8, alpha=16, targets=["attn", "ffn"]),
        context={
            "window_len": 896,
            "max_sent_tokens": 128,
            "max_targets": 4,
            "target_tokens": 128,
            "left_tokens": 128,
            "right_tokens": 128,
            "sep_token": True,
        },
        **train(lr=1e-4, head_lr=3e-4, seed=42),
    ),
    # 13: No class weighting
    "B_lora_unweighted": exp(
        "B", tech("lora", r=8, alpha=16, targets=["attn", "ffn"]),
        **train(lr=1e-4, head_lr=3e-4, class_weights=None, seed=42),
    ),
}

C2_RUN = [
    "B_rslora_r16_lr5e5",
    "B_rslora_r16_lr2e4",
    "B_lora_seed123",
    "KD_seed123",
    "B_lora_crf",
    "B_lora_ctx_wide",
    "B_lora_unweighted",
]
assert len(C2_RUN) == 7 and len(set(C2_RUN)) == 7

EXPERIMENTS, RUN = {}, []
for _n in C1_RUN + C2_RUN:
    _e = copy.deepcopy(C1[_n] if _n in C1 else C2[_n])
    if _e.get("kd"):
        _e["kd"]["teacher_run"] = os.path.join(WORK, "runs", os.path.basename(str(_e["kd"]["teacher_run"]).rstrip("/")) + SUFFIX)
    EXPERIMENTS[_n + SUFFIX] = _e
    RUN.append(_n + SUFFIX)
assert len(RUN) == 27 and len(set(RUN)) == 27

def find_prev_run(name):
    # a finished campaign-1 run (best_model.pt + summary.json) in WORK/runs or an attached /kaggle/input output
    for d in [os.path.join(WORK, "runs", name)] + sorted(glob.glob(f"/kaggle/input/**/runs/{name}", recursive=True)):
        if all(os.path.exists(os.path.join(d, f)) for f in ("best_model.pt", "summary.json", "config.json", "labels.json")):
            return d
    return None

# KD teachers: a run earlier in RUN (trained in this session) or a finished campaign-1 run
for _n in RUN:
    _kd = EXPERIMENTS[_n].get("kd")
    if _kd:
        _t = os.path.basename(str(_kd["teacher_run"]).rstrip("/"))
        if _t in RUN[:RUN.index(_n)]:
            _kd["teacher_run"] = os.path.join(WORK, "runs", _t)
        else:
            _d = find_prev_run(_t)
            assert _d, (f"{_n}: KD teacher {_t} not found: attach the campaign-1 output (Add Input -> previous version "
                        f"of this notebook) so that /kaggle/input/**/runs/{_t}/best_model.pt exists")
            _kd["teacher_run"] = _d
        print(f"{_n}: KD teacher -> {_kd['teacher_run']}")



import train_rr, backbones
for _n in RUN:                                   # a technique chosen for A/C/D/KD gets its option keys automatically
    _t = EXPERIMENTS[_n]["technique"]
    if _t.get("name") in TECH_OPTS:
        EXPERIMENTS[_n]["technique"] = {**tech(_t["name"]), **_t}
_ph = [n for n in RUN if "<" in str(EXPERIMENTS[n].get("kd", {}).get("teacher_run", ""))]
assert not _ph, f"{_ph}: set kd.teacher_run to a real run name"
CFG_PATHS = []
for name in RUN:
    exp = copy.deepcopy(EXPERIMENTS[name])
    model = exp.pop("model")
    assert model in PHASE1, f"{name}: set 'model' to one of {list(PHASE1)}"
    cfg = deep_update(copy.deepcopy(BASE), exp)
    cfg["name"] = name
    cfg["phase1_ckpt"] = PHASE1[model]
    assert cfg["phase1_ckpt"], f"{name}: no Phase-1 checkpoint found for model {model}"
    train_rr.check_config(cfg)                 # required keys + technique options vs YOUR apply() signature
    _, kind, spec, _ = backbones.load_phase1(cfg["phase1_ckpt"], load_weights=False)
    if kind == "encdec":
        limit = spec["encoder"]["max_seq_len"] if cfg["backbone"]["encdec_mode"] == "encoder" else \
            min(spec["encoder"]["max_seq_len"], spec["decoder"]["max_seq_len"])
    else:
        limit = spec["model"]["max_seq_len"]
    prompt_tokens = cfg["technique"].get("n_tokens", 0) if cfg["technique"]["name"] == "prompt_tuning" else 0
    assert cfg["context"]["window_len"] + prompt_tokens <= limit,         f"{name}: window_len + prompt tokens > {limit} (max length of model {model})"
    if cfg.get("kd"):
        teacher = os.path.basename(cfg["kd"]["teacher_run"].rstrip("/"))
        assert teacher in RUN[:RUN.index(name)] or os.path.exists(os.path.join(cfg["kd"]["teacher_run"], "summary.json")),             f"{name}: teacher {teacher} must come earlier in RUN or be a finished run"
    path = os.path.join(WORK, "configs", f"{name}.json")
    json.dump(cfg, open(path, "w"), indent=2)
    CFG_PATHS.append(path)
    print(f"{name:12s} | model={model} ({kind}) | technique={cfg['technique']['name']:13s} | head={cfg['head']['name']} | "
          f"pool={cfg['pooling']} | window={cfg['context']['window_len']}/{limit} | ctx L/R={cfg['context']['left_tokens']}/{cfg['context']['right_tokens']}")
PHASE3_LIST = {"lora", "qlora", "adalora", "rslora", "lorma", "prefix_tuning", "prompt_tuning", "ia3"}   # + KD (§9.2.1)
_cfgs = [json.load(open(p)) for p in CFG_PATHS]
_listed = {c["technique"]["name"] for c in _cfgs if not c.get("kd")} & PHASE3_LIST
_off = sorted({c["name"] for c in _cfgs if not c.get("kd") and c["technique"]["name"] not in PHASE3_LIST})
_kd = any(c.get("kd") for c in _cfgs)
print(f"{len(CFG_PATHS)} configs | Phase-3-listed techniques: {sorted(_listed)} ({len(_listed)}) + KD: {_kd} "
      f"-> {len(_listed) + int(_kd)} distinct (need >= 6, LoRMA compulsory: {'lorma' in _listed})")
if _off:
    print("WARNING: runs using a technique NOT on the Phase-3 list (do not count towards the 6):", _off)

## Stage 3b — real-model smoke test (few steps per run)

In [ ]:
# Real-model smoke test (~5-10 min on 2xT4) BEFORE the full runs: every config in RUN, unchanged (real checkpoint,
# technique, head, window, micro-batch -> real GPU memory), but only 3 optimizer steps on 4 train docs, then
# evaluation on 1 dev doc per domain, save -> reload, prediction on 1 test doc per domain and the zip. Outputs go to runs_smoke/ (not
# submittable). If anything crashes or runs out of memory, fix it here instead of an hour into the real run.
RUN_SMOKE = True
if RUN_SMOKE:
    smoke_paths = []
    for path in CFG_PATHS:
        c = json.load(open(path))
        c["smoke"] = {"steps": 3}
        c["out_dir"] = os.path.join(WORK, "runs_smoke")
        c["train"]["max_minutes"] = None
        if c.get("kd") and os.path.basename(c["kd"]["teacher_run"].rstrip("/")) in RUN:   # teacher trained here:
            c["kd"]["teacher_run"] = os.path.join(WORK, "runs_smoke", os.path.basename(c["kd"]["teacher_run"].rstrip("/")))
        # (a finished teacher from an attached earlier output is used as is)
        sp = os.path.join(WORK, "configs", f"smoke_{c['name']}.json")
        json.dump(c, open(sp, "w"), indent=2)
        smoke_paths.append(sp)
    t0 = time.time()
    gpus = ",".join(str(i) for i in range(max(1, N_GPUS)))
    sp_proc = subprocess.run([sys.executable, "run_many.py", "--gpus", gpus, "--every", "60", *smoke_paths])
    print(f"\nsmoke test: {(time.time() - t0) / 60:.1f} min")
    for path in CFG_PATHS:
        name = json.load(open(path))["name"]
        sm = os.path.join(WORK, "runs_smoke", name, "summary.json")
        if os.path.exists(sm):
            s_ = json.load(open(sm))
            print(f"  OK   {name:10s} peak GPU memory {s_['peak_mem_GB']} GB of ~15 | trainable {s_['params']['trainable']:,} "
                  f"({s_['params']['trainable_pct']}%) | zip written")
        else:
            print(f"  FAIL {name:10s} -> see {os.path.join(WORK, 'runs_smoke', name + '.stdout')}")
    assert sp_proc.returncode == 0, "smoke test failed: fix the runs marked FAIL before launching the full runs"
    # free disk: smoke checkpoints are full state dicts (~0.6-0.8 GB each); Kaggle's /kaggle/working limit is 20 GB
    freed = 0
    for ck in glob.glob(os.path.join(WORK, "runs_smoke", "*", "best_model.pt")):
        freed += os.path.getsize(ck)
        os.remove(ck)
    print(f"removed smoke checkpoints: {freed / 1e9:.1f} GB freed (logs and summaries kept)")
    print("SMOKE TEST PASSED -> run the next cell for the full runs")

## Stage 4 — runs (both GPUs)

In [ ]:
# Runs every config in RUN on BOTH T4s in parallel: one training process per GPU at a time, two fine-tunings at once.
# Shared queue: whichever GPU frees up first takes the next runnable config (in RUN order). A KD run is skipped until its
# teacher has finished (summary.json), so it never blocks a GPU. Status every 5 min; logs: runs/<name>.stdout,
# runs/<name>/log.jsonl. Running `nvidia-smi` in another cell while this runs should show one python process per GPU.
assert N_GPUS >= 1, "no GPU visible: set Accelerator to 'GPU T4 x2'"
if N_GPUS < 2:
    print("WARNING: only", N_GPUS, "GPU visible -> runs will be sequential. Set Accelerator to 'GPU T4 x2'.")
gpus = ",".join(str(i) for i in range(N_GPUS))
print("launching on GPUs", gpus)
SESSION_HOURS = 11.5        # Kaggle kills a session at 12 h: runs that cannot finish before this are SKIPPED, not started
STOP_AT = SESSION_START + SESSION_HOURS * 3600
print(f"runs may start until they fit before {SESSION_HOURS} h of session time "
      f"({(STOP_AT - time.time()) / 3600:.2f} h left now)")
proc = subprocess.run([sys.executable, "run_many.py", "--gpus", gpus, "--every", "300", "--stop-at", str(STOP_AT), *CFG_PATHS])
print(f"run_many exit code {proc.returncode} (non-zero = at least one run failed: see the summary above)")
print(f"session time used: {(time.time() - SESSION_START) / 3600:.2f} h")

## Stage 5 — results and zips

In [ ]:
# results table + all zips in /kaggle/working/submissions
import pandas as pd
rows = []
os.makedirs("/kaggle/working/submissions", exist_ok=True)
for sp in sorted(glob.glob(os.path.join(WORK, "runs", "*", "summary.json"))):
    s = json.load(open(sp))
    if "dev_macro_f1" not in s or "phase1" not in s:
        continue
    rows.append(dict(name=s["name"], phase1=s["phase1"].get("tag") or s["phase1"]["module"], technique=s["technique"]["name"],
                     head=s["head"]["name"], pooling=s["pooling"], kd=s["kd"], dev_macro_f1=s["dev_macro_f1"],
                     dev_CL=s["dev_CL_macro_f1"], dev_IT=s["dev_IT_macro_f1"], dev_P=s["dev_macro_p"], dev_R=s["dev_macro_r"],
                     trainable=s["params"]["trainable"], trainable_pct=s["params"]["trainable_pct"], total_params=s["params"]["total"],
                     best_epoch=s["best_epoch"], train_min=round(s["train_seconds"] / 60, 1), peak_GB=s["peak_mem_GB"]))
    z = os.path.join(os.path.dirname(sp), f"submission_{s['name']}.zip")
    if os.path.exists(z):
        shutil.copy(z, "/kaggle/working/submissions/")
df = pd.DataFrame(rows)
display(df)
df.to_csv("/kaggle/working/submissions/summary.csv", index=False)
print(sorted(os.listdir("/kaggle/working/submissions")))

In [ ]:
# per-label dev scores (all / CL / IT) and the most frequent confusions (counts only; interpretation is yours)
for vp in sorted(glob.glob(os.path.join(WORK, "runs", "*", "dev_scores.json"))):
    v = json.load(open(vp))
    print(os.path.basename(os.path.dirname(vp)), "| macro-F1 all", v["all"]["macro_f1"], "| CL", v.get("CL", {}).get("macro_f1"),
          "| IT", v.get("IT", {}).get("macro_f1"))
    display(pd.DataFrame(v["all"]["per_label"]).T)
    print("top confusions (gold->pred):", list(v["all"]["confusion"].items())[:8])

## Stage 6 — optional voting ensemble

In [ ]:
# Sentence-level weighted majority-vote ensembles of finished runs (members / weights are your choice, decided after
# looking at the dev scores of the 20 runs). One entry per ensemble submission:
#   name -> {"members": [run names], "weights": None or one number per member}
# Works in the SAME session after Stage 4, or in a LATER session: attach this notebook's output as input, run the
# setup + data cells (Stages 0-2, no GPU needed), then this cell. Runs are looked up in WORK/runs first, then in any
# attached /kaggle/input/**/runs/<name>/ folder.
# Campaign 3: fill after the 27 _allep runs finish (members = run names, e.g. "B_lorma_allep"); skipped while None.
ENSEMBLES = {
    "ENS_new": {"members": None, "weights": None},
}

def find_run(name):
    for d in [os.path.join(WORK, "runs", name)] + sorted(glob.glob(f"/kaggle/input/**/runs/{name}", recursive=True)):
        if os.path.exists(os.path.join(d, "prediction.json")) and os.path.exists(os.path.join(d, "dev_predictions.json")):
            return d
    return None

os.makedirs("/kaggle/working/submissions", exist_ok=True)
for ens_name, e in ENSEMBLES.items():
    if not e["members"]:
        print(f"{ens_name}: skipped (fill members)")
        continue
    dirs = {m: find_run(m) for m in e["members"]}
    missing = [m for m, d in dirs.items() if d is None]
    assert not missing, f"{ens_name}: no finished run found for {missing} (in WORK/runs or /kaggle/input/**/runs)"
    print(f"{ens_name}: members", {m: d for m, d in dirs.items()})
    cmd = [sys.executable, "rr_ensemble.py", "--name", ens_name, "--out", os.path.join(WORK, "ensembles"),
           "--members", *dirs.values()]
    if e["weights"]:
        assert len(e["weights"]) == len(e["members"]), f"{ens_name}: one weight per member"
        cmd += ["--weights", *map(str, e["weights"])]
    subprocess.run(cmd, check=True)
    shutil.copy(os.path.join(WORK, "ensembles", ens_name, f"submission_{ens_name}.zip"), "/kaggle/working/submissions/")
    print(f"{ens_name}: dev macro-F1", json.load(open(os.path.join(WORK, "ensembles", ens_name, "summary.json"))).get("dev_macro_f1"),
          "| zip copied to /kaggle/working/submissions/")